> **Edición pública anonimizada.** Nombres y rutas comerciales se sustituyeron por etiquetas genéricas; no se incluyen bases ni modelos originales. Los resultados históricos se consultan en la tesis adaptada. La demostración pública usa datos y modelos sintéticos.


In [ ]:
from pathlib import Path
import os, sys

# Find the repository independently of the computer or clone location.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / ".tesis-portafolio").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Abre Jupyter desde la carpeta de este repositorio.")
os.chdir(REPO_ROOT / '02 K Means')


# Análisis de Clusters K-Means — Leads Inmobiliarios

## Alcance: análisis exploratorio de la población completa de leads

**Dataset:** `leads_enriched_final.csv` | ~408 k registros × 62 variables  
**Objetivo:** Caracterizar la población completa de leads y evaluar la viabilidad de aplicar K-Means sobre el conjunto sin restricciones.

---


## Metodología y organización del análisis

Este notebook documenta de forma reproducible el proceso completo de segmentación de leads inmobiliarios mediante K-Means. La estructura se organiza en tres componentes principales: preparación y exploración de datos, construcción del modelo de clustering y validación estadística y de negocio. Se conserva el código original en cada etapa para garantizar trazabilidad y reproducibilidad.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Prepara el entorno: importa las librerías, fija parámetros reproducibles y define las rutas utilizadas por el análisis.</div>


In [ ]:
# !pip install scikit-learn seaborn scipy   # descomentar si es necesario

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})
sns.set_palette('Set2')

DATA_PATH           = 'Data/RAW/leads_enriched_final.csv'
RANDOM_STATE        = 42
SAMPLE_K_SELECTION  = 30_000   # filas para evaluar k
K_RANGE             = range(2, 11)
K_OPTIMO            = 4        # ajustar según sección 8


## 0. Preparación general y exploración inicial


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Carga el archivo de leads en memoria y muestra una primera vista para comprobar que los datos se leyeron correctamente.</div>


In [ ]:
df_raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig', low_memory=False)
print(f'Shape: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
df_raw.head(3)


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Inspecciona la estructura del dataset: número de registros, columnas, tipos de datos y cobertura de valores no nulos.</div>


In [ ]:
df_raw.info(verbose=True, show_counts=True)


### 0.1 Preparación inicial

Eliminamos columnas que no aportan al clustering:
- IDs y claves de negocio
- CSI / NPS (post-compra, alto missingness en leads no convertidos)
- Fechas, referidos con alta cardinalidad


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Elimina identificadores y variables fuera del alcance del clustering, conservando únicamente campos útiles para el análisis.</div>


In [ ]:
id_cols       = ['id_trato', 'person_id', 'user_id',
                 'apartamento_no', 'apto_referidor',
                 'bodega_asignada', 'parqueo_1', 'parqueo_2', 'parqueo_3']
csi_nps_cols  = [c for c in df_raw.columns if c.startswith('csi_') or c.startswith('nps_')]
date_cols     = ['fecha_creacion_trato']
ref_cols      = ['proyecto_referidor', 'sector_referidor', 'torre_referidora',
                 'ubicacion_bodega', 'ubicacion_parqueos', 'campania',
                 'fuente_pt2', 'fuente_dataset', 'proyecto_fuera_alcance']

to_drop = set(id_cols + csi_nps_cols + date_cols + ref_cols)
to_drop = [c for c in to_drop if c in df_raw.columns]

df = df_raw.drop(columns=to_drop).copy()
print(f'Columnas tras limpieza: {df.shape[1]} ({len(to_drop)} eliminadas)')
print(sorted(df.columns.tolist()))


### 0.2 EDA específico para clustering


#### 0.2.1 Análisis de missingness


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
miss_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]

fig, ax = plt.subplots(figsize=(10, max(4, len(miss_pct) * 0.3)))
miss_pct.sort_values().plot(kind='barh', ax=ax, color='#E07B54')
ax.axvline(50, color='red', linestyle='--', linewidth=1, label='50 %')
ax.set_xlabel('% valores faltantes')
ax.set_title('Missingness por variable')
ax.legend()
plt.tight_layout()
plt.show()

print(miss_pct.to_frame('% missing').to_string())


#### 0.2.2 Distribución de variables numéricas


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Grafica las distribuciones numéricas para revisar asimetrías, concentración y posibles valores atípicos.</div>


In [ ]:
num_cols_eda = [c for c in df.select_dtypes(include='number').columns
               if c != 'target_won']

ncols = 3
nrows = -(-len(num_cols_eda) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    df[col].dropna().hist(bins=40, ax=axes[i], color='#5B8DB8', edgecolor='white')
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de variables numéricas', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


#### 0.2.3 Distribución de variables categóricas


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica las categorías más frecuentes para conocer la composición de las variables cualitativas.</div>


In [ ]:
cat_cols_eda = [c for c in df.select_dtypes(include='object').columns
               if df[c].nunique() <= 30]

for col in cat_cols_eda:
    vc = df[col].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(9, 3))
    vc.sort_values().plot(kind='barh', ax=ax, color='#5B8DB8', edgecolor='white')
    ax.set_title(f'{col}  (n_unique={df[col].nunique()})')
    plt.tight_layout()
    plt.show()


#### 0.2.4 Correlaciones entre variables numéricas


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula la matriz de correlaciones y la representa como mapa de calor para detectar relaciones y redundancias entre variables numéricas.</div>


In [ ]:
num_corr_df = df.select_dtypes(include='number').drop(columns=['target_won'], errors='ignore')
corr = num_corr_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, square=True)
ax.set_title('Correlación entre variables numéricas')
plt.tight_layout()
plt.show()


### 0.3 Transición hacia los experimentos


## 1. Experimento 1 — Segmentación con variables de adquisición y canal

El bloque siguiente introduce la segunda fase del análisis, centrada en la construcción y evaluación de un modelo de clustering para leads calificados. En esta etapa se prioriza la calidad del universo analítico y la estabilidad de la segmentación.


### 1.1 Configuración del experimento


### 1.2 Carga de datos


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Aplica el criterio de completitud y construye el universo analítico que se utilizará en las etapas de clustering.</div>


In [ ]:
DATA_PATH_F2           = (f'{REPO_ROOT}/'
                          '01 Limpieza Código/data/features/'
                          'leads_enriched_qualified_final.csv')
RANDOM_STATE_F2        = 42
SAMPLE_K_SEL_F2        = 20_000
K_RANGE_F2             = range(2, 11)
K_OPTIMO_F2            = 4        # ajustar según resultados de la sección 8
PARSE_RATE_NON_NULL    = 0.90     # fracción mínima de no-nulos parseables para convertir a numérico
COMPLETENESS_THRESHOLD = 0.85     # fracción mínima de variables clave completas para incluir un registro en el clustering

#### 1.2.1 Validación inicial de la carga


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Carga el archivo de leads en memoria y muestra una primera vista para comprobar que los datos se leyeron correctamente.</div>


In [ ]:
df2_raw = pd.read_csv(DATA_PATH_F2, encoding='utf-8-sig', low_memory=False)
print(f'Shape: {df2_raw.shape[0]:,} filas × {df2_raw.shape[1]} columnas')
df2_raw.head(3)

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Inspecciona la estructura del dataset: número de registros, columnas, tipos de datos y cobertura de valores no nulos.</div>


In [ ]:
df2_raw.info(verbose=True, show_counts=True)

### 1.3 Preparación inicial

Se eliminan las columnas que no aportan al proceso de segmentación:
- **Identificadores de negocio:** `id_trato`, `person_id`, `user_id`.
- **Variables CSI y NPS:** información post-compra con missingness extremo en leads no convertidos.
- **Fecha de creación del trato.**
- **Variables de referidos y contexto de atención:** canal de anuencia, lugar de trabajo, lugar de vivienda, opciones visitadas, sector referidor, campañas y fuente de datos.
- **Identificadores geográficos y de proyecto:** `proyecto`, `municipio_proyecto`, `zona_proyecto`, `subzona_proyecto`.
- **Variables de producto redundantes o no discriminantes:** `m2_maximo`, `precio_minimo`, `precio_maximo`, `precio_m2_promedio` (redundantes con `m2_minimo` y `precio_promedio`), `rango_edad_modelo` (clasificación interna) y `campania_audiencia` (segmentación de pauta, no atributo del lead).

Se retienen `m2_minimo` y `precio_promedio` como los mejores proxies del segmento de producto de interés del lead: representan el tamaño mínimo disponible y el precio de mercado del proyecto al que se acercó, y cuentan con cobertura completa en el dataset. Su correlación alta es esperada y se evalúa en la sección 3.4.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Elimina identificadores y variables fuera del alcance del clustering, conservando únicamente campos útiles para el análisis.</div>


In [ ]:
id_cols_2      = ['id_trato', 'person_id', 'user_id']
csi_nps_cols_2 = [c for c in df2_raw.columns
                  if c.startswith('csi_') or c.startswith('nps_')]
date_cols_2    = ['fecha_creacion_trato']
ref_cols_2     = ['sector_referidor', 'campania', 'fuente_pt2', 'fuente_dataset',
                  'proyecto_fuera_alcance', 'canal_de_anuencia',
                  'lugar_trabajo', 'lugar_vivienda', 'opciones_visitadas']
proj_cols_2    = ['proyecto', 'municipio_proyecto', 'zona_proyecto', 'subzona_proyecto',
                  'rango_edad_modelo', 'campania_audiencia',
                  'm2_maximo', 'precio_minimo', 'precio_maximo', 'precio_m2_promedio']

to_drop_2 = set(id_cols_2 + csi_nps_cols_2
                + date_cols_2 + ref_cols_2 + proj_cols_2)
to_drop_2 = [c for c in to_drop_2 if c in df2_raw.columns]

df2 = df2_raw.drop(columns=to_drop_2).copy()
print(f'Columnas tras limpieza: {df2.shape[1]} ({len(to_drop_2)} eliminadas)')
print(sorted(df2.columns.tolist()))

### 1.4 Análisis exploratorio del universo de clustering


#### 1.4.1 Análisis de missingness

Se examina la proporción de valores faltantes en cada variable del conjunto de leads calificados. A diferencia de la población completa analizada en la Fase 1, se espera que las variables demográficas y socioeconómicas presenten tasas de missingness considerablemente menores, lo que justifica la restricción del análisis a este subconjunto.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
miss2 = (df2.isnull().sum() / len(df2) * 100).sort_values(ascending=False)
miss2 = miss2[miss2 > 0]

fig, ax = plt.subplots(figsize=(10, max(4, len(miss2) * 0.35)))
miss2.sort_values().plot(kind='barh', ax=ax, color='#E07B54')
ax.axvline(50, color='red', linestyle='--', lw=1, label='50 %')
ax.set_xlabel('% valores faltantes')
ax.set_title('Missingness por variable — Leads calificados')
ax.legend()
plt.tight_layout()
plt.show()

print(miss2.to_frame('% missing').to_string())

#### 1.4.2 Eliminación adicional por missingness extremo

A partir del análisis de missingness anterior, se identificó un segundo grupo de variables que, aunque no fueron eliminadas en la preparación inicial, presentan una proporción extremadamente alta de valores faltantes. Estas variables corresponden principalmente a información de parqueos, bodegas, apartamentos, referidos o atributos muy específicos del proceso comercial.

Dado que su nivel de completitud es insuficiente para aportar valor al análisis de segmentación y su imputación introduciría ruido artificial en el modelo, se decidió excluirlas antes de continuar con el análisis exploratorio.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Excluye las variables con missingness extremo y verifica las dimensiones del conjunto de datos resultante.</div>


In [ ]:
high_missing_cols_2 = [
    'ubicacion_parqueos_label',
    'parqueo_1_label',
    'ingreso_crm_label',
    'parqueo_2_label',
    'ubicacion_bodega_label',
    'proyref_label',
    'bodega_asignada_label',
    'apto_referidor_label',
    'razon_inversion_label',
    'parqueo_3_label',
    'cantidad_parqueos_label',
    'tipo_modelo_label',
    'modelo_label',
    'apartamento_no_label',
    'tipovisita_label',
    'torre_referidora_label'
]

high_missing_cols_2 = [
    c for c in high_missing_cols_2
    if c in df2.columns
]

df2 = df2.drop(columns=high_missing_cols_2).copy()

print(f'Columnas eliminadas por missingness extremo: {len(high_missing_cols_2)}')
print(high_missing_cols_2)

print(f'\nColumnas restantes tras eliminación adicional: {df2.shape[1]}')
print(sorted(df2.columns.tolist()))

#### 1.4.3 Missingness posterior a la eliminación adicional

Después de eliminar las variables con missingness extremo, se vuelve a calcular la proporción de valores faltantes para verificar la estructura final del conjunto de datos antes de continuar con el análisis exploratorio.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
miss2_clean = (
    df2.isnull().sum()
    .div(len(df2))
    .mul(100)
    .sort_values(ascending=False)
)

miss2_clean = miss2_clean[miss2_clean > 0]

fig, ax = plt.subplots(figsize=(10, max(4, len(miss2_clean) * 0.35)))

miss2_clean.sort_values().plot(
    kind='barh',
    ax=ax,
    color='#E07B54'
)

ax.axvline(50, color='red', linestyle='--', lw=1, label='50 %')
ax.set_xlabel('% valores faltantes')
ax.set_title('Missingness posterior a eliminación adicional — Leads calificados')
ax.legend()

plt.tight_layout()
plt.show()

print(miss2_clean.to_frame('% missing').to_string())

#### 1.4.4 Definición del universo final para clustering

El análisis de missingness (§ 3.1 a 3.1.2) evidencia que incluso en la subpoblación de leads calificados varias variables clave —especialmente las demográficas y socioeconómicas— presentan tasas de ausencia moderadas. Aplicar K-Means sobre el total de leads calificados implicaría que una parte considerable de los clusters quedaría determinada por valores imputados antes que por información real del perfil del lead.

Para evitar este problema, se define un **universo analítico** restringido a los registros con completitud suficiente en el conjunto de variables clave del modelo. Este criterio:

- No elimina registros de forma arbitraria, sino que establece un umbral explícito y reproducible (`COMPLETENESS_THRESHOLD`, definido en la sección de configuración).
- Garantiza que los clusters representen perfiles reales, no artefactos derivados de imputación masiva.
- Reduce la imputación residual en § 5 a un porcentaje menor de observaciones, preservando la integridad del modelo.
- El universo resultante (`df2_cluster_base`) constituye la base de todo el flujo de preprocesamiento y clustering a partir de este punto.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Aplica el criterio de completitud y construye el universo analítico que se utilizará en las etapas de clustering.</div>


In [ ]:
# Variables clave para evaluar la completitud del perfil del lead
KEY_VARS_COMPLETENESS = [
    'edad', 'sexo', 'estado_civil', 'nucleo_familiar',
    'participantes_en_credito', 'ingreso_familiar',
    'razon_de_compra', 'tipo_unidad', 'canal_de_contacto',
    'campania_tipo', 'campania_plataforma',
    'precio_promedio', 'm2_minimo',
]
KEY_VARS_COMPLETENESS = [c for c in KEY_VARS_COMPLETENESS if c in df2.columns]
n_key = len(KEY_VARS_COMPLETENESS)

# Completitud por registro
df2['n_vars_completas'] = df2[KEY_VARS_COMPLETENESS].notna().sum(axis=1)
df2['pct_completitud']  = df2['n_vars_completas'] / n_key

print(f'Variables clave evaluadas ({n_key}): {KEY_VARS_COMPLETENESS}')
print('\nEstadísticos de completitud por registro:')
print(df2['pct_completitud'].describe().map('{:.3f}'.format).to_string())

# Histograma + conteo por rangos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df2['pct_completitud'], bins=30, color='#5B8DB8', edgecolor='white')
axes[0].axvline(COMPLETENESS_THRESHOLD, color='red', linestyle='--', lw=1.5,
                label=f'Umbral: {COMPLETENESS_THRESHOLD:.0%}')
axes[0].set(xlabel='% de variables clave completas', ylabel='Registros',
            title='Distribución de completitud del perfil del lead')
axes[0].legend()

rangos = pd.cut(df2['pct_completitud'],
                bins=[0, 0.40, 0.55, 0.70, 0.85, 1.01],
                labels=['< 40 %', '40–55 %', '55–70 %', '70–85 %', '≥ 85 %'],
                right=False)
conteo_rangos = rangos.value_counts().sort_index()
conteo_rangos.plot(kind='barh', ax=axes[1], color='#E07B54', edgecolor='white')
for i, (label, val) in enumerate(conteo_rangos.items()):
    pct_r = val / len(df2) * 100
    axes[1].text(val + 30, i, f'{val:,}  ({pct_r:.1f} %)', va='center', fontsize=9)
axes[1].set(xlabel='Registros', title='Registros por rango de completitud')

plt.tight_layout()
plt.show()

# Definición del universo analítico
mask_completo    = df2['pct_completitud'] >= COMPLETENESS_THRESHOLD
df2_cluster_base = df2.loc[mask_completo].drop(
    columns=['n_vars_completas', 'pct_completitud']
).copy()
df2.drop(columns=['n_vars_completas', 'pct_completitud'], inplace=True)

n_total   = len(df2)
n_cluster = len(df2_cluster_base)
print(f'\nLeads calificados totales              : {n_total:,}')
print(f'Universo de clustering (≥ {COMPLETENESS_THRESHOLD:.0%} completitud): '
      f'{n_cluster:,}  ({n_cluster / n_total:.1%})')
print(f'Excluidos por completitud insuficiente : {n_total - n_cluster:,}')
print(f'\nShape df2_cluster_base: {df2_cluster_base.shape}')

#### 1.4.5 Distribución de variables numéricas


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Grafica las distribuciones numéricas para revisar asimetrías, concentración y posibles valores atípicos.</div>


In [ ]:
num_cols2_eda = [c for c in df2_cluster_base.select_dtypes(include='number').columns
                 if c != 'target_won']

ncols = 3
nrows = -(-len(num_cols2_eda) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(num_cols2_eda):
    df2_cluster_base[col].dropna().hist(bins=40, ax=axes[i], color='#5B8DB8', edgecolor='white')
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de variables numéricas — Universo de clustering', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

#### 1.4.6 Distribución de variables categóricas


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica las categorías más frecuentes para conocer la composición de las variables cualitativas.</div>


In [ ]:
cat_cols2_eda = [c for c in df2_cluster_base.select_dtypes(include='object').columns
                 if df2_cluster_base[c].nunique() <= 30]

for col in cat_cols2_eda:
    vc = df2_cluster_base[col].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(9, 3))
    vc.sort_values().plot(kind='barh', ax=ax, color='#5B8DB8', edgecolor='white')
    ax.set_title(f'{col}  (n_unique={df2_cluster_base[col].nunique()})')
    plt.tight_layout()
    plt.show()

#### 1.4.7 Correlaciones entre variables numéricas


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula la matriz de correlaciones y la representa como mapa de calor para detectar relaciones y redundancias entre variables numéricas.</div>


In [ ]:
num_corr2 = df2_cluster_base.select_dtypes(include='number').drop(columns=['target_won'], errors='ignore')
corr2 = num_corr2.corr()

mask2 = np.triu(np.ones_like(corr2, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr2, mask=mask2, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, square=True)
ax.set_title('Correlación entre variables numéricas — Universo de clustering')
plt.tight_layout()
plt.show()

### 1.5 Selección de variables

---

**Variables incluidas (13):** `edad`, `sexo`, `estado_civil`, `nucleo_familiar`, `ingreso_familiar`, `participantes_en_credito`, `tipo_unidad`, `precio_promedio`, `m2_minimo`, `razon_de_compra`, `canal_de_contacto`, `campania_tipo`, `campania_plataforma`.

**Objetivo de este experimento:** construir una segmentación que integre tanto el perfil demográfico y socioeconómico del lead como el canal y la estrategia de adquisición a través del cual llegó al CRM. Los resultados sirven como línea base para la comparación con el Experimento 2.


#### 1.5.1 Variables incluidas y excluidas

Se seleccionan trece variables con cobertura suficiente en la subpoblación de leads calificados, organizadas en cinco dimensiones de análisis. La variable `target_won` se excluye del proceso de clustering y se reserva exclusivamente para la validación externa en la etapa de perfilamiento.

**Variables excluidas con cobertura insuficiente:**
- `tiene_hijos` (≈ 64 % de missingness): su imputación generaría una categoría `'Desconocido'` predominante que distorsionaría la segmentación sin aportar señal real.
- `cuota_deseada_label` (≈ 74 % de missingness): misma limitación; además es información declarada voluntariamente solo por una minoría de leads.

**Decisión sobre `m2_minimo` y `precio_promedio`:** ambas variables presentan correlación alta entre sí (ver sección 3.4). Se retienen deliberadamente porque capturan dimensiones complementarias del producto de interés del lead —tamaño mínimo del inmueble y valor de mercado del proyecto— y su cobertura es completa. En K-Means estandarizado, la redundancia parcial entre dos variables da mayor peso a esa dimensión en el espacio de features, lo cual es coherente con la relevancia del segmento de producto para la segmentación.

| Grupo | Variables |
|-------|-----------|
| Demográfico | `edad`, `sexo`, `estado_civil`, `nucleo_familiar` |
| Socioeconómico | `ingreso_familiar`, `participantes_en_credito` |
| Preferencia de producto | `tipo_unidad`, `precio_promedio`, `m2_minimo` |
| Comportamental | `canal_de_contacto`, `razon_de_compra` |
| Marketing | `campania_tipo`, `campania_plataforma` |


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
VARS_NUM_F2 = [
    'edad',
    'nucleo_familiar',
    'participantes_en_credito',
    'm2_minimo',
    'precio_promedio',
]

VARS_CAT_F2 = [
    'sexo',
    'estado_civil',
    'ingreso_familiar',
    'tipo_unidad',
    'canal_de_contacto',
    'razon_de_compra',
    'campania_tipo',
    'campania_plataforma',
]

VARS_NUM_F2 = [c for c in VARS_NUM_F2 if c in df2_cluster_base.columns]
VARS_CAT_F2 = [c for c in VARS_CAT_F2 if c in df2_cluster_base.columns]
ALL_VARS_F2 = VARS_NUM_F2 + VARS_CAT_F2

df2_feat = df2_cluster_base[ALL_VARS_F2 + ['target_won']].copy()

print(f'Numéricas   ({len(VARS_NUM_F2)}): {VARS_NUM_F2}')
print(f'Categóricas ({len(VARS_CAT_F2)}): {VARS_CAT_F2}')
print(f'Shape: {df2_feat.shape}')
df2_feat.head()

#### 1.5.2 Validación y armonización de tipos de datos

Antes de cualquier preprocesamiento se inspecciona el tipo de almacenamiento real de cada variable. Para variables declaradas como numéricas que se encuentran almacenadas como cadena de texto, se evalúa la tasa de parseo **entre los valores no nulos**: si al menos el 90 % de los registros con información son convertibles a número, la variable se convierte a tipo numérico; en caso contrario, se reclasifica como categórica.

Esta estrategia es más robusta que evaluar la tasa sobre el total de filas, dado que el missingness no penaliza variables que son genuinamente numéricas pero con alta proporción de ausencia. Las listas `VARS_NUM_F2`, `VARS_CAT_F2` y `ALL_VARS_F2` se actualizan automáticamente en consecuencia.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
_num_f2 = list(VARS_NUM_F2)
_cat_f2 = list(VARS_CAT_F2)
rows_f2 = []

for col in list(ALL_VARS_F2):
    conceptual  = 'numérica'  if col in _num_f2 else 'categórica'
    orig_dtype  = str(df2_feat[col].dtype)
    final_dtype = orig_dtype
    action      = 'mantenida'

    if col in _num_f2:
        if pd.api.types.is_numeric_dtype(df2_feat[col]):
            pass  # ya es numérico
        else:
            non_null  = df2_feat[col].notna()
            conv_nn   = pd.to_numeric(df2_feat[col][non_null], errors='coerce')
            p_rate    = conv_nn.notna().mean()
            if p_rate >= PARSE_RATE_NON_NULL:
                df2_feat[col] = pd.to_numeric(df2_feat[col], errors='coerce')
                action        = f'convertida (parse {p_rate:.0%} s/no nulos)'
                final_dtype   = str(df2_feat[col].dtype)
            else:
                _num_f2.remove(col)
                _cat_f2.append(col)
                action = f'→ categórica (parse {p_rate:.0%} s/no nulos)'
    else:
        if pd.api.types.is_numeric_dtype(df2_feat[col]):
            df2_feat[col] = df2_feat[col].astype(str).replace('nan', np.nan)
            action        = 'convertida a string'
            final_dtype   = str(df2_feat[col].dtype)

    rows_f2.append({'variable': col, 'tipo_conceptual': conceptual,
                    'dtype_original': orig_dtype, 'dtype_final': final_dtype,
                    'acción': action})

VARS_NUM_F2 = _num_f2
VARS_CAT_F2 = _cat_f2
ALL_VARS_F2 = VARS_NUM_F2 + VARS_CAT_F2

display(pd.DataFrame(rows_f2))
print(f'\nNuméricas   ({len(VARS_NUM_F2)}): {VARS_NUM_F2}')
print(f'Categóricas ({len(VARS_CAT_F2)}): {VARS_CAT_F2}')

#### 1.5.3 Tratamiento de valores faltantes

El filtro de completitud aplicado en § 3.X garantiza que cada registro del universo de clustering cubre al menos el `COMPLETENESS_THRESHOLD` de las variables clave. Como resultado, la imputación se reduce a valores faltantes residuales: aquellos presentes en variables seleccionadas que no formaron parte del criterio de completitud, o cuya ausencia es puntual dentro del universo filtrado.

La estrategia de imputación es diferenciada por tipo de variable:

- **Variables numéricas:** imputación por la mediana, por su robustez frente a valores extremos. Se aplica únicamente cuando el missingness residual es bajo o moderado; si una variable aún presenta alta tasa de ausencia, se reevalúa su inclusión.
- **Variables categóricas con < 50 % de missingness:** imputación por la moda.
- **Variables categóricas con ≥ 50 % de missingness:** asignación de la categoría `'Desconocido'`. Dado el filtro previo, se espera que este caso sea infrecuente; su ocurrencia señalaría que la variable tiene baja cobertura incluso dentro del universo analítico y debería reconsiderarse su inclusión.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
print('Missingness en variables seleccionadas — leads calificados:')
m2v = df2_feat[ALL_VARS_F2].isnull().sum()
print(pd.DataFrame({'n': m2v, '%': (m2v / len(df2_feat) * 100).round(1)})
      .loc[m2v > 0].sort_values('%', ascending=False).to_string())

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
for col in VARS_NUM_F2:
    n = df2_feat[col].isnull().sum()
    if n > 0:
        med = df2_feat[col].median()
        df2_feat[col] = df2_feat[col].fillna(med)
        print(f'  [num] {col}: {n:,} → mediana={med:.2f}')

for col in VARS_CAT_F2:
    n = df2_feat[col].isnull().sum()
    if n == 0:
        continue
    pct = n / len(df2_feat)
    if pct >= 0.50:
        df2_feat[col] = df2_feat[col].fillna('Desconocido')
        print(f'  [cat] {col}: {pct:.1%} faltante → "Desconocido"')
    else:
        mode_val = df2_feat[col].mode()[0]
        df2_feat[col] = df2_feat[col].fillna(mode_val)
        print(f'  [cat] {col}: {n:,} → moda="{mode_val}"')

print(f'\nMissing restante: {df2_feat[ALL_VARS_F2].isnull().sum().sum()}')

#### 1.5.4 Tratamiento de valores atípicos

Se aplica winsorización con un factor IQR × 3. Este criterio conservador recorta únicamente los extremos más severos, sin eliminar variabilidad legítima de la distribución. Su uso en el contexto de K-Means está justificado por la sensibilidad del algoritmo a valores extremos, que pueden desplazar los centroides de forma artificial y producir clusters de tamaño muy desigual.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define o aplica el tratamiento de valores atípicos para reducir su influencia sobre las distancias y los centroides.</div>


In [ ]:
def winsorize_iqr_f2(series, factor=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - factor * iqr, q3 + factor * iqr
    return series.clip(lower=lo, upper=hi), lo, hi

print('Límites de winsorización (IQR × 3):')
for col in VARS_NUM_F2:
    df2_feat[col], lo, hi = winsorize_iqr_f2(df2_feat[col])
    print(f'  {col:30s}: [{lo:.2f}, {hi:.2f}]')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define o aplica el tratamiento de valores atípicos para reducir su influencia sobre las distancias y los centroides.</div>


In [ ]:
n_num_f2 = len(VARS_NUM_F2)
fig, axes = plt.subplots(1, n_num_f2, figsize=(3.5 * n_num_f2, 4))
if n_num_f2 == 1:
    axes = [axes]
for ax, col in zip(axes, VARS_NUM_F2):
    ax.boxplot(df2_feat[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#5B8DB8', alpha=0.7))
    ax.set_title(col, fontsize=10)
    ax.set_xticks([])
plt.suptitle('Boxplots post-winsorización — Leads calificados', fontsize=12)
plt.tight_layout()
plt.show()

#### 1.5.5 Codificación y escalamiento

El proceso de preparación final de las variables sigue el siguiente orden:

1. **`ingreso_familiar`:** se extrae el punto medio del rango declarado (p. ej., `'Q34k A 37.9k'` → 35 950 Q). Los valores `'Desconocido'` o nulos se imputan con la mediana de los rangos válidos.
2. **Resto de variables categóricas:** codificación *One-Hot Encoding* con `drop_first=True` para evitar multicolinealidad perfecta.
3. **Todas las variables:** estandarización mediante `StandardScaler` (media 0, desviación estándar 1). Este paso es indispensable en K-Means, ya que el algoritmo opera sobre distancias euclidianas y, sin escalamiento, las variables con mayor rango numérico dominarían la segmentación.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
def parse_income_f2(s):
    """Punto medio del rango de ingreso: 'Q34k A 37.9k' → (34 000 + 37 900) / 2."""
    if pd.isna(s) or str(s).strip() in ('', 'Desconocido'):
        return np.nan
    nums = re.findall(r'(\d+(?:\.\d+)?)\s*k', str(s).lower())
    if not nums:
        return np.nan
    return sum(float(n) * 1_000 for n in nums) / len(nums)

df2_enc = df2_feat[ALL_VARS_F2].copy()

if 'ingreso_familiar' in VARS_CAT_F2:
    df2_enc['ingreso_numerico'] = df2_enc['ingreso_familiar'].apply(parse_income_f2)
    df2_enc['ingreso_numerico'].fillna(df2_enc['ingreso_numerico'].median(), inplace=True)
    VARS_NUM_F2_FINAL = VARS_NUM_F2 + ['ingreso_numerico']
    VARS_CAT_F2_OHE   = [c for c in VARS_CAT_F2 if c != 'ingreso_familiar']
else:
    VARS_NUM_F2_FINAL = VARS_NUM_F2
    VARS_CAT_F2_OHE   = VARS_CAT_F2

df2_ohe = pd.get_dummies(df2_enc[VARS_CAT_F2_OHE], drop_first=True, dtype=float)

X2_pre = pd.concat(
    [df2_enc[VARS_NUM_F2_FINAL].reset_index(drop=True),
     df2_ohe.reset_index(drop=True)],
    axis=1
)

feature_names_f2 = X2_pre.columns.tolist()
print(f'Features antes de escalar: {X2_pre.shape[1]}')
print(feature_names_f2)

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Escala las variables para que sus rangos sean comparables y ninguna domine el cálculo de distancias por su unidad de medida.</div>


In [ ]:
scaler_f2 = StandardScaler()
X2_scaled = scaler_f2.fit_transform(X2_pre)
print(f'Dimensiones post-escalamiento: {X2_scaled.shape}')

### 1.6 Selección del número óptimo de clusters

Se evalúan cuatro métricas complementarias sobre una muestra aleatoria de `SAMPLE_K_SEL_F2` registros para un rango de k entre 2 y 10. La decisión final sobre el valor de k debe integrar tanto los criterios cuantitativos como la interpretabilidad de negocio de los segmentos resultantes.

| Métrica | Criterio | Interpretación |
|---------|----------|----------------|
| Inercia (codo) | Punto de inflexión | Reducción marginal de la compactness |
| Silhouette | ↑ mejor (máx. 1) | Cohesión intra-cluster y separación inter-cluster |
| Davies-Bouldin | ↓ mejor (mín. 0) | Dispersión interna relativa a distancia entre clusters |
| Calinski-Harabasz | ↑ mejor | Razón de varianza inter-cluster a intra-cluster |


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Evalúa distintos valores de k mediante Silhouette, Davies–Bouldin, Calinski–Harabasz e inercia.</div>


In [ ]:
np.random.seed(RANDOM_STATE_F2)
sample_idx2 = np.random.choice(
    len(X2_scaled), size=min(SAMPLE_K_SEL_F2, len(X2_scaled)), replace=False
)
X2_sample = X2_scaled[sample_idx2]

inertias_f2, silhs_f2, db_f2, ch_f2 = [], [], [], []

print(f'Evaluando k ∈ {list(K_RANGE_F2)} sobre {len(X2_sample):,} filas...')
for k in K_RANGE_F2:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE_F2, n_init=10)
    lbls = km.fit_predict(X2_sample)
    inertias_f2.append(km.inertia_)
    silhs_f2.append(silhouette_score(X2_sample, lbls,
                    sample_size=min(5_000, len(X2_sample)), random_state=RANDOM_STATE_F2))
    db_f2.append(davies_bouldin_score(X2_sample, lbls))
    ch_f2.append(calinski_harabasz_score(X2_sample, lbls))
    print(f'  k={k:2d} | Silhouette={silhs_f2[-1]:.3f} | '
          f'DB={db_f2[-1]:.3f} | CH={ch_f2[-1]:.0f}')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Evalúa distintos valores de k mediante Silhouette, Davies–Bouldin, Calinski–Harabasz e inercia.</div>


In [ ]:
k_vals_f2 = list(K_RANGE_F2)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(k_vals_f2, inertias_f2, 'o-', color='#5B8DB8', lw=2, ms=7)
axes[0,0].set(xlabel='k', ylabel='Inercia', title='Método del Codo')
axes[0,0].grid(alpha=0.4)

axes[0,1].plot(k_vals_f2, silhs_f2, 's-', color='#E07B54', lw=2, ms=7)
axes[0,1].set(xlabel='k', ylabel='Silhouette', title='Silhouette (↑ mejor)')
axes[0,1].grid(alpha=0.4)

axes[1,0].plot(k_vals_f2, db_f2, '^-', color='#6BAF72', lw=2, ms=7)
axes[1,0].set(xlabel='k', ylabel='Davies-Bouldin', title='Davies-Bouldin (↓ mejor)')
axes[1,0].grid(alpha=0.4)

axes[1,1].plot(k_vals_f2, ch_f2, 'D-', color='#9B59B6', lw=2, ms=7)
axes[1,1].set(xlabel='k', ylabel='Calinski-Harabasz', title='Calinski-Harabasz (↑ mejor)')
axes[1,1].grid(alpha=0.4)

plt.suptitle('Métricas de selección de k — Leads calificados', fontsize=14)
plt.tight_layout()
plt.show()

print(f'k con mejor Silhouette : {k_vals_f2[int(np.argmax(silhs_f2))]}')
print(f'k con mejor DB         : {k_vals_f2[int(np.argmin(db_f2))]}')
print(f'k con mejor CH         : {k_vals_f2[int(np.argmax(ch_f2))]}')
print(f'K_OPTIMO_F2 actual     : {K_OPTIMO_F2}  (ajustar en celda de configuración)')

### 1.7 Entrenamiento del modelo K-Means final

Se entrena K-Means sobre la totalidad de los leads calificados con `n_init=20` inicializaciones aleatorias para garantizar la estabilidad de los centroides. Se emplea el algoritmo de Lloyd con un máximo de 500 iteraciones.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Entrena el modelo K-Means con la configuración seleccionada y asigna un cluster a cada observación.</div>


In [ ]:
km_f2 = KMeans(
    n_clusters=K_OPTIMO_F2,
    random_state=RANDOM_STATE_F2,
    n_init=20,
    max_iter=500,
    algorithm='lloyd'
)

print(f'Entrenando K-Means k={K_OPTIMO_F2} sobre {len(X2_scaled):,} leads calificados...')
cluster_labels_f2 = km_f2.fit_predict(X2_scaled)
df2_feat = df2_feat.reset_index(drop=True)
df2_feat['cluster'] = cluster_labels_f2

dist_f2 = pd.Series(cluster_labels_f2).value_counts().sort_index()
print('\nDistribución de clusters:')
for k, n in dist_f2.items():
    print(f'  Cluster {k}: {n:>7,}  ({n/len(cluster_labels_f2):.1%})')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula las métricas finales de cohesión y separación sobre una muestra reproducible del modelo ajustado.</div>


In [ ]:
labels_sample_f2 = cluster_labels_f2[sample_idx2]
sil_f2_final = silhouette_score(X2_sample, labels_sample_f2,
                                sample_size=min(5_000, len(X2_sample)),
                                random_state=RANDOM_STATE_F2)
db_f2_final  = davies_bouldin_score(X2_sample, labels_sample_f2)
ch_f2_final  = calinski_harabasz_score(X2_sample, labels_sample_f2)

print(f'Métricas del modelo final (k={K_OPTIMO_F2}):')
print(f'  Silhouette:        {sil_f2_final:.4f}')
print(f'  Davies-Bouldin:    {db_f2_final:.4f}')
print(f'  Calinski-Harabasz: {ch_f2_final:.1f}')

### 1.8 Visualización — Proyección PCA

#### 1.8.1 Dispersión bidimensional por cluster

Se aplica Análisis de Componentes Principales (PCA) para proyectar el espacio de features en dos dimensiones. Esta representación no constituye el espacio en que opera K-Means, sino que funciona como herramienta de diagnóstico visual para evaluar la separabilidad y cohesión de los clusters obtenidos.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Proyecta el espacio de variables a dos componentes principales para facilitar la inspección visual de los clusters.</div>


In [ ]:
pca_f2 = PCA(n_components=2, random_state=RANDOM_STATE_F2)
X2_pca = pca_f2.fit_transform(X2_scaled)
var_exp_f2 = pca_f2.explained_variance_ratio_
print(f'Varianza explicada: PC1={var_exp_f2[0]:.1%}  PC2={var_exp_f2[1]:.1%}  '
      f'Total={var_exp_f2.sum():.1%}')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Dibuja la proyección PCA coloreada por cluster y marca los centroides como diagnóstico visual de separación.</div>


In [ ]:
colors_f2 = sns.color_palette('Set2', K_OPTIMO_F2)
N_PLOT_F2 = min(3_000, len(X2_scaled))

fig, ax = plt.subplots(figsize=(12, 8))
for k in range(K_OPTIMO_F2):
    idx_k = np.where(cluster_labels_f2 == k)[0][:N_PLOT_F2]
    ax.scatter(X2_pca[idx_k, 0], X2_pca[idx_k, 1],
               c=[colors_f2[k]], label=f'Cluster {k}  (n={dist_f2[k]:,})',
               alpha=0.45, s=15, linewidths=0)

centroids_pca_f2 = pca_f2.transform(km_f2.cluster_centers_)
ax.scatter(centroids_pca_f2[:, 0], centroids_pca_f2[:, 1],
           c='black', marker='X', s=250, zorder=10, label='Centroides')
for k, (cx, cy) in enumerate(centroids_pca_f2):
    ax.annotate(str(k), (cx, cy), xytext=(8, 4),
                textcoords='offset points', fontsize=12, fontweight='bold')

ax.set_xlabel(f'PC1 ({var_exp_f2[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({var_exp_f2[1]:.1%})', fontsize=12)
ax.set_title(f'K-Means k={K_OPTIMO_F2} — Proyección PCA 2D (Leads calificados)', fontsize=14)
ax.legend(markerscale=2, fontsize=10)
plt.tight_layout()
plt.show()

#### 1.8.2 Distribución de tamaños por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ejecuta el procesamiento correspondiente a «1.8.2 Distribución de tamaños por cluster» y produce los insumos o resultados utilizados por el siguiente paso.</div>


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(dist_f2.index, dist_f2.values, color=colors_f2,
              edgecolor='white', linewidth=1.5)
for bar, (k, n) in zip(bars, dist_f2.items()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + len(cluster_labels_f2) * 0.003,
            f'{n:,}\n({n/len(cluster_labels_f2):.1%})',
            ha='center', va='bottom', fontsize=10)
ax.set(xlabel='Cluster', ylabel='N leads',
       title='Distribución de tamaños por cluster — Leads calificados')
ax.set_xticks(range(K_OPTIMO_F2))
plt.tight_layout()
plt.show()

### 1.9 Perfilamiento de clusters

El perfilamiento describe las características centrales de cada cluster a partir de tres dimensiones: variables numéricas (medias comparadas entre grupos), variables categóricas (distribución de frecuencias relativas por cluster), y tasa de conversión (`target_won`), que actúa como validador externo de la coherencia de la segmentación sin haber participado en el proceso de entrenamiento.


#### 1.9.1 Variables numéricas — mapa de calor de medias por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume las variables originales por cluster para identificar las características que distinguen cada perfil.</div>


In [ ]:
profile_num_f2 = df2_feat.groupby('cluster')[VARS_NUM_F2].mean()
if 'ingreso_numerico' in df2_enc.columns:
    df2_feat['ingreso_numerico'] = df2_enc['ingreso_numerico'].values
    profile_num_f2['ingreso_numerico'] = (
        df2_feat.groupby('cluster')['ingreso_numerico'].mean()
    )

print('Media por cluster — variables numéricas:')
print(profile_num_f2.T.to_string())

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Estandariza los perfiles y genera un mapa de calor que permite comparar visualmente los clusters.</div>


In [ ]:
profile_norm_f2 = (profile_num_f2 - profile_num_f2.mean()) / profile_num_f2.std()

fig, ax = plt.subplots(
    figsize=(max(8, K_OPTIMO_F2 * 1.8), len(profile_num_f2.columns) * 0.7 + 2)
)
sns.heatmap(
    profile_norm_f2.T,
    annot=profile_num_f2.T.round(1), fmt='g',
    cmap='RdBu_r', center=0, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Z-score respecto a media global'}
)
ax.set_title('Perfil de clusters — variables numéricas (Leads calificados)', fontsize=13)
ax.set_xlabel('Cluster')
plt.tight_layout()
plt.show()

#### 1.9.2 Variables categóricas — distribución por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica las categorías más frecuentes para conocer la composición de las variables cualitativas.</div>


In [ ]:
VARS_CAT_PROF_F2 = [c for c in VARS_CAT_F2
                    if c in df2_feat.columns and c != 'ingreso_familiar']

for col in VARS_CAT_PROF_F2:
    top_cats = df2_feat[col].value_counts().head(8).index
    ct = (
        df2_feat.groupby('cluster')[col]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex(columns=top_cats, fill_value=0) * 100
    )
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ct.T.plot(kind='bar', ax=ax, color=colors_f2, edgecolor='white', width=0.8)
    ax.set_title(f'{col} por cluster (%)', fontsize=12)
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Cluster', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

#### 1.9.3 Tasa de conversión por cluster

La variable `target_won` no participó en el proceso de clustering. Su distribución por cluster constituye una validación externa de la coherencia de la segmentación: diferencias significativas en la tasa de conversión entre grupos sugieren que el modelo captura patrones asociados al comportamiento de compra, más allá de las similitudes en las variables de entrada.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Mide la conversión por cluster como validación externa; esta variable no interviene en la formación de los segmentos.</div>


In [ ]:
conv_f2 = df2_feat.groupby('cluster')['target_won'].agg(['mean', 'sum', 'count'])
conv_f2.columns = ['tasa_conversion', 'convertidos', 'total']
conv_f2['tasa_%'] = (conv_f2['tasa_conversion'] * 100).round(2)
print(conv_f2[['total', 'convertidos', 'tasa_%']].to_string())

global_rate_f2 = df2_feat['target_won'].mean() * 100
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(conv_f2.index, conv_f2['tasa_%'], color=colors_f2,
              edgecolor='white', linewidth=1.5)
ax.axhline(global_rate_f2, color='red', linestyle='--', lw=1.2,
           label=f'Tasa global: {global_rate_f2:.1f}%')
for bar, rate in zip(bars, conv_f2['tasa_%']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.2, f'{rate:.1f}%',
            ha='center', fontsize=10)
ax.set(xlabel='Cluster', ylabel='Tasa de conversión (%)',
       title='Tasa de conversión por cluster — Leads calificados')
ax.legend()
plt.tight_layout()
plt.show()

### 1.10 Interpretación de resultados y hallazgos

#### Hallazgos principales

- El modelo final se ajustó con k=4 y obtuvo Silhouette 0.1195, Davies–Bouldin 2.3931 y Calinski–Harabasz 726.6. La separación geométrica es débil y exige interpretar los segmentos con cautela.
- El cluster 3 concentra 6 115 de 11 964 registros (51.1 %), mientras el cluster 0 reúne solo 665 (5.6 %); por tanto, la solución presenta desbalance.
- El cluster 1 identifica el producto de mayor escala: precio promedio aproximado de Q191 421 y 45.6 m². Los clusters 2 y 3 se sitúan cerca de Q124 mil y 32 m².
- La conversión varía entre 1.05 % y 2.60 %. El cluster 3 presenta la tasa mayor y el cluster 0 la menor.

#### Implicaciones para el negocio

- El cluster 1 puede tratarse como un segmento de producto de mayor valor, pero su conversión de 2.30 % no es la más alta.
- El cluster 3 combina volumen y la mayor conversión observada, por lo que merece prioridad operativa, sujeta a validación temporal.
- Dada la baja separación, estos grupos funcionan mejor como línea base comparativa que como segmentación definitiva.

#### Limitaciones

- El k configurado no coincide con un ganador único de las métricas: Silhouette y CH favorecen k=2, mientras Davies–Bouldin favorece k=9.
- Las tasas de conversión son validación externa descriptiva; no prueban causalidad ni participaron en el entrenamiento.


### 1.11 Exportación de resultados

Los resultados del clustering se almacenan en archivos tabulares para su reutilización posterior en reportes, validaciones externas o integración con herramientas de negocio. Esta práctica facilita la trazabilidad del análisis y la comparación entre configuraciones.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Crea la carpeta de salida y exporta etiquetas, perfiles y métricas para reutilizarlos fuera del notebook.</div>


In [ ]:
import os

OUT_DIR = 'Output'
os.makedirs(OUT_DIR, exist_ok=True)

# Dataset de leads calificados con cluster asignado donde aplica.
# Los registros fuera del universo de clustering (completitud insuficiente)
# conservan cluster = NaN para trazabilidad.
df2_out = df2_raw.copy()
df2_out['cluster'] = np.nan
df2_out.loc[df2_cluster_base.index, 'cluster'] = cluster_labels_f2
out_path = os.path.join(OUT_DIR, 'leads_calificados_clustered.csv')
df2_out.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Dataset exportado: {out_path}')
print(f'  Registros con cluster asignado : {len(df2_cluster_base):,}')
print(f'  Leads calificados totales      : {len(df2_out):,}')

# Perfil numérico por cluster
profile_num_f2.to_csv(
    os.path.join(OUT_DIR, 'perfil_numerico_clusters.csv'), encoding='utf-8-sig'
)
print('Perfil numérico por cluster exportado.')

# Tasa de conversión por cluster
conv_f2.to_csv(
    os.path.join(OUT_DIR, 'conversion_por_cluster.csv'), encoding='utf-8-sig'
)
print('Tasa de conversión por cluster exportada.')

print('\nExportación completada.')

---

## 2. Experimento 2 — Segmentación de perfiles de cliente (sin variables de adquisición)

Se replica exactamente el flujo del Experimento 1 sobre el mismo universo analítico (`df2_cluster_base`), con una única diferencia en la selección de variables: se excluyen `canal_de_contacto`, `campania_tipo` y `campania_plataforma`.

La motivación es que estas variables caracterizan el **medio de adquisición del lead** —no al lead como comprador potencial— y su inclusión puede producir clusters diferenciados principalmente por el canal de captación antes que por el perfil del cliente. Al eliminarlas, los clusters se formarán únicamente a partir de características demográficas, socioeconómicas y de preferencia de producto, lo que se alinea directamente con el objetivo de segmentación de clientes de la tesis.

**Variables incluidas (10):** `edad`, `sexo`, `estado_civil`, `nucleo_familiar`, `ingreso_familiar`, `participantes_en_credito`, `tipo_unidad`, `precio_promedio`, `m2_minimo`, `razon_de_compra`.

**Variables excluidas respecto al Experimento 1:** `canal_de_contacto`, `campania_tipo`, `campania_plataforma`.

### 2.1 Configuración del experimento


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ejecuta el procesamiento correspondiente a «2.1 Configuración del experimento» y produce los insumos o resultados utilizados por el siguiente paso.</div>


## 2. Experimento 2 — Segmentación con perfil de cliente (sin variables de adquisición)

Se repite el protocolo del Experimento 1 sobre el mismo universo analítico, pero excluyendo las variables de canal y campaña para que la segmentación refleje perfiles de cliente en lugar de perfiles de adquisición.

### 2.1 Configuración del experimento

In [ ]:
K_OPTIMO_E2 = 7  # ajustar según los resultados de la sección 8 de este experimento

Se utilizan diez variables distribuidas en cuatro dimensiones de análisis. Se excluyen deliberadamente `canal_de_contacto`, `campania_tipo` y `campania_plataforma` para que el modelo capture perfiles de cliente en lugar de perfiles de adquisición.

| Grupo | Variables |
|-------|-----------|
| Demográfico | `edad`, `sexo`, `estado_civil`, `nucleo_familiar` |
| Socioeconómico | `ingreso_familiar`, `participantes_en_credito` |
| Preferencia de producto | `tipo_unidad`, `precio_promedio`, `m2_minimo` |
| Comportamental | `razon_de_compra` |


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
VARS_NUM_E2 = [
    'edad',
    'nucleo_familiar',
    'participantes_en_credito',
    'm2_minimo',
    'precio_promedio',
]

VARS_CAT_E2 = [
    'sexo',
    'estado_civil',
    'ingreso_familiar',
    'tipo_unidad',
    'razon_de_compra',
]

VARS_NUM_E2 = [c for c in VARS_NUM_E2 if c in df2_cluster_base.columns]
VARS_CAT_E2 = [c for c in VARS_CAT_E2 if c in df2_cluster_base.columns]
ALL_VARS_E2 = VARS_NUM_E2 + VARS_CAT_E2

df2_feat_E2 = df2_cluster_base[ALL_VARS_E2 + ['target_won']].copy()

print(f'Numéricas   ({len(VARS_NUM_E2)}): {VARS_NUM_E2}')
print(f'Categóricas ({len(VARS_CAT_E2)}): {VARS_CAT_E2}')
print(f'Shape: {df2_feat_E2.shape}')
df2_feat_E2.head()

### 2.2 Selección de variables

Misma estrategia que en el Experimento 1: verificación del tipo de almacenamiento real para variables declaradas como numéricas y conversión cuando el 90 % o más de los valores no nulos son parseables a número.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
_num_e2 = list(VARS_NUM_E2)
_cat_e2 = list(VARS_CAT_E2)
rows_e2 = []

for col in list(ALL_VARS_E2):
    conceptual  = 'numérica'  if col in _num_e2 else 'categórica'
    orig_dtype  = str(df2_feat_E2[col].dtype)
    final_dtype = orig_dtype
    action      = 'mantenida'

    if col in _num_e2:
        if pd.api.types.is_numeric_dtype(df2_feat_E2[col]):
            pass
        else:
            non_null = df2_feat_E2[col].notna()
            conv_nn  = pd.to_numeric(df2_feat_E2[col][non_null], errors='coerce')
            p_rate   = conv_nn.notna().mean()
            if p_rate >= PARSE_RATE_NON_NULL:
                df2_feat_E2[col] = pd.to_numeric(df2_feat_E2[col], errors='coerce')
                action      = f'convertida (parse {p_rate:.0%} s/no nulos)'
                final_dtype = str(df2_feat_E2[col].dtype)
            else:
                _num_e2.remove(col)
                _cat_e2.append(col)
                action = f'→ categórica (parse {p_rate:.0%} s/no nulos)'
    else:
        if pd.api.types.is_numeric_dtype(df2_feat_E2[col]):
            df2_feat_E2[col] = df2_feat_E2[col].astype(str).replace('nan', np.nan)
            action      = 'convertida a string'
            final_dtype = str(df2_feat_E2[col].dtype)

    rows_e2.append({'variable': col, 'tipo_conceptual': conceptual,
                    'dtype_original': orig_dtype, 'dtype_final': final_dtype,
                    'acción': action})

VARS_NUM_E2 = _num_e2
VARS_CAT_E2 = _cat_e2
ALL_VARS_E2 = VARS_NUM_E2 + VARS_CAT_E2

display(pd.DataFrame(rows_e2))
print(f'\nNuméricas   ({len(VARS_NUM_E2)}): {VARS_NUM_E2}')
print(f'Categóricas ({len(VARS_CAT_E2)}): {VARS_CAT_E2}')

### 2.3 Tratamiento de valores faltantes

Se aplica la misma estrategia diferenciada que en el Experimento 1. Al operar sobre el mismo universo analítico (`df2_cluster_base`) con un subconjunto reducido de variables, los patrones de missingness residual en las variables retenidas son idénticos a los del Experimento 1 para las variables compartidas.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
print('Missingness en variables seleccionadas — Experimento 2:')
m_e2 = df2_feat_E2[ALL_VARS_E2].isnull().sum()
print(pd.DataFrame({'n': m_e2, '%': (m_e2 / len(df2_feat_E2) * 100).round(1)})
      .loc[m_e2 > 0].sort_values('%', ascending=False).to_string())

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
for col in VARS_NUM_E2:
    n = df2_feat_E2[col].isnull().sum()
    if n > 0:
        med = df2_feat_E2[col].median()
        df2_feat_E2[col] = df2_feat_E2[col].fillna(med)
        print(f'  [num] {col}: {n:,} → mediana={med:.2f}')

for col in VARS_CAT_E2:
    n = df2_feat_E2[col].isnull().sum()
    if n == 0:
        continue
    pct = n / len(df2_feat_E2)
    if pct >= 0.50:
        df2_feat_E2[col] = df2_feat_E2[col].fillna('Desconocido')
        print(f'  [cat] {col}: {pct:.1%} faltante → "Desconocido"')
    else:
        mode_val = df2_feat_E2[col].mode()[0]
        df2_feat_E2[col] = df2_feat_E2[col].fillna(mode_val)
        print(f'  [cat] {col}: {n:,} → moda="{mode_val}"')

print(f'\nMissing restante: {df2_feat_E2[ALL_VARS_E2].isnull().sum().sum()}')

### 2.4 Tratamiento de valores atípicos

Se aplica winsorización con factor IQR × 3 sobre las variables numéricas, siguiendo el mismo criterio conservador del Experimento 1.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define o aplica el tratamiento de valores atípicos para reducir su influencia sobre las distancias y los centroides.</div>


In [ ]:
print('Límites de winsorización (IQR × 3) — Experimento 2:')
for col in VARS_NUM_E2:
    df2_feat_E2[col], lo, hi = winsorize_iqr_f2(df2_feat_E2[col])
    print(f'  {col:30s}: [{lo:.2f}, {hi:.2f}]')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define o aplica el tratamiento de valores atípicos para reducir su influencia sobre las distancias y los centroides.</div>


In [ ]:
n_num_e2 = len(VARS_NUM_E2)
fig, axes = plt.subplots(1, n_num_e2, figsize=(3.5 * n_num_e2, 4))
if n_num_e2 == 1:
    axes = [axes]
for ax, col in zip(axes, VARS_NUM_E2):
    ax.boxplot(df2_feat_E2[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#5B8DB8', alpha=0.7))
    ax.set_title(col, fontsize=10)
    ax.set_xticks([])
plt.suptitle('Boxplots post-winsorización — Experimento 2', fontsize=12)
plt.tight_layout()
plt.show()

### 2.5 Codificación y escalamiento

Mismo proceso que en el Experimento 1: conversión del ingreso familiar a valor numérico continuo mediante el punto medio del rango declarado, codificación OHE del resto de variables categóricas y estandarización mediante `StandardScaler`.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
df2_enc_E2 = df2_feat_E2[ALL_VARS_E2].copy()

if 'ingreso_familiar' in VARS_CAT_E2:
    df2_enc_E2['ingreso_numerico'] = df2_enc_E2['ingreso_familiar'].apply(parse_income_f2)
    df2_enc_E2['ingreso_numerico'].fillna(df2_enc_E2['ingreso_numerico'].median(), inplace=True)
    VARS_NUM_E2_FINAL = VARS_NUM_E2 + ['ingreso_numerico']
    VARS_CAT_E2_OHE   = [c for c in VARS_CAT_E2 if c != 'ingreso_familiar']
else:
    VARS_NUM_E2_FINAL = VARS_NUM_E2
    VARS_CAT_E2_OHE   = VARS_CAT_E2

df2_ohe_E2 = pd.get_dummies(df2_enc_E2[VARS_CAT_E2_OHE], drop_first=True, dtype=float)

X2_pre_E2 = pd.concat(
    [df2_enc_E2[VARS_NUM_E2_FINAL].reset_index(drop=True),
     df2_ohe_E2.reset_index(drop=True)],
    axis=1
)

feature_names_E2 = X2_pre_E2.columns.tolist()
print(f'Features antes de escalar — E2: {X2_pre_E2.shape[1]}')
print(feature_names_E2)

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Escala las variables para que sus rangos sean comparables y ninguna domine el cálculo de distancias por su unidad de medida.</div>


In [ ]:
scaler_E2    = StandardScaler()
X2_scaled_E2 = scaler_E2.fit_transform(X2_pre_E2)
print(f'Dimensiones post-escalamiento — E2: {X2_scaled_E2.shape}')

### 2.6 Selección del número óptimo de clusters

Se evalúan las cuatro métricas complementarias sobre una muestra de `SAMPLE_K_SEL_F2` registros para el rango de k ∈ [2, 10]. La eliminación de las variables de canal y campaña modifica el espacio de features y puede producir un k óptimo diferente al del Experimento 1.

| Métrica | Criterio | Interpretación |
|---------|----------|----------------|
| Inercia (codo) | Punto de inflexión | Reducción marginal de la compactness |
| Silhouette | ↑ mejor (máx. 1) | Cohesión intra-cluster y separación inter-cluster |
| Davies-Bouldin | ↓ mejor (mín. 0) | Dispersión interna relativa a distancia entre clusters |
| Calinski-Harabasz | ↑ mejor | Razón de varianza inter-cluster a intra-cluster |


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Evalúa distintos valores de k mediante Silhouette, Davies–Bouldin, Calinski–Harabasz e inercia.</div>


In [ ]:
np.random.seed(RANDOM_STATE_F2)
sample_idx_E2 = np.random.choice(
    len(X2_scaled_E2), size=min(SAMPLE_K_SEL_F2, len(X2_scaled_E2)), replace=False
)
X2_sample_E2 = X2_scaled_E2[sample_idx_E2]

inertias_E2, silhs_E2, db_E2, ch_E2 = [], [], [], []

print(f'Evaluando k ∈ {list(K_RANGE_F2)} sobre {len(X2_sample_E2):,} filas...')
for k in K_RANGE_F2:
    km_tmp = KMeans(n_clusters=k, random_state=RANDOM_STATE_F2, n_init=10)
    lbls   = km_tmp.fit_predict(X2_sample_E2)
    inertias_E2.append(km_tmp.inertia_)
    silhs_E2.append(silhouette_score(X2_sample_E2, lbls,
                    sample_size=min(5_000, len(X2_sample_E2)), random_state=RANDOM_STATE_F2))
    db_E2.append(davies_bouldin_score(X2_sample_E2, lbls))
    ch_E2.append(calinski_harabasz_score(X2_sample_E2, lbls))
    print(f'  k={k:2d} | Silhouette={silhs_E2[-1]:.3f} | '
          f'DB={db_E2[-1]:.3f} | CH={ch_E2[-1]:.0f}')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Evalúa distintos valores de k mediante Silhouette, Davies–Bouldin, Calinski–Harabasz e inercia.</div>


In [ ]:
k_vals_E2 = list(K_RANGE_F2)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(k_vals_E2, inertias_E2, 'o-', color='#5B8DB8', lw=2, ms=7)
axes[0,0].set(xlabel='k', ylabel='Inercia', title='Método del Codo — E2')
axes[0,0].grid(alpha=0.4)

axes[0,1].plot(k_vals_E2, silhs_E2, 's-', color='#E07B54', lw=2, ms=7)
axes[0,1].set(xlabel='k', ylabel='Silhouette', title='Silhouette (↑ mejor) — E2')
axes[0,1].grid(alpha=0.4)

axes[1,0].plot(k_vals_E2, db_E2, '^-', color='#6BAF72', lw=2, ms=7)
axes[1,0].set(xlabel='k', ylabel='Davies-Bouldin', title='Davies-Bouldin (↓ mejor) — E2')
axes[1,0].grid(alpha=0.4)

axes[1,1].plot(k_vals_E2, ch_E2, 'D-', color='#9B59B6', lw=2, ms=7)
axes[1,1].set(xlabel='k', ylabel='Calinski-Harabasz', title='Calinski-Harabasz (↑ mejor) — E2')
axes[1,1].grid(alpha=0.4)

plt.suptitle('Métricas de selección de k — Experimento 2', fontsize=14)
plt.tight_layout()
plt.show()

print(f'k con mejor Silhouette : {k_vals_E2[int(np.argmax(silhs_E2))]}')
print(f'k con mejor DB         : {k_vals_E2[int(np.argmin(db_E2))]}')
print(f'k con mejor CH         : {k_vals_E2[int(np.argmax(ch_E2))]}')
print(f'K_OPTIMO_E2 actual     : {K_OPTIMO_E2}  (ajustar en celda de configuración)')

### 2.7 Entrenamiento del modelo K-Means final

Se entrena K-Means con `n_init=20` inicializaciones sobre la totalidad del universo de clustering. El valor `K_OPTIMO_E2` puede diferir del Experimento 1 si las métricas indican un número distinto de clusters óptimo al excluir las variables de adquisición.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Entrena el modelo K-Means con la configuración seleccionada y asigna un cluster a cada observación.</div>


In [ ]:
km_E2 = KMeans(
    n_clusters=K_OPTIMO_E2,
    random_state=RANDOM_STATE_F2,
    n_init=20,
    max_iter=500,
    algorithm='lloyd'
)

print(f'Entrenando K-Means k={K_OPTIMO_E2} — Experimento 2 — '
      f'{len(X2_scaled_E2):,} registros...')
cluster_labels_E2 = km_E2.fit_predict(X2_scaled_E2)
df2_feat_E2 = df2_feat_E2.reset_index(drop=True)
df2_feat_E2['cluster'] = cluster_labels_E2

dist_E2 = pd.Series(cluster_labels_E2).value_counts().sort_index()
print('\nDistribución de clusters:')
for k, n in dist_E2.items():
    print(f'  Cluster {k}: {n:>7,}  ({n/len(cluster_labels_E2):.1%})')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula las métricas finales de cohesión y separación sobre una muestra reproducible del modelo ajustado.</div>


In [ ]:
labels_sample_E2 = cluster_labels_E2[sample_idx_E2]
sil_E2_final = silhouette_score(X2_sample_E2, labels_sample_E2,
                                sample_size=min(5_000, len(X2_sample_E2)),
                                random_state=RANDOM_STATE_F2)
db_E2_final  = davies_bouldin_score(X2_sample_E2, labels_sample_E2)
ch_E2_final  = calinski_harabasz_score(X2_sample_E2, labels_sample_E2)

print(f'Métricas del modelo final — Experimento 2 (k={K_OPTIMO_E2}):')
print(f'  Silhouette:        {sil_E2_final:.4f}')
print(f'  Davies-Bouldin:    {db_E2_final:.4f}')
print(f'  Calinski-Harabasz: {ch_E2_final:.1f}')

### 2.8 Visualización — proyección PCA

#### 2.8.1 Dispersión bidimensional por cluster

Proyección en dos componentes principales para diagnóstico visual de la separabilidad y cohesión de los clusters del Experimento 2.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Proyecta el espacio de variables a dos componentes principales para facilitar la inspección visual de los clusters.</div>


In [ ]:
pca_E2 = PCA(n_components=2, random_state=RANDOM_STATE_F2)
X2_pca_E2 = pca_E2.fit_transform(X2_scaled_E2)
var_exp_E2 = pca_E2.explained_variance_ratio_
print(f'Varianza explicada — E2: PC1={var_exp_E2[0]:.1%}  PC2={var_exp_E2[1]:.1%}  '
      f'Total={var_exp_E2.sum():.1%}')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Dibuja la proyección PCA coloreada por cluster y marca los centroides como diagnóstico visual de separación.</div>


In [ ]:
colors_E2 = sns.color_palette('Set2', K_OPTIMO_E2)
N_PLOT_E2 = min(3_000, len(X2_scaled_E2))

fig, ax = plt.subplots(figsize=(12, 8))
for k in range(K_OPTIMO_E2):
    idx_k = np.where(cluster_labels_E2 == k)[0][:N_PLOT_E2]
    ax.scatter(X2_pca_E2[idx_k, 0], X2_pca_E2[idx_k, 1],
               c=[colors_E2[k]], label=f'Cluster {k}  (n={dist_E2[k]:,})',
               alpha=0.45, s=15, linewidths=0)

centroids_pca_E2 = pca_E2.transform(km_E2.cluster_centers_)
ax.scatter(centroids_pca_E2[:, 0], centroids_pca_E2[:, 1],
           c='black', marker='X', s=250, zorder=10, label='Centroides')
for k, (cx, cy) in enumerate(centroids_pca_E2):
    ax.annotate(str(k), (cx, cy), xytext=(8, 4),
                textcoords='offset points', fontsize=12, fontweight='bold')

ax.set_xlabel(f'PC1 ({var_exp_E2[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({var_exp_E2[1]:.1%})', fontsize=12)
ax.set_title(f'K-Means k={K_OPTIMO_E2} — Proyección PCA 2D — Experimento 2', fontsize=14)
ax.legend(markerscale=2, fontsize=10)
plt.tight_layout()
plt.show()

#### 2.8.2 Distribución de tamaños por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ejecuta el procesamiento correspondiente a «2.8.2 Distribución de tamaños por cluster» y produce los insumos o resultados utilizados por el siguiente paso.</div>


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(dist_E2.index, dist_E2.values, color=colors_E2,
              edgecolor='white', linewidth=1.5)
for bar, (k, n) in zip(bars, dist_E2.items()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + len(cluster_labels_E2) * 0.003,
            f'{n:,}\n({n/len(cluster_labels_E2):.1%})',
            ha='center', va='bottom', fontsize=10)
ax.set(xlabel='Cluster', ylabel='N leads',
       title='Distribución de tamaños por cluster — Experimento 2')
ax.set_xticks(range(K_OPTIMO_E2))
plt.tight_layout()
plt.show()

### 2.9 Perfilamiento de clusters

El perfilamiento describe las características centrales de cada cluster mediante variables numéricas, categóricas y tasa de conversión (`target_won`) como validador externo. Al haber excluido las variables de canal y campaña, la diferenciación entre grupos depende exclusivamente de las dimensiones demográfica, socioeconómica y de preferencia de producto.


#### 2.9.1 Variables numéricas — mapa de calor de medias por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume las variables originales por cluster para identificar las características que distinguen cada perfil.</div>


In [ ]:
profile_num_E2 = df2_feat_E2.groupby('cluster')[VARS_NUM_E2].mean()
if 'ingreso_numerico' in df2_enc_E2.columns:
    df2_feat_E2['ingreso_numerico'] = df2_enc_E2['ingreso_numerico'].values
    profile_num_E2['ingreso_numerico'] = (
        df2_feat_E2.groupby('cluster')['ingreso_numerico'].mean()
    )

print('Media por cluster — variables numéricas (Experimento 2):')
print(profile_num_E2.T.to_string())

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Estandariza los perfiles y genera un mapa de calor que permite comparar visualmente los clusters.</div>


In [ ]:
profile_norm_E2 = (profile_num_E2 - profile_num_E2.mean()) / profile_num_E2.std()

fig, ax = plt.subplots(
    figsize=(max(8, K_OPTIMO_E2 * 1.8), len(profile_num_E2.columns) * 0.7 + 2)
)
sns.heatmap(
    profile_norm_E2.T,
    annot=profile_num_E2.T.round(1), fmt='g',
    cmap='RdBu_r', center=0, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Z-score respecto a media global'}
)
ax.set_title('Perfil de clusters — variables numéricas — Experimento 2', fontsize=13)
ax.set_xlabel('Cluster')
plt.tight_layout()
plt.show()

#### 2.9.2 Variables categóricas — distribución por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica las categorías más frecuentes para conocer la composición de las variables cualitativas.</div>


In [ ]:
VARS_CAT_PROF_E2 = [c for c in VARS_CAT_E2
                    if c in df2_feat_E2.columns and c != 'ingreso_familiar']

for col in VARS_CAT_PROF_E2:
    top_cats = df2_feat_E2[col].value_counts().head(8).index
    ct = (
        df2_feat_E2.groupby('cluster')[col]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex(columns=top_cats, fill_value=0) * 100
    )
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ct.T.plot(kind='bar', ax=ax, color=colors_E2, edgecolor='white', width=0.8)
    ax.set_title(f'{col} por cluster (%) — Experimento 2', fontsize=12)
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Cluster', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

#### 2.9.3 Tasa de conversión por cluster

La variable `target_won` no participó en el clustering. Su distribución por cluster sirve como validador externo de la coherencia de la segmentación. En el Experimento 2, diferencias en la tasa de conversión entre grupos son atribuibles exclusivamente al perfil del cliente —no al canal de captación—, lo que refuerza la validez del modelo para objetivos de segmentación comercial.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Mide la conversión por cluster como validación externa; esta variable no interviene en la formación de los segmentos.</div>


In [ ]:
conv_E2 = df2_feat_E2.groupby('cluster')['target_won'].agg(['mean', 'sum', 'count'])
conv_E2.columns = ['tasa_conversion', 'convertidos', 'total']
conv_E2['tasa_%'] = (conv_E2['tasa_conversion'] * 100).round(2)
print(conv_E2[['total', 'convertidos', 'tasa_%']].to_string())

global_rate_E2 = df2_feat_E2['target_won'].mean() * 100
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(conv_E2.index, conv_E2['tasa_%'], color=colors_E2,
              edgecolor='white', linewidth=1.5)
ax.axhline(global_rate_E2, color='red', linestyle='--', lw=1.2,
           label=f'Tasa global: {global_rate_E2:.1f}%')
for bar, rate in zip(bars, conv_E2['tasa_%']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.2, f'{rate:.1f}%',
            ha='center', fontsize=10)
ax.set(xlabel='Cluster', ylabel='Tasa de conversión (%)',
       title='Tasa de conversión por cluster — Experimento 2')
ax.legend()
plt.tight_layout()
plt.show()

### 2.10 Interpretación de resultados y hallazgos

#### Hallazgos principales

- Al excluir adquisición y campaña, el modelo k=7 alcanza Silhouette 0.1714, Davies–Bouldin 1.4651 y Calinski–Harabasz 1 845.5. Mejora al Experimento 1, aunque la cohesión continúa siendo moderada-baja.
- El cluster 1 concentra el producto de mayor valor (≈Q199 654; 47.7 m²) y el ingreso medio más alto (≈Q36 549), pero convierte 1.87 %.
- El cluster 4 obtiene la conversión más alta, 4.10 %, frente a 1.29 % del cluster 6.
- El cluster 5 contiene solo 38 registros; su tasa de 2.63 % equivale a una conversión y no es estable para decisiones.

#### Implicaciones para la estrategia comercial

- La exclusión de variables de adquisición mejora la geometría y permite leer perfiles más cercanos al cliente, pero no elimina la fragmentación.
- El cluster 4 amerita validación adicional como segmento de alto potencial. El cluster 1 no debe priorizarse únicamente por precio o ingreso.
- Conviene fusionar o descartar segmentos diminutos antes de operacionalizar el modelo.

#### Limitaciones

- Las métricas de selección discrepan: Silhouette favorece k=4, Davies–Bouldin k=6 y Calinski–Harabasz k=3; k=7 no es el óptimo de ninguna.
- La imputación residual y el tamaño mínimo de algunos clusters reducen la estabilidad de la interpretación.


### 2.11 Exportación de resultados


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Crea la carpeta de salida y exporta etiquetas, perfiles y métricas para reutilizarlos fuera del notebook.</div>


In [ ]:
df2_out_E2 = df2_raw.copy()
df2_out_E2['cluster_E2'] = np.nan
df2_out_E2.loc[df2_cluster_base.index, 'cluster_E2'] = cluster_labels_E2
out_path_E2 = os.path.join(OUT_DIR, 'leads_calificados_clustered_E2.csv')
df2_out_E2.to_csv(out_path_E2, index=False, encoding='utf-8-sig')
print(f'Dataset exportado: {out_path_E2}')
print(f'  Registros con cluster asignado : {len(df2_cluster_base):,}')
print(f'  Leads calificados totales      : {len(df2_out_E2):,}')

profile_num_E2.to_csv(
    os.path.join(OUT_DIR, 'perfil_numerico_clusters_E2.csv'), encoding='utf-8-sig'
)
print('Perfil numérico por cluster — E2 exportado.')

conv_E2.to_csv(
    os.path.join(OUT_DIR, 'conversion_por_cluster_E2.csv'), encoding='utf-8-sig'
)
print('Tasa de conversión por cluster — E2 exportada.')
print('\nExportación Experimento 2 completada.')

---

## 3. Experimento 3 — Segmentación de perfiles de cliente (sin núcleo familiar, participantes en crédito ni m² mínimo)

Se replica exactamente el flujo del Experimento 2, conservando sus componentes, código, visualizaciones y paleta de colores. La única modificación conceptual es la exclusión de `nucleo_familiar`, `participantes_en_credito` y `m2_minimo`.

Para garantizar que estas variables no influyan ni siquiera indirectamente en el análisis, también se excluyen del criterio de completitud usado para construir el universo analítico específico del Experimento 3 (`df2_cluster_base_E3`).

**Variables incluidas (7):** `edad`, `sexo`, `estado_civil`, `ingreso_familiar`, `tipo_unidad`, `precio_promedio`, `razon_de_compra`.

**Variables excluidas respecto al Experimento 2:** `nucleo_familiar`, `participantes_en_credito`, `m2_minimo`.

### 3.1 Configuración del experimento


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Aplica el criterio de completitud y construye el universo analítico que se utilizará en las etapas de clustering.</div>


## 3. Experimento 3 — Segmentación con universo reducido de variables

Se repite el protocolo de los experimentos anteriores reduciendo el universo a siete variables, excluyendo `nucleo_familiar`, `participantes_en_credito` y `m2_minimo`, ninguna de las cuales participa en el universo analítico, el preprocesamiento, el modelo ni el perfilamiento.

### 3.1 Configuración del experimento

In [ ]:
K_OPTIMO_E3 = 7  # ajustar según los resultados de la sección 8 de este experimento

# Universo propio: las variables excluidas no intervienen en el filtro de completitud.
EXCLUDED_VARS_E3 = ['nucleo_familiar', 'participantes_en_credito', 'm2_minimo']
KEY_VARS_COMPLETENESS_E3 = [
    c for c in KEY_VARS_COMPLETENESS
    if c not in EXCLUDED_VARS_E3 and c in df2.columns
]

pct_completitud_E3 = (
    df2[KEY_VARS_COMPLETENESS_E3].notna().sum(axis=1)
    / len(KEY_VARS_COMPLETENESS_E3)
)
mask_completo_E3 = pct_completitud_E3 >= COMPLETENESS_THRESHOLD
df2_cluster_base_E3 = df2.loc[mask_completo_E3].copy()

print(f'Variables de completitud E3 ({len(KEY_VARS_COMPLETENESS_E3)}): '
      f'{KEY_VARS_COMPLETENESS_E3}')
print(f'Variables excluidas por completo: {EXCLUDED_VARS_E3}')
print(f'Universo E3 (≥ {COMPLETENESS_THRESHOLD:.0%} completitud): '
      f'{len(df2_cluster_base_E3):,} de {len(df2):,} leads '
      f'({len(df2_cluster_base_E3) / len(df2):.1%})')


Se utilizan siete variables distribuidas en cuatro dimensiones de análisis. Se excluyen deliberadamente `nucleo_familiar`, `participantes_en_credito` y `m2_minimo`; ninguna participa en el universo analítico, el preprocesamiento, el modelo ni el perfilamiento.

| Grupo | Variables |
|-------|-----------|
| Demográfico | `edad`, `sexo`, `estado_civil` |
| Socioeconómico | `ingreso_familiar` |
| Preferencia de producto | `tipo_unidad`, `precio_promedio` |
| Comportamental | `razon_de_compra` |


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
VARS_NUM_E3 = [
    'edad',
    'precio_promedio',
]

VARS_CAT_E3 = [
    'sexo',
    'estado_civil',
    'ingreso_familiar',
    'tipo_unidad',
    'razon_de_compra',
]

VARS_NUM_E3 = [c for c in VARS_NUM_E3 if c in df2_cluster_base_E3.columns]
VARS_CAT_E3 = [c for c in VARS_CAT_E3 if c in df2_cluster_base_E3.columns]
ALL_VARS_E3 = VARS_NUM_E3 + VARS_CAT_E3

df2_feat_E3 = df2_cluster_base_E3[ALL_VARS_E3 + ['target_won']].copy()

print(f'Numéricas   ({len(VARS_NUM_E3)}): {VARS_NUM_E3}')
print(f'Categóricas ({len(VARS_CAT_E3)}): {VARS_CAT_E3}')
print(f'Shape: {df2_feat_E3.shape}')
df2_feat_E3.head()


### 3.2 Selección de variables

Misma estrategia que en el Experimento 1: verificación del tipo de almacenamiento real para variables declaradas como numéricas y conversión cuando el 90 % o más de los valores no nulos son parseables a número.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
_num_e3 = list(VARS_NUM_E3)
_cat_e3 = list(VARS_CAT_E3)
rows_e3 = []

for col in list(ALL_VARS_E3):
    conceptual  = 'numérica'  if col in _num_e3 else 'categórica'
    orig_dtype  = str(df2_feat_E3[col].dtype)
    final_dtype = orig_dtype
    action      = 'mantenida'

    if col in _num_e3:
        if pd.api.types.is_numeric_dtype(df2_feat_E3[col]):
            pass
        else:
            non_null = df2_feat_E3[col].notna()
            conv_nn  = pd.to_numeric(df2_feat_E3[col][non_null], errors='coerce')
            p_rate   = conv_nn.notna().mean()
            if p_rate >= PARSE_RATE_NON_NULL:
                df2_feat_E3[col] = pd.to_numeric(df2_feat_E3[col], errors='coerce')
                action      = f'convertida (parse {p_rate:.0%} s/no nulos)'
                final_dtype = str(df2_feat_E3[col].dtype)
            else:
                _num_e3.remove(col)
                _cat_e3.append(col)
                action = f'→ categórica (parse {p_rate:.0%} s/no nulos)'
    else:
        if pd.api.types.is_numeric_dtype(df2_feat_E3[col]):
            df2_feat_E3[col] = df2_feat_E3[col].astype(str).replace('nan', np.nan)
            action      = 'convertida a string'
            final_dtype = str(df2_feat_E3[col].dtype)

    rows_e3.append({'variable': col, 'tipo_conceptual': conceptual,
                    'dtype_original': orig_dtype, 'dtype_final': final_dtype,
                    'acción': action})

VARS_NUM_E3 = _num_e3
VARS_CAT_E3 = _cat_e3
ALL_VARS_E3 = VARS_NUM_E3 + VARS_CAT_E3

display(pd.DataFrame(rows_e3))
print(f'\nNuméricas   ({len(VARS_NUM_E3)}): {VARS_NUM_E3}')
print(f'Categóricas ({len(VARS_CAT_E3)}): {VARS_CAT_E3}')

### 3.3 Tratamiento de valores faltantes

Se aplica la misma estrategia diferenciada que en el Experimento 2. Como el universo analítico E3 se recalcula sin las tres variables excluidas, el missingness residual se vuelve a medir e imputar únicamente sobre las siete variables retenidas.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
print('Missingness en variables seleccionadas — Experimento 3:')
m_e3 = df2_feat_E3[ALL_VARS_E3].isnull().sum()
print(pd.DataFrame({'n': m_e3, '%': (m_e3 / len(df2_feat_E3) * 100).round(1)})
      .loc[m_e3 > 0].sort_values('%', ascending=False).to_string())

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula y visualiza el porcentaje de valores faltantes para identificar variables con cobertura insuficiente.</div>


In [ ]:
for col in VARS_NUM_E3:
    n = df2_feat_E3[col].isnull().sum()
    if n > 0:
        med = df2_feat_E3[col].median()
        df2_feat_E3[col] = df2_feat_E3[col].fillna(med)
        print(f'  [num] {col}: {n:,} → mediana={med:.2f}')

for col in VARS_CAT_E3:
    n = df2_feat_E3[col].isnull().sum()
    if n == 0:
        continue
    pct = n / len(df2_feat_E3)
    if pct >= 0.50:
        df2_feat_E3[col] = df2_feat_E3[col].fillna('Desconocido')
        print(f'  [cat] {col}: {pct:.1%} faltante → "Desconocido"')
    else:
        mode_val = df2_feat_E3[col].mode()[0]
        df2_feat_E3[col] = df2_feat_E3[col].fillna(mode_val)
        print(f'  [cat] {col}: {n:,} → moda="{mode_val}"')

print(f'\nMissing restante: {df2_feat_E3[ALL_VARS_E3].isnull().sum().sum()}')

### 3.4 Tratamiento de valores atípicos

Se aplica winsorización con factor IQR × 3 sobre las variables numéricas, siguiendo el mismo criterio conservador del Experimento 1.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define o aplica el tratamiento de valores atípicos para reducir su influencia sobre las distancias y los centroides.</div>


In [ ]:
print('Límites de winsorización (IQR × 3) — Experimento 3:')
for col in VARS_NUM_E3:
    df2_feat_E3[col], lo, hi = winsorize_iqr_f2(df2_feat_E3[col])
    print(f'  {col:30s}: [{lo:.2f}, {hi:.2f}]')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define o aplica el tratamiento de valores atípicos para reducir su influencia sobre las distancias y los centroides.</div>


In [ ]:
n_num_e3 = len(VARS_NUM_E3)
fig, axes = plt.subplots(1, n_num_e3, figsize=(3.5 * n_num_e3, 4))
if n_num_e3 == 1:
    axes = [axes]
for ax, col in zip(axes, VARS_NUM_E3):
    ax.boxplot(df2_feat_E3[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#5B8DB8', alpha=0.7))
    ax.set_title(col, fontsize=10)
    ax.set_xticks([])
plt.suptitle('Boxplots post-winsorización — Experimento 3', fontsize=12)
plt.tight_layout()
plt.show()

### 3.5 Codificación y escalamiento

Mismo proceso que en el Experimento 1: conversión del ingreso familiar a valor numérico continuo mediante el punto medio del rango declarado, codificación OHE del resto de variables categóricas y estandarización mediante `StandardScaler`.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define las variables numéricas y categóricas del experimento y valida que todas estén disponibles en el universo analítico.</div>


In [ ]:
df2_enc_E3 = df2_feat_E3[ALL_VARS_E3].copy()

if 'ingreso_familiar' in VARS_CAT_E3:
    df2_enc_E3['ingreso_numerico'] = df2_enc_E3['ingreso_familiar'].apply(parse_income_f2)
    df2_enc_E3['ingreso_numerico'].fillna(df2_enc_E3['ingreso_numerico'].median(), inplace=True)
    VARS_NUM_E3_FINAL = VARS_NUM_E3 + ['ingreso_numerico']
    VARS_CAT_E3_OHE   = [c for c in VARS_CAT_E3 if c != 'ingreso_familiar']
else:
    VARS_NUM_E3_FINAL = VARS_NUM_E3
    VARS_CAT_E3_OHE   = VARS_CAT_E3

df2_ohe_E3 = pd.get_dummies(df2_enc_E3[VARS_CAT_E3_OHE], drop_first=True, dtype=float)

X2_pre_E3 = pd.concat(
    [df2_enc_E3[VARS_NUM_E3_FINAL].reset_index(drop=True),
     df2_ohe_E3.reset_index(drop=True)],
    axis=1
)

feature_names_E3 = X2_pre_E3.columns.tolist()
print(f'Features antes de escalar — E3: {X2_pre_E3.shape[1]}')
print(feature_names_E3)

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Escala las variables para que sus rangos sean comparables y ninguna domine el cálculo de distancias por su unidad de medida.</div>


In [ ]:
scaler_E3    = StandardScaler()
X2_scaled_E3 = scaler_E3.fit_transform(X2_pre_E3)
print(f'Dimensiones post-escalamiento — E3: {X2_scaled_E3.shape}')

### 3.6 Selección del número óptimo de clusters

Se evalúan las cuatro métricas complementarias sobre una muestra de `SAMPLE_K_SEL_F2` registros para el rango de k ∈ [2, 10]. La eliminación de núcleo familiar, participantes en crédito y m² mínimo modifica el espacio de features y puede producir un k óptimo diferente al del Experimento 2.

| Métrica | Criterio | Interpretación |
|---------|----------|----------------|
| Inercia (codo) | Punto de inflexión | Reducción marginal de la compactness |
| Silhouette | ↑ mejor (máx. 1) | Cohesión intra-cluster y separación inter-cluster |
| Davies-Bouldin | ↓ mejor (mín. 0) | Dispersión interna relativa a distancia entre clusters |
| Calinski-Harabasz | ↑ mejor | Razón de varianza inter-cluster a intra-cluster |


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Evalúa distintos valores de k mediante Silhouette, Davies–Bouldin, Calinski–Harabasz e inercia.</div>


In [ ]:
np.random.seed(RANDOM_STATE_F2)
sample_idx_E3 = np.random.choice(
    len(X2_scaled_E3), size=min(SAMPLE_K_SEL_F2, len(X2_scaled_E3)), replace=False
)
X2_sample_E3 = X2_scaled_E3[sample_idx_E3]

inertias_E3, silhs_E3, db_E3, ch_E3 = [], [], [], []

print(f'Evaluando k ∈ {list(K_RANGE_F2)} sobre {len(X2_sample_E3):,} filas...')
for k in K_RANGE_F2:
    km_tmp = KMeans(n_clusters=k, random_state=RANDOM_STATE_F2, n_init=10)
    lbls   = km_tmp.fit_predict(X2_sample_E3)
    inertias_E3.append(km_tmp.inertia_)
    silhs_E3.append(silhouette_score(X2_sample_E3, lbls,
                    sample_size=min(5_000, len(X2_sample_E3)), random_state=RANDOM_STATE_F2))
    db_E3.append(davies_bouldin_score(X2_sample_E3, lbls))
    ch_E3.append(calinski_harabasz_score(X2_sample_E3, lbls))
    print(f'  k={k:2d} | Silhouette={silhs_E3[-1]:.3f} | '
          f'DB={db_E3[-1]:.3f} | CH={ch_E3[-1]:.0f}')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Evalúa distintos valores de k mediante Silhouette, Davies–Bouldin, Calinski–Harabasz e inercia.</div>


In [ ]:
k_vals_E3 = list(K_RANGE_F2)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(k_vals_E3, inertias_E3, 'o-', color='#5B8DB8', lw=2, ms=7)
axes[0,0].set(xlabel='k', ylabel='Inercia', title='Método del Codo — E3')
axes[0,0].grid(alpha=0.4)

axes[0,1].plot(k_vals_E3, silhs_E3, 's-', color='#E07B54', lw=2, ms=7)
axes[0,1].set(xlabel='k', ylabel='Silhouette', title='Silhouette (↑ mejor) — E3')
axes[0,1].grid(alpha=0.4)

axes[1,0].plot(k_vals_E3, db_E3, '^-', color='#6BAF72', lw=2, ms=7)
axes[1,0].set(xlabel='k', ylabel='Davies-Bouldin', title='Davies-Bouldin (↓ mejor) — E3')
axes[1,0].grid(alpha=0.4)

axes[1,1].plot(k_vals_E3, ch_E3, 'D-', color='#9B59B6', lw=2, ms=7)
axes[1,1].set(xlabel='k', ylabel='Calinski-Harabasz', title='Calinski-Harabasz (↑ mejor) — E3')
axes[1,1].grid(alpha=0.4)

plt.suptitle('Métricas de selección de k — Experimento 3', fontsize=14)
plt.tight_layout()
plt.show()

print(f'k con mejor Silhouette : {k_vals_E3[int(np.argmax(silhs_E3))]}')
print(f'k con mejor DB         : {k_vals_E3[int(np.argmin(db_E3))]}')
print(f'k con mejor CH         : {k_vals_E3[int(np.argmax(ch_E3))]}')
print(f'K_OPTIMO_E3 actual     : {K_OPTIMO_E3}  (ajustar en celda de configuración)')

### 3.7 Entrenamiento del modelo K-Means final

Se entrena K-Means con `n_init=20` inicializaciones sobre la totalidad del universo de clustering. El valor `K_OPTIMO_E3` puede diferir del Experimento 2 si las métricas indican un número distinto de clusters óptimo al excluir núcleo familiar, participantes en crédito y m² mínimo.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Entrena el modelo K-Means con la configuración seleccionada y asigna un cluster a cada observación.</div>


In [ ]:
km_E3 = KMeans(
    n_clusters=K_OPTIMO_E3,
    random_state=RANDOM_STATE_F2,
    n_init=20,
    max_iter=500,
    algorithm='lloyd'
)

print(f'Entrenando K-Means k={K_OPTIMO_E3} — Experimento 3 — '
      f'{len(X2_scaled_E3):,} registros...')
cluster_labels_E3 = km_E3.fit_predict(X2_scaled_E3)
df2_feat_E3 = df2_feat_E3.reset_index(drop=True)
df2_feat_E3['cluster'] = cluster_labels_E3

dist_E3 = pd.Series(cluster_labels_E3).value_counts().sort_index()
print('\nDistribución de clusters:')
for k, n in dist_E3.items():
    print(f'  Cluster {k}: {n:>7,}  ({n/len(cluster_labels_E3):.1%})')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula las métricas finales de cohesión y separación sobre una muestra reproducible del modelo ajustado.</div>


In [ ]:
labels_sample_E3 = cluster_labels_E3[sample_idx_E3]
sil_E3_final = silhouette_score(X2_sample_E3, labels_sample_E3,
                                sample_size=min(5_000, len(X2_sample_E3)),
                                random_state=RANDOM_STATE_F2)
db_E3_final  = davies_bouldin_score(X2_sample_E3, labels_sample_E3)
ch_E3_final  = calinski_harabasz_score(X2_sample_E3, labels_sample_E3)

print(f'Métricas del modelo final — Experimento 3 (k={K_OPTIMO_E3}):')
print(f'  Silhouette:        {sil_E3_final:.4f}')
print(f'  Davies-Bouldin:    {db_E3_final:.4f}')
print(f'  Calinski-Harabasz: {ch_E3_final:.1f}')

### 3.8 Visualización — proyección PCA

#### 3.8.1 Dispersión bidimensional por cluster

Proyección en dos componentes principales para diagnóstico visual de la separabilidad y cohesión de los clusters del Experimento 3.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Proyecta el espacio de variables a dos componentes principales para facilitar la inspección visual de los clusters.</div>


In [ ]:
pca_E3 = PCA(n_components=2, random_state=RANDOM_STATE_F2)
X2_pca_E3 = pca_E3.fit_transform(X2_scaled_E3)
var_exp_E3 = pca_E3.explained_variance_ratio_
print(f'Varianza explicada — E3: PC1={var_exp_E3[0]:.1%}  PC2={var_exp_E3[1]:.1%}  '
      f'Total={var_exp_E3.sum():.1%}')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Dibuja la proyección PCA coloreada por cluster y marca los centroides como diagnóstico visual de separación.</div>


In [ ]:
colors_E3 = sns.color_palette('Set2', K_OPTIMO_E3)
N_PLOT_E3 = min(3_000, len(X2_scaled_E3))

fig, ax = plt.subplots(figsize=(12, 8))
for k in range(K_OPTIMO_E3):
    idx_k = np.where(cluster_labels_E3 == k)[0][:N_PLOT_E3]
    ax.scatter(X2_pca_E3[idx_k, 0], X2_pca_E3[idx_k, 1],
               c=[colors_E3[k]], label=f'Cluster {k}  (n={dist_E3[k]:,})',
               alpha=0.45, s=15, linewidths=0)

centroids_pca_E3 = pca_E3.transform(km_E3.cluster_centers_)
ax.scatter(centroids_pca_E3[:, 0], centroids_pca_E3[:, 1],
           c='black', marker='X', s=250, zorder=10, label='Centroides')
for k, (cx, cy) in enumerate(centroids_pca_E3):
    ax.annotate(str(k), (cx, cy), xytext=(8, 4),
                textcoords='offset points', fontsize=12, fontweight='bold')

ax.set_xlabel(f'PC1 ({var_exp_E3[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({var_exp_E3[1]:.1%})', fontsize=12)
ax.set_title(f'K-Means k={K_OPTIMO_E3} — Proyección PCA 2D — Experimento 3', fontsize=14)
ax.legend(markerscale=2, fontsize=10)
plt.tight_layout()
plt.show()

#### 3.8.2 Distribución de tamaños por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ejecuta el procesamiento correspondiente a «3.8.2 Distribución de tamaños por cluster» y produce los insumos o resultados utilizados por el siguiente paso.</div>


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(dist_E3.index, dist_E3.values, color=colors_E3,
              edgecolor='white', linewidth=1.5)
for bar, (k, n) in zip(bars, dist_E3.items()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + len(cluster_labels_E3) * 0.003,
            f'{n:,}\n({n/len(cluster_labels_E3):.1%})',
            ha='center', va='bottom', fontsize=10)
ax.set(xlabel='Cluster', ylabel='N leads',
       title='Distribución de tamaños por cluster — Experimento 3')
ax.set_xticks(range(K_OPTIMO_E3))
plt.tight_layout()
plt.show()

### 3.9 Perfilamiento de clusters

El perfilamiento describe las características centrales de cada cluster mediante variables numéricas, categóricas y tasa de conversión (`target_won`) como validador externo. Al haber excluido núcleo familiar, participantes en crédito y m² mínimo, la diferenciación entre grupos depende de las siete variables restantes del perfil del cliente y su preferencia de producto.


#### 3.9.1 Variables numéricas — mapa de calor de medias por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume las variables originales por cluster para identificar las características que distinguen cada perfil.</div>


In [ ]:
profile_num_E3 = df2_feat_E3.groupby('cluster')[VARS_NUM_E3].mean()
if 'ingreso_numerico' in df2_enc_E3.columns:
    df2_feat_E3['ingreso_numerico'] = df2_enc_E3['ingreso_numerico'].values
    profile_num_E3['ingreso_numerico'] = (
        df2_feat_E3.groupby('cluster')['ingreso_numerico'].mean()
    )

print('Media por cluster — variables numéricas (Experimento 3):')
print(profile_num_E3.T.to_string())

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Estandariza los perfiles y genera un mapa de calor que permite comparar visualmente los clusters.</div>


In [ ]:
profile_norm_E3 = (profile_num_E3 - profile_num_E3.mean()) / profile_num_E3.std()

fig, ax = plt.subplots(
    figsize=(max(8, K_OPTIMO_E3 * 1.8), len(profile_num_E3.columns) * 0.7 + 2)
)
sns.heatmap(
    profile_norm_E3.T,
    annot=profile_num_E3.T.round(1), fmt='g',
    cmap='RdBu_r', center=0, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Z-score respecto a media global'}
)
ax.set_title('Perfil de clusters — variables numéricas — Experimento 3', fontsize=13)
ax.set_xlabel('Cluster')
plt.tight_layout()
plt.show()

#### 3.9.2 Variables categóricas — distribución por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica las categorías más frecuentes para conocer la composición de las variables cualitativas.</div>


In [ ]:
VARS_CAT_PROF_E3 = [c for c in VARS_CAT_E3
                    if c in df2_feat_E3.columns and c != 'ingreso_familiar']

for col in VARS_CAT_PROF_E3:
    top_cats = df2_feat_E3[col].value_counts().head(8).index
    ct = (
        df2_feat_E3.groupby('cluster')[col]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex(columns=top_cats, fill_value=0) * 100
    )
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ct.T.plot(kind='bar', ax=ax, color=colors_E3, edgecolor='white', width=0.8)
    ax.set_title(f'{col} por cluster (%) — Experimento 3', fontsize=12)
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Cluster', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

#### 3.9.3 Tasa de conversión por cluster

La variable `target_won` no participó en el clustering. Su distribución por cluster sirve como validador externo de la coherencia de la segmentación. En el Experimento 3, las diferencias en la tasa de conversión entre grupos permiten validar externamente la segmentación construida sin núcleo familiar ni participantes en crédito.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Mide la conversión por cluster como validación externa; esta variable no interviene en la formación de los segmentos.</div>


In [ ]:
conv_E3 = df2_feat_E3.groupby('cluster')['target_won'].agg(['mean', 'sum', 'count'])
conv_E3.columns = ['tasa_conversion', 'convertidos', 'total']
conv_E3['tasa_%'] = (conv_E3['tasa_conversion'] * 100).round(2)
print(conv_E3[['total', 'convertidos', 'tasa_%']].to_string())

global_rate_E3 = df2_feat_E3['target_won'].mean() * 100
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(conv_E3.index, conv_E3['tasa_%'], color=colors_E3,
              edgecolor='white', linewidth=1.5)
ax.axhline(global_rate_E3, color='red', linestyle='--', lw=1.2,
           label=f'Tasa global: {global_rate_E3:.1f}%')
for bar, rate in zip(bars, conv_E3['tasa_%']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.2, f'{rate:.1f}%',
            ha='center', fontsize=10)
ax.set(xlabel='Cluster', ylabel='Tasa de conversión (%)',
       title='Tasa de conversión por cluster — Experimento 3')
ax.legend()
plt.tight_layout()
plt.show()

### 3.10 Interpretación de resultados y hallazgos

#### Hallazgos principales

- El modelo k=7 obtiene Silhouette 0.2153, Davies–Bouldin 1.4518 y Calinski–Harabasz 2 536.0. Es el mejor de los tres experimentos base en las métricas reportadas.
- Silhouette, Davies–Bouldin y Calinski–Harabasz coinciden en que k=5 es preferible, por lo que el ajuste final con k=7 debe considerarse una especificación exploratoria.
- El cluster 0 presenta el precio medio más alto (≈Q156 270) y la conversión mayor (3.26 %). El cluster 3 registra la menor conversión (1.00 %).
- El cluster 6 reúne solo 41 registros, de modo que su tasa de 2.44 % corresponde a una única conversión y no es concluyente.

#### Implicaciones para la estrategia comercial

- Retirar núcleo familiar, participantes en crédito y m² mejora la separación respecto de los Experimentos 1 y 2, lo que sugiere que esas variables añadían solapamiento.
- El cluster 0 es el candidato más claro para validación comercial posterior por combinar mayor valor de producto y conversión.
- Antes de desplegar el modelo conviene reentrenar con k=5 y comparar estabilidad, tamaños y perfiles.

#### Limitaciones

- La simplificación mejora las métricas, pero reduce la riqueza familiar y de producto de los perfiles.
- Los clusters diminutos y la validación en una sola ejecución impiden asumir estabilidad temporal.


### 3.11 Exportación de resultados


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Crea la carpeta de salida y exporta etiquetas, perfiles y métricas para reutilizarlos fuera del notebook.</div>


In [ ]:
df2_out_E3 = df2_raw.copy()
df2_out_E3['cluster_E3'] = np.nan
df2_out_E3.loc[df2_cluster_base_E3.index, 'cluster_E3'] = cluster_labels_E3
out_path_E3 = os.path.join(OUT_DIR, 'leads_calificados_clustered_E3.csv')
df2_out_E3.to_csv(out_path_E3, index=False, encoding='utf-8-sig')
print(f'Dataset exportado: {out_path_E3}')
print(f'  Registros con cluster asignado : {len(df2_cluster_base_E3):,}')
print(f'  Leads calificados totales      : {len(df2_out_E3):,}')

profile_num_E3.to_csv(
    os.path.join(OUT_DIR, 'perfil_numerico_clusters_E3.csv'), encoding='utf-8-sig'
)
print('Perfil numérico por cluster — E3 exportado.')

conv_E3.to_csv(
    os.path.join(OUT_DIR, 'conversion_por_cluster_E3.csv'), encoding='utf-8-sig'
)
print('Tasa de conversión por cluster — E3 exportada.')
print('\nExportación Experimento 3 completada.')

## 4. Múltiples experimentos adicionales

---

**Comparación entre experimentos**

La siguiente tabla resume las diferencias estructurales entre los tres experimentos:

| Dimensión | Experimento 1 | Experimento 2 | Experimento 3 |
|-----------|---------------|---------------|---------------|
| Variables de entrada | 13 | 10 | 7 |
| Variables de adquisición incluidas | `canal_de_contacto`, `campania_tipo`, `campania_plataforma` | — (excluidas) | — (excluidas) |
| Variables adicionales excluidas | — | — | `nucleo_familiar`, `participantes_en_credito`, `m2_minimo` |
| Dimensiones cubiertas | Demográfico, socioeconómico, producto, comportamental, marketing/adquisición | Demográfico, socioeconómico, producto, comportamental | Demográfico, socioeconómico, producto, comportamental |
| Universo de clustering | `df2_cluster_base` | `df2_cluster_base` | `df2_cluster_base_E3` (sin influencia de las tres variables excluidas) |
| Objetivo de segmentación | Perfil integral lead–adquisición | Perfil del cliente potencial | Perfil del cliente potencial sin composición familiar, estructura del crédito ni m² mínimo |


### 4.1 Protocolo experimental común

Con el fin de evaluar la robustez de la segmentación, se comparan de forma controlada configuraciones alternativas de variables, preprocesamiento y selección de $k$. Este bloque permite identificar si los hallazgos son consistentes más allá de una sola especificación metodológica.


### 4.2 Configuraciones evaluadas

---

**Experimentos adicionales para mejora de calidad de clustering**

**Objetivo y criterio de decisión**

Esta sección amplía los Experimentos 1–3 sin sustituirlos: aquellos se conservan como *baseline*. El objetivo es elevar el coeficiente Silhouette —idealmente a 0.50 o más— mediante cambios controlados en variables, ponderación, transformación, outliers, *encoding* y reducción dimensional, sin sacrificar la interpretación de negocio.

La selección no se basa únicamente en Silhouette. También se consideran Davies–Bouldin, Calinski–Harabasz, balance de tamaños, diversidad real de los perfiles, estabilidad conceptual y utilidad comercial. `target_won` permanece completamente aislada del ajuste y se usa solo después del clustering como validación externa.



<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Prepara el entorno: importa las librerías, fija parámetros reproducibles y define las rutas utilizadas por el análisis.</div>


In [ ]:
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler

RANDOM_STATE_ADIC = 42
K_RANGE_ADIC = range(2, 11)
N_EVAL_ADIC = 6_000
N_SIL_ADIC = 5_000
COMPLETENESS_ADIC = 0.85

df_adic = df2.copy()
df_adic['ingreso_numerico'] = df_adic['ingreso_familiar'].apply(parse_income_f2)

VARS_UNION_ADIC = [
    'edad', 'ingreso_numerico', 'precio_promedio', 'm2_minimo',
    'razon_de_compra', 'tipo_unidad', 'sexo'
]
pct_completitud_adic = df_adic[VARS_UNION_ADIC].notna().mean(axis=1)
df_adic_base = df_adic.loc[
    pct_completitud_adic >= COMPLETENESS_ADIC,
    VARS_UNION_ADIC + ['target_won']
].copy()

NUM_UNION_ADIC = ['edad', 'ingreso_numerico', 'precio_promedio', 'm2_minimo']
CAT_UNION_ADIC = ['razon_de_compra', 'tipo_unidad', 'sexo']
for col in NUM_UNION_ADIC:
    df_adic_base[col] = pd.to_numeric(df_adic_base[col], errors='coerce')
    df_adic_base[col] = df_adic_base[col].fillna(df_adic_base[col].median())
for col in CAT_UNION_ADIC:
    df_adic_base[col] = df_adic_base[col].fillna('Desconocido').astype(str)

print(f'Universo experimental común: {len(df_adic_base):,} registros')
print(f'Completitud mínima: {COMPLETENESS_ADIC:.0%}')
print('target_won reservada exclusivamente para validación externa.')

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Declara las configuraciones que se compararán, especificando variables, transformaciones, ponderaciones y valor final de k.</div>


In [ ]:
EXPERIMENTOS_ADIC = [
    dict(id='A1', nombre='Core 7 — OHE balanceado', nums=NUM_UNION_ADIC, cats=CAT_UNION_ADIC,
         transform='log', outlier='iqr3', encoding='ohe_balanceado', pesos={}, pca=None, k=2,
         interpretacion='Dos niveles económico-producto; control metodológico mejorado.',
         observacion='Más interpretable que el baseline, pero aún con separación moderada.'),
    dict(id='A2', nombre='Core 7 — categóricas ponderadas', nums=NUM_UNION_ADIC, cats=CAT_UNION_ADIC,
         transform='log', outlier='iqr3', encoding='ohe_balanceado',
         pesos={'razon_de_compra': 1.5, 'tipo_unidad': 1.5, 'sexo': 0.6}, pca=None, k=2,
         interpretacion='Prioriza intención y producto.',
         observacion='La ponderación manual no mejoró la cohesión.'),
    dict(id='A3', nombre='Core 6 — sin sexo', nums=NUM_UNION_ADIC,
         cats=['razon_de_compra', 'tipo_unidad'], transform='log', outlier='iqr3',
         encoding='ohe_balanceado', pesos={}, pca=None, k=2,
         interpretacion='Perfil económico-producto sin una variable poco discriminante.',
         observacion='Mejora leve frente a A1; sexo aporta poca estructura.'),
    dict(id='A4', nombre='Core 6 — sin m² mínimo', nums=['edad','ingreso_numerico','precio_promedio'],
         cats=CAT_UNION_ADIC, transform='log', outlier='iqr3', encoding='ohe_balanceado',
         pesos={}, pca=None, k=4, interpretacion='Perfiles sin dimensión explícita de tamaño.',
         observacion='La exclusión de m² redujo claramente la separación.'),
    dict(id='A5', nombre='Producto e intención', nums=['ingreso_numerico','precio_promedio','m2_minimo'],
         cats=['razon_de_compra','tipo_unidad'], transform='log', outlier='iqr3',
         encoding='ohe_balanceado', pesos={'razon_de_compra':1.4,'tipo_unidad':1.4}, pca=None, k=2,
         interpretacion='Segmentos de capacidad, producto e intención de compra.',
         observacion='Cercano a 0.50 y comercialmente legible.'),
    dict(id='A6', nombre='Numéricas 4 — log robusto', nums=NUM_UNION_ADIC, cats=[],
         transform='log', outlier='iqr15', encoding='ninguno', pesos={}, pca=None, k=2,
         scaler='robust', interpretacion='Dos perfiles puramente cuantitativos.',
         observacion='Robusto a extremos, pero pierde motivación y producto categórico.'),
    dict(id='A7', nombre='Producto numérico — recomendado',
         nums=['ingreso_numerico','precio_promedio','m2_minimo'], cats=[], transform='log',
         outlier='iqr15', encoding='ninguno', pesos={}, pca=None, k=6,
         interpretacion='Seis estratos económico-producto con conversión diferenciada.',
         observacion='Mejor equilibrio entre calidad, tamaños y utilidad comercial.'),
    dict(id='A8', nombre='Edad–ingreso–precio por cuantiles',
         nums=['edad','ingreso_numerico','precio_promedio'], cats=[], transform='cuantiles',
         outlier='q0199', encoding='ninguno', pesos={}, pca=None, k=2,
         interpretacion='Dos niveles principalmente definidos por precio.',
         observacion='Score alto, pero segmentación poco rica.'),
    dict(id='A9', nombre='Razón–tipo–sexo', nums=[], cats=CAT_UNION_ADIC, transform='ninguna',
         outlier='ninguno', encoding='ohe_balanceado', pesos={}, pca=None, k=6,
         interpretacion='Combinaciones de intención, unidad y sexo.',
         observacion='Alta separación, parcialmente tautológica.'),
    dict(id='A10', nombre='Razón–tipo — prueba de estrés', nums=[],
         cats=['razon_de_compra','tipo_unidad'], transform='ninguna', outlier='ninguno',
         encoding='ohe_balanceado', pesos={}, pca=None, k=5,
         interpretacion='Reproduce combinaciones categóricas observadas.',
         observacion='Silhouette muy alto, rechazado como modelo principal por circularidad.'),
    dict(id='A11', nombre='Core 7 — frecuencia', nums=NUM_UNION_ADIC, cats=CAT_UNION_ADIC,
         transform='log', outlier='iqr3', encoding='frecuencia', pesos={}, pca=None, k=2,
         interpretacion='Categorías representadas por prevalencia.',
         observacion='Peor score y distancias categóricas menos defendibles.'),
    dict(id='A12', nombre='Core 7 — PCA 85 %', nums=NUM_UNION_ADIC, cats=CAT_UNION_ADIC,
         transform='log', outlier='iqr3', encoding='ohe_balanceado', pesos={}, pca=0.85, k=2,
         interpretacion='Tres componentes compactan la información conjunta.',
         observacion='Mejora A1, pero no alcanza 0.50.'),
    dict(id='A13', nombre='Core 6 sin sexo — PCA 85 %', nums=NUM_UNION_ADIC,
         cats=['razon_de_compra','tipo_unidad'], transform='log', outlier='iqr3',
         encoding='ohe_balanceado', pesos={}, pca=0.85, k=2,
         interpretacion='Reducción dimensional sin sexo.',
         observacion='Resultado casi idéntico a A12.'),
    dict(id='A14', nombre='Core 7 — cuantiles', nums=NUM_UNION_ADIC, cats=CAT_UNION_ADIC,
         transform='cuantiles', outlier='q0199', encoding='ohe_balanceado', pesos={}, pca=None, k=2,
         interpretacion='Dos niveles de mercado dominados por precio.',
         observacion='Mejor score no trivial, pero menor riqueza que A7.'),
]

EXPERIMENTOS_RELEVANTES = ['A1', 'A5', 'A7', 'A8', 'A12', 'A14']

### 4.3 Selección de k y evaluación controlada

- Universo común para garantizar comparabilidad: registros con al menos 85 % de completitud en la unión de las siete variables de interés.
- Variables: `edad`, `ingreso_numerico`, `precio_promedio`, `m2_minimo`, `razon_de_compra`, `tipo_unidad` y `sexo`.
- Selección de k: k ∈ [2, 10] sobre una muestra fija de 6 000 observaciones.
- K-Means: `random_state=42`, `n_init=20`, máximo de 500 iteraciones.
- Silhouette: muestra reproducible de hasta 5 000 registros; Davies–Bouldin y Calinski–Harabasz sobre la muestra fija completa.
- Modelo final: ajuste sobre todo el universo experimental; las métricas finales usan las etiquetas del modelo completo proyectadas sobre la muestra de evaluación.
- Categóricas faltantes: categoría explícita `Desconocido`; numéricas faltantes: mediana.

El balanceo OHE divide cada bloque categórico por √(*número de niveles*), evitando que una variable con más categorías gane peso solo por producir más columnas. La codificación por frecuencia se conserva únicamente como contraste, porque impone una distancia ordinal basada en prevalencia y reduce interpretabilidad.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define o aplica el tratamiento de valores atípicos para reducir su influencia sobre las distancias y los centroides.</div>


In [ ]:
def winsorizar_adic(s, factor=3.0):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return s.clip(q1 - factor * iqr, q3 + factor * iqr)


def construir_matriz_adic(cfg):
    bloques, nombres = [], []
    if cfg['nums']:
        num = df_adic_base[cfg['nums']].copy()
        if cfg['outlier'] == 'iqr3':
            num = num.apply(lambda s: winsorizar_adic(s, 3.0))
        elif cfg['outlier'] == 'iqr15':
            num = num.apply(lambda s: winsorizar_adic(s, 1.5))
        elif cfg['outlier'] == 'q0199':
            num = num.apply(lambda s: s.clip(s.quantile(.01), s.quantile(.99)))

        if cfg['transform'] == 'log':
            for col in set(cfg['nums']) & {'ingreso_numerico','precio_promedio','m2_minimo'}:
                num[col] = np.log1p(num[col].clip(lower=0))
            scaler = RobustScaler() if cfg.get('scaler') == 'robust' else StandardScaler()
            arr_num = scaler.fit_transform(num)
        elif cfg['transform'] == 'cuantiles':
            arr_num = QuantileTransformer(
                n_quantiles=min(1_000, len(num)), output_distribution='normal',
                random_state=RANDOM_STATE_ADIC
            ).fit_transform(num)
        else:
            arr_num = StandardScaler().fit_transform(num)
        bloques.append(arr_num)
        nombres.extend(cfg['nums'])

    if cfg['cats']:
        cat_blocks = []
        if cfg['encoding'] == 'frecuencia':
            for col in cfg['cats']:
                freq = df_adic_base[col].value_counts(normalize=True)
                cat_blocks.append(df_adic_base[col].map(freq).to_numpy()[:, None])
                nombres.append(f'{col}_frecuencia')
            bloques.append(StandardScaler().fit_transform(np.hstack(cat_blocks)))
        else:
            for col in cfg['cats']:
                enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
                one = enc.fit_transform(df_adic_base[[col]])
                one /= np.sqrt(one.shape[1])
                one *= cfg.get('pesos', {}).get(col, 1.0)
                cat_blocks.append(one)
                nombres.extend([f'{col}_{v}' for v in enc.categories_[0]])
            bloques.append(np.hstack(cat_blocks))

    X = np.hstack(bloques).astype(float)
    meta = {'pca_componentes': None, 'varianza_pca': None}
    if cfg.get('pca'):
        pca = PCA(n_components=cfg['pca'], random_state=RANDOM_STATE_ADIC)
        X = pca.fit_transform(X)
        meta = {'pca_componentes': X.shape[1],
                'varianza_pca': pca.explained_variance_ratio_.sum()}
    return X, nombres, meta

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Evalúa distintos valores de k mediante Silhouette, Davies–Bouldin, Calinski–Harabasz e inercia.</div>


In [ ]:
rng_adic = np.random.default_rng(RANDOM_STATE_ADIC)
idx_eval_adic = rng_adic.choice(
    len(df_adic_base), size=min(N_EVAL_ADIC, len(df_adic_base)), replace=False
)

matrices_adic, curvas_adic = {}, {}
for cfg in EXPERIMENTOS_ADIC:
    X, nombres, meta = construir_matriz_adic(cfg)
    matrices_adic[cfg['id']] = {'X': X, 'nombres': nombres, 'meta': meta}
    Xe = X[idx_eval_adic]
    filas = []
    for k in K_RANGE_ADIC:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE_ADIC,
                    n_init=10, max_iter=500)
        labels = km.fit_predict(Xe)
        filas.append({
            'k': k,
            'inercia': km.inertia_,
            'silhouette': silhouette_score(
                Xe, labels, sample_size=min(2_500, len(Xe)),
                random_state=RANDOM_STATE_ADIC
            ),
            'davies_bouldin': davies_bouldin_score(Xe, labels),
            'calinski_harabasz': calinski_harabasz_score(Xe, labels),
        })
    curvas_adic[cfg['id']] = pd.DataFrame(filas)

for exp_id in EXPERIMENTOS_RELEVANTES:
    cfg = next(c for c in EXPERIMENTOS_ADIC if c['id'] == exp_id)
    curva = curvas_adic[exp_id]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    specs = [('inercia','#5B8DB8','Inercia'),
             ('silhouette','#E07B54','Silhouette (↑)'),
             ('davies_bouldin','#6BAF72','Davies–Bouldin (↓)'),
             ('calinski_harabasz','#9B59B6','Calinski–Harabasz (↑)')]
    for ax, (col, color, title) in zip(axes.flat, specs):
        ax.plot(curva['k'], curva[col], 'o-', color=color, lw=2)
        ax.axvline(cfg['k'], color='black', ls='--', lw=1)
        ax.set(title=title, xlabel='k', ylabel=col)
        ax.grid(alpha=.3)
    plt.suptitle(f"Selección de k — {exp_id}: {cfg['nombre']}", fontsize=13)
    plt.tight_layout()
    plt.show()

### 4.4 Tabla documentada de resultados

Las configuraciones cambian una dimensión metodológica a la vez o conforman una combinación final defendible. Los valores de k registrados abajo integran calidad geométrica, parsimonia, balance y lectura comercial; por eso A7 usa k=6 aunque su Silhouette continúe aumentando hasta k=10.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Entrena el modelo K-Means con la configuración seleccionada y asigna un cluster a cada observación.</div>


In [ ]:
modelos_adic, etiquetas_adic, filas_resumen_adic = {}, {}, []
for cfg in EXPERIMENTOS_ADIC:
    X = matrices_adic[cfg['id']]['X']
    km = KMeans(n_clusters=cfg['k'], random_state=RANDOM_STATE_ADIC,
                n_init=20, max_iter=500)
    labels = km.fit_predict(X)
    modelos_adic[cfg['id']] = km
    etiquetas_adic[cfg['id']] = labels
    Xe, le = X[idx_eval_adic], labels[idx_eval_adic]
    sil = silhouette_score(Xe, le, sample_size=min(N_SIL_ADIC, len(Xe)),
                           random_state=RANDOM_STATE_ADIC)
    filas_resumen_adic.append({
        'nombre del experimento': f"{cfg['id']} — {cfg['nombre']}",
        'variables utilizadas': ', '.join(cfg['nums'] + cfg['cats']),
        'transformaciones aplicadas': (
            f"num={cfg['transform']}; outliers={cfg['outlier']}; "
            f"encoding={cfg['encoding']}" +
            (f"; PCA={cfg['pca']:.0%}" if cfg.get('pca') else '')
        ),
        'k elegido': cfg['k'],
        'Silhouette': sil,
        'Davies-Bouldin': davies_bouldin_score(Xe, le),
        'Calinski-Harabasz': calinski_harabasz_score(Xe, le),
        'interpretación breve': cfg['interpretacion'],
        'observaciones metodológicas': cfg['observacion'],
    })

resultados_adic = pd.DataFrame(filas_resumen_adic)
display(resultados_adic.style.format({
    'Silhouette':'{:.3f}', 'Davies-Bouldin':'{:.3f}',
    'Calinski-Harabasz':'{:,.0f}'
}).background_gradient(subset=['Silhouette'], cmap='YlGn'))

### 4.5 Diagnóstico completo de experimentos relevantes

Las cuatro métricas se calculan para todos los valores de k. Las curvas completas se muestran para los experimentos relevantes; el resto permanece disponible en `curvas_adic`. El k final se fija explícitamente en la configuración y no se asigna mediante `argmax(Silhouette)`.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica las categorías más frecuentes para conocer la composición de las variables cualitativas.</div>


In [ ]:
perfiles_num_adic, perfiles_cat_adic, conversion_adic = {}, {}, {}

for exp_id in EXPERIMENTOS_RELEVANTES:
    cfg = next(c for c in EXPERIMENTOS_ADIC if c['id'] == exp_id)
    X = matrices_adic[exp_id]['X']
    labels = etiquetas_adic[exp_id]
    k = cfg['k']
    colors = sns.color_palette('Set2', k)
    perfil = df_adic_base.copy()
    perfil['cluster'] = labels

    # PCA 2D diagnóstico
    pca_vis = PCA(n_components=2, random_state=RANDOM_STATE_ADIC)
    coords = pca_vis.fit_transform(X)
    idx_plot = rng_adic.choice(len(X), size=min(3_000, len(X)), replace=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    for cluster in range(k):
        idx = idx_plot[labels[idx_plot] == cluster]
        ax.scatter(coords[idx,0], coords[idx,1], s=14, alpha=.45,
                   color=colors[cluster], label=f'Cluster {cluster}')
    ax.set(title=f'PCA 2D — {exp_id}: {cfg["nombre"]}',
           xlabel=f'PC1 ({pca_vis.explained_variance_ratio_[0]:.1%})',
           ylabel=f'PC2 ({pca_vis.explained_variance_ratio_[1]:.1%})')
    ax.legend(ncol=min(k, 3), fontsize=9)
    plt.tight_layout(); plt.show()

    # Tamaños
    dist = pd.Series(labels).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(dist.index, dist.values, color=colors)
    for bar, n in zip(bars, dist.values):
        ax.text(bar.get_x()+bar.get_width()/2, n, f'{n:,}\n({n/len(labels):.1%})',
                ha='center', va='bottom', fontsize=9)
    ax.set(title=f'Tamaño de clusters — {exp_id}', xlabel='Cluster', ylabel='Leads')
    plt.tight_layout(); plt.show()

    # Perfil numérico en unidades originales
    if cfg['nums']:
        pnum = perfil.groupby('cluster')[cfg['nums']].mean()
        perfiles_num_adic[exp_id] = pnum
        z = (pnum - pnum.mean()) / pnum.std().replace(0, 1)
        fig, ax = plt.subplots(figsize=(max(7, k*1.4), 2+len(cfg['nums'])*.65))
        sns.heatmap(z.T, annot=pnum.T.round(1), fmt='g', cmap='RdBu_r', center=0,
                    linewidths=.5, ax=ax, cbar_kws={'label':'Z-score global'})
        ax.set_title(f'Perfil numérico original — {exp_id}')
        plt.tight_layout(); plt.show()

    # Perfil categórico
    perfiles_cat_adic[exp_id] = {}
    for col in cfg['cats']:
        ct = perfil.groupby('cluster')[col].value_counts(normalize=True).unstack(fill_value=0)*100
        perfiles_cat_adic[exp_id][col] = ct
        fig, ax = plt.subplots(figsize=(9, 3.5))
        ct.T.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
        ax.set(title=f'{col} por cluster (%) — {exp_id}', ylabel='%')
        ax.tick_params(axis='x', rotation=25)
        ax.legend(title='Cluster', bbox_to_anchor=(1.01,1), loc='upper left')
        plt.tight_layout(); plt.show()

    # target_won se consulta por primera vez aquí, después del clustering.
    conv = perfil.groupby('cluster')['target_won'].agg(['mean','sum','count'])
    conv.columns = ['tasa_conversion','convertidos','total']
    conv['tasa_%'] = conv['tasa_conversion']*100
    conversion_adic[exp_id] = conv
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(conv.index, conv['tasa_%'], color=colors)
    for bar, rate in zip(bars, conv['tasa_%']):
        ax.text(bar.get_x()+bar.get_width()/2, rate, f'{rate:.2f}%', ha='center', va='bottom')
    ax.set(title=f'Conversión por cluster — {exp_id} (validación externa)',
           xlabel='Cluster', ylabel='Conversión (%)')
    plt.tight_layout(); plt.show()

### 4.6 Interpretación integrada y elección metodológica

**Hallazgo general.** Se supera 0.50 en varias configuraciones. A7 (k=6; Silhouette ≈0.602) ofrece el mejor equilibrio interpretativo; A14 (≈0.661) queda dominado por precio y A10 (≈0.859) reproduce combinaciones categóricas casi directas.

| Exp. | Variables / cambio controlado | Transformaciones | k | Silhouette | DB | CH | Interpretación breve | Observación metodológica |
|---|---|---|---:|---:|---:|---:|---|---|
| A1 | Siete variables | Log + IQR×3 + OHE balanceado | 2 | 0.381 | 1.006 | 3 328 | Dos niveles económico-producto | Control mejorado; separación moderada |
| A2 | Siete variables, mayor peso a razón/tipo | Log + IQR×3 + OHE ponderado | 2 | 0.367 | 1.055 | 3 113 | Prioriza intención y producto | Ponderar empeoró la cohesión |
| A3 | A1 sin sexo | Log + IQR×3 + OHE balanceado | 2 | 0.398 | 0.963 | 3 537 | Perfil económico-producto | Sexo mostró baja discriminación |
| A4 | A1 sin m² mínimo | Log + IQR×3 + OHE balanceado | 4 | 0.249 | 1.304 | 2 100 | Perfiles sin tamaño explícito | Confirma valor discriminante de m² |
| A5 | Ingreso, precio, m², razón y tipo | Log + IQR×3 + OHE ponderado | 2 | 0.458 | 0.837 | 4 905 | Producto e intención | Legible, pero no llega a 0.50 |
| A6 | Cuatro numéricas | Log + IQR×1.5 + RobustScaler | 2 | 0.408 | 0.896 | 3 799 | Dos perfiles cuantitativos | Robusto, pierde contexto categórico |
| **A7** | **Ingreso, precio y m²** | **Log + IQR×1.5 + StandardScaler** | **6** | **0.602** | **0.591** | **9 520** | **Seis estratos económico-producto** | **Recomendado por equilibrio métrica–negocio** |
| A8 | Edad, ingreso y precio | Cuantiles normales + recorte 1–99 % | 2 | 0.642 | 0.528 | 10 599 | Dos niveles de mercado | Dominado por precio; poca riqueza |
| A9 | Razón, tipo y sexo | OHE balanceado | 6 | 0.658 | 0.690 | 4 582 | Combinaciones categóricas | Separación parcialmente tautológica |
| A10 | Razón y tipo | OHE balanceado | 5 | 0.859 | 0.531 | 10 058 | Combinaciones observadas | Rechazado: score alto por circularidad |
| A11 | Siete variables | Log + IQR×3 + frecuencia | 2 | 0.273 | 1.466 | 1 749 | Categorías por prevalencia | Encoding poco defendible y peor score |
| A12 | Siete variables | Log + IQR×3 + OHE + PCA 85 % | 2 | 0.426 | 0.869 | 3 969 | Tres componentes | Mejora A1, no alcanza 0.50 |
| A13 | A12 sin sexo | Log + IQR×3 + OHE + PCA 85 % | 2 | 0.426 | 0.869 | 3 971 | Tres componentes | Quitar sexo casi no cambia el resultado |
| A14 | Siete variables | Cuantiles normales + recorte 1–99 % + OHE | 2 | 0.661 | 0.474 | 14 103 | Dos niveles de mercado | Mejor score no trivial, dominado por precio |

Las cifras corresponden a la ejecución reproducible con semilla 42 y universo de 23 429 registros. Redondeos pueden variar en la última milésima según versión de `scikit-learn`.


### 4.7 Conclusiones del bloque multicriterio

A continuación se revisan los experimentos más relevantes a partir de métricas de separación, estabilidad y desempeño comercial. La comparación se enfoca en la consistencia de los perfiles de cluster y en la utilidad práctica de cada configuración.


### 4.8 Recomendación metodológica

#### Modelo recomendado: A7 — Producto numérico, k=6

A7 alcanza Silhouette ≈ 0.602 con Davies–Bouldin ≈ 0.591 y seis clusters de tamaño sustantivo (aprox. 8.9 %–21.9 % en el ajuste completo). Los perfiles forman estratos claros de capacidad económica y producto: ingreso medio aproximado entre Q13.9 mil y Q34.6 mil, precios promedio entre Q99.3 mil y Q184.6 mil, y m² promedio entre 31 y 43.9. La conversión externa varía aproximadamente entre 0.62 % y 5.10 %, evidencia útil que no participó en el ajuste.

Se recomienda A7 porque conserva una narrativa accionable de mercado y evita que los clusters sean meras etiquetas categóricas. Elegir k=6 sacrifica alrededor de 0.036 puntos de Silhouette frente a k=10, pero reduce fragmentación y evita segmentos excesivamente pequeños.

#### Por qué no se elige automáticamente el mayor Silhouette

- **A10 (≈ 0.859, k=5):** cada cluster reproduce casi directamente combinaciones de `razon_de_compra` y `tipo_unidad`; el resultado es geométricamente fácil, pero cercano a una tabla cruzada y débil como descubrimiento latente.
- **A14 (≈ 0.661, k=2):** el corte separa principalmente el precio Q99 339 del resto. Las distribuciones de razón, tipo y sexo permanecen muy similares, y la conversión solo cambia aproximadamente de 2.45 % a 2.92 %.
- **A8 (≈ 0.642, k=2):** presenta el mismo trade-off de un corte de precio dominante y poca riqueza de perfiles.
- **A9 (≈ 0.658, k=6):** ofrece categorías nítidas, pero parte de la separación es tautológica por la geometría discreta del OHE.

#### Conclusión

El objetivo de Silhouette ≥ 0.50 se alcanza en varias configuraciones. No obstante, **A7 es el candidato principal para tesis** por su equilibrio entre cohesión, separación, parsimonia, tamaños utilizables y sentido comercial. A14 puede presentarse como máximo geométrico no trivial y A10 como prueba de por qué una métrica alta no basta para seleccionar un modelo.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Crea la carpeta de salida y exporta etiquetas, perfiles y métricas para reutilizarlos fuera del notebook.</div>


In [ ]:
OUT_ADIC = os.path.join(OUT_DIR, 'experimentos_adicionales')
os.makedirs(OUT_ADIC, exist_ok=True)

resultados_adic.to_csv(
    os.path.join(OUT_ADIC, 'resumen_experimentos_adicionales.csv'),
    index=False, encoding='utf-8-sig'
)
clusters_out_adic = pd.DataFrame(index=df_adic_base.index)
for exp_id, labels in etiquetas_adic.items():
    clusters_out_adic[f'cluster_{exp_id}'] = labels
clusters_out_adic.to_csv(
    os.path.join(OUT_ADIC, 'clusters_experimentos_adicionales.csv'),
    index_label='indice_original', encoding='utf-8-sig'
)
for exp_id in EXPERIMENTOS_RELEVANTES:
    if exp_id in perfiles_num_adic:
        perfiles_num_adic[exp_id].to_csv(
            os.path.join(OUT_ADIC, f'perfil_numerico_{exp_id}.csv'), encoding='utf-8-sig'
        )
    conversion_adic[exp_id].to_csv(
        os.path.join(OUT_ADIC, f'conversion_{exp_id}.csv'), encoding='utf-8-sig'
    )
print(f'Resultados exportados en: {OUT_ADIC}')

## 5. Experimentos demográficos

---

**Experimentos de clustering orientados a perfiles demográficos de clientes**

**Objetivo sustantivo**

Esta sección constituye un experimento nuevo e independiente. Los experimentos anteriores se conservan sin cambios. El propósito ya no es maximizar la separación de estratos económico-producto, sino identificar **perfiles demográficos y de hogar** interpretables.

Las cinco variables obligatorias —`edad`, `sexo`, `estado_civil`, `razon_de_compra` y `tipo_unidad`— permanecen en todas las configuraciones. Las variables económicas se omiten o reciben pesos reducidos; se audita explícitamente su asociación con los clusters para evitar que ingreso, precio o m² dominen la solución. `target_won` se reserva exclusivamente para validación externa posterior.



### 5.1 Universo y viabilidad de variables familiares

Se define un universo común exigiendo completitud en las cinco variables obligatorias, con el fin de evitar clusters determinados por ausencia de información. Después de construirlo se cuantifican su tamaño y el missingness residual.

`tiene_hijos` se prueba, pero no se incorpora al modelo recomendado por su cobertura inferior y porque reduce la calidad sin añadir suficiente separación estable. Las variables numéricas familiares se imputan por mediana; la ausencia de `tiene_hijos` se representa como `Desconocido` únicamente en los experimentos que la evalúan.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Valida los tipos de datos y convierte a formato numérico las variables que pueden analizarse cuantitativamente.</div>


In [ ]:
RANDOM_STATE_DEMO=42
K_RANGE_DEMO=range(2,11)
N_EVAL_DEMO=6_000
N_SIL_DEMO=5_000

REQ_DEMO=['edad','sexo','estado_civil','razon_de_compra','tipo_unidad']
df_demo=df2.loc[df2[REQ_DEMO].notna().all(axis=1)].copy()
df_demo['ingreso_numerico']=df_demo['ingreso_familiar'].apply(parse_income_f2)

def parse_num_demo(x,top):
    if pd.isna(x): return np.nan
    nums=re.findall(r'\d+(?:\.\d+)?',str(x))
    if not nums: return np.nan
    return top if '+' in str(x) else float(nums[0])

df_demo['participantes_credito_num']=df_demo['participantes_en_credito'].apply(lambda x:parse_num_demo(x,5))
df_demo['nucleo_familiar_num']=df_demo['nucleo_familiar'].apply(lambda x:parse_num_demo(x,6))
NUM_DEMO=['edad','ingreso_numerico','precio_promedio','m2_minimo',
          'participantes_credito_num','nucleo_familiar_num']
for col in NUM_DEMO:
    df_demo[col]=pd.to_numeric(df_demo[col],errors='coerce')
    df_demo[col]=df_demo[col].fillna(df_demo[col].median())
for col in ['sexo','estado_civil','razon_de_compra','tipo_unidad']:
    df_demo[col]=df_demo[col].astype(str)
df_demo['tiene_hijos']=df_demo['tiene_hijos'].fillna('Desconocido').astype(str)
df_demo['rango_edad']=pd.cut(df_demo['edad'],[0,25,35,45,55,200],
    labels=['≤25','26–35','36–45','46–55','>55']).astype(str)

print(f'Universo demográfico común: {len(df_demo):,}')
print('Variables obligatorias completas y target_won excluida del modelado.')

#### 5.1.1 Resultado del universo analítico

El universo resultante contiene **18 315 leads**. El missingness residual es aproximadamente 1.9 % en ingreso, 15.2 % en participantes en crédito, 18.7 % en núcleo familiar y 27.5 % en `tiene_hijos`.


### 5.2 Configuraciones controladas

El OHE balanceado divide cada bloque categórico por √(*número de niveles*). D6 y D7 reducen los pesos de ingreso a 0.30 y de precio/m² a 0.18. D10 conserva 90% de varianza mediante PCA y siempre interpreta perfiles en las variables originales. Los k finales se fijan por parsimonia, balance y lectura comercial, no por `argmax(Silhouette)`.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Declara las configuraciones que se compararán, especificando variables, transformaciones, ponderaciones y valor final de k.</div>


In [ ]:
CAT_REQ_DEMO=['sexo','estado_civil','razon_de_compra','tipo_unidad']
FAM_NUM_DEMO=['participantes_credito_num','nucleo_familiar_num']
CONFIGS_DEMO=[
 dict(id='D1',nombre='Obligatorias base',nums=['edad'],cats=CAT_REQ_DEMO,pesos={},trans='standard',enc='balanceado',pca=None,k=2,obs='Baseline demográfico sin economía.'),
 dict(id='D2',nombre='Obligatorias ponderadas',nums=['edad'],cats=CAT_REQ_DEMO,pesos={'edad':1.2,'sexo':1.15,'estado_civil':1.25,'razon_de_compra':1.1,'tipo_unidad':.9},trans='standard',enc='balanceado',pca=None,k=2,obs='Mayor Silhouette; dos perfiles amplios por edad y estado civil.'),
 dict(id='D3',nombre='Demografía y hogar',nums=['edad']+FAM_NUM_DEMO,cats=CAT_REQ_DEMO,pesos={'sexo':1.1,'estado_civil':1.2,'razon_de_compra':1.1},trans='robust',enc='balanceado',pca=None,k=6,obs='Seis perfiles demográfico-familiares balanceados.'),
 dict(id='D4',nombre='Demografía, hogar e hijos',nums=['edad']+FAM_NUM_DEMO,cats=CAT_REQ_DEMO+['tiene_hijos'],pesos={'sexo':1.1,'estado_civil':1.2,'razon_de_compra':1.1,'tiene_hijos':.8},trans='robust',enc='balanceado',pca=None,k=6,obs='Hijos aporta contexto, pero su missingness reduce cohesión.'),
 dict(id='D5',nombre='Hogar con ingreso atenuado',nums=['edad']+FAM_NUM_DEMO+['ingreso_numerico'],cats=CAT_REQ_DEMO+['tiene_hijos'],pesos={'sexo':1.1,'estado_civil':1.2,'razon_de_compra':1.1,'ingreso_numerico':.35,'tiene_hijos':.8},trans='mixta',enc='balanceado',pca=None,k=3,obs='Ingreso complementario, no conductor.'),
 dict(id='D6',nombre='Perfil ampliado atenuado',nums=['edad']+FAM_NUM_DEMO+['ingreso_numerico','precio_promedio','m2_minimo'],cats=CAT_REQ_DEMO+['tiene_hijos'],pesos={'sexo':1.1,'estado_civil':1.2,'razon_de_compra':1.1,'ingreso_numerico':.30,'precio_promedio':.18,'m2_minimo':.18,'tiene_hijos':.8},trans='mixta',enc='balanceado',pca=None,k=3,obs='Economía controlada; hijos conserva incertidumbre.'),
 dict(id='D7',nombre='Perfil ampliado sin hijos — recomendado',nums=['edad']+FAM_NUM_DEMO+['ingreso_numerico','precio_promedio','m2_minimo'],cats=CAT_REQ_DEMO,pesos={'sexo':1.1,'estado_civil':1.2,'razon_de_compra':1.1,'ingreso_numerico':.30,'precio_promedio':.18,'m2_minimo':.18},trans='mixta',enc='balanceado',pca=None,k=3,obs='Mejor equilibrio: tres perfiles ricos y economía no dominante.'),
 dict(id='D8',nombre='Edad por etapas de vida',nums=FAM_NUM_DEMO,cats=['rango_edad']+CAT_REQ_DEMO+['tiene_hijos'],pesos={'rango_edad':1.2,'sexo':1.1,'estado_civil':1.25,'razon_de_compra':1.1,'tiene_hijos':.8},trans='robust',enc='balanceado',pca=None,k=5,obs='Edad categorizada; pierde gradiente continuo.'),
 dict(id='D9',nombre='OHE estandarizado — contraste',nums=['edad']+FAM_NUM_DEMO,cats=CAT_REQ_DEMO+['tiene_hijos'],pesos={},trans='standard',enc='estandarizado',pca=None,k=4,obs='Categorías raras sobrerrepresentadas; no recomendado.'),
 dict(id='D10',nombre='Demografía-hogar PCA 90%',nums=['edad']+FAM_NUM_DEMO,cats=CAT_REQ_DEMO+['tiene_hijos'],pesos={'sexo':1.1,'estado_civil':1.2,'razon_de_compra':1.1,'tiene_hijos':.8},trans='robust',enc='balanceado',pca=.90,k=6,obs='Conserva ≈91% de varianza y mejora D4; interpretación post hoc.'),
]
RELEVANTES_DEMO=['D2','D3','D4','D7','D10']

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define o aplica el tratamiento de valores atípicos para reducir su influencia sobre las distancias y los centroides.</div>


In [ ]:
def winsor_demo(s,f=3):
    q1,q3=s.quantile([.25,.75]); iqr=q3-q1
    return s.clip(q1-f*iqr,q3+f*iqr)

def matriz_demo(cfg):
    blocks,names=[],[]
    if cfg['nums']:
        num=df_demo[cfg['nums']].copy().apply(winsor_demo)
        if cfg['trans']=='robust': arr=RobustScaler().fit_transform(num)
        else:
            for v in set(cfg['nums'])&{'ingreso_numerico','precio_promedio','m2_minimo'}:
                num[v]=np.log1p(num[v].clip(lower=0))
            arr=StandardScaler().fit_transform(num)
        for j,v in enumerate(cfg['nums']): arr[:,j]*=cfg['pesos'].get(v,1)
        blocks.append(arr); names+=cfg['nums']
    for v in cfg['cats']:
        enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
        one=enc.fit_transform(df_demo[[v]])
        if cfg['enc']=='balanceado': one/=np.sqrt(one.shape[1])
        else: one=StandardScaler().fit_transform(one)
        one*=cfg['pesos'].get(v,1); blocks.append(one)
        names += [f'{v}_{z}' for z in enc.categories_[0]]
    X=np.hstack(blocks); meta={'pca_n':None,'pca_var':None,'pca_loadings':None}
    if cfg.get('pca'):
        pca=PCA(n_components=cfg['pca'],random_state=RANDOM_STATE_DEMO)
        X=pca.fit_transform(X)
        meta={'pca_n':X.shape[1],'pca_var':pca.explained_variance_ratio_.sum(),
              'pca_loadings':pd.DataFrame(pca.components_,columns=names)}
    return X,names,meta

def eta2_demo(values,labels):
    x=np.asarray(values,float); total=((x-x.mean())**2).sum()
    return 0 if total==0 else sum(len(x[labels==g])*(x[labels==g].mean()-x.mean())**2 for g in np.unique(labels))/total

def cramer_demo(values,labels):
    ct=pd.crosstab(values,labels).to_numpy(); n=ct.sum()
    ex=ct.sum(1)[:,None]*ct.sum(0)[None,:]/n
    chi=np.where(ex>0,(ct-ex)**2/ex,0).sum(); q=min(ct.shape)-1
    return 0 if q==0 else np.sqrt((chi/n)/q)

### 5.3 Selección de k, métricas y auditoría de dominancia

Además de Silhouette, Davies–Bouldin y Calinski–Harabasz, se calcula η² para numéricas y V de Cramér para categóricas. Estas asociaciones no eligen el modelo, pero permiten verificar qué variables explican la partición.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Evalúa distintos valores de k mediante Silhouette, Davies–Bouldin, Calinski–Harabasz e inercia.</div>


In [ ]:
rng_demo=np.random.default_rng(RANDOM_STATE_DEMO)
idx_demo=rng_demo.choice(len(df_demo),size=min(N_EVAL_DEMO,len(df_demo)),replace=False)
matrices_demo,curvas_demo={},{ }
for cfg in CONFIGS_DEMO:
    X,nombres,meta=matriz_demo(cfg); matrices_demo[cfg['id']]={'X':X,'nombres':nombres,'meta':meta}
    rows=[]
    for k in K_RANGE_DEMO:
        km=KMeans(n_clusters=k,random_state=RANDOM_STATE_DEMO,n_init=10,max_iter=500)
        lab=km.fit_predict(X[idx_demo])
        rows.append({'k':k,'inercia':km.inertia_,
          'silhouette':silhouette_score(X[idx_demo],lab,sample_size=3_000,random_state=RANDOM_STATE_DEMO),
          'davies_bouldin':davies_bouldin_score(X[idx_demo],lab),
          'calinski_harabasz':calinski_harabasz_score(X[idx_demo],lab)})
    curvas_demo[cfg['id']]=pd.DataFrame(rows)

for exp_id in RELEVANTES_DEMO:
    cfg=next(c for c in CONFIGS_DEMO if c['id']==exp_id); curve=curvas_demo[exp_id]
    fig,axes=plt.subplots(2,2,figsize=(13,8))
    specs=[('inercia','#5B8DB8','Inercia'),('silhouette','#E07B54','Silhouette (↑)'),
           ('davies_bouldin','#6BAF72','Davies–Bouldin (↓)'),
           ('calinski_harabasz','#9B59B6','Calinski–Harabasz (↑)')]
    for ax,(col,color,title_) in zip(axes.flat,specs):
        ax.plot(curve.k,curve[col],'o-',color=color,lw=2); ax.axvline(cfg['k'],color='black',ls='--')
        ax.set(title=title_,xlabel='k',ylabel=col); ax.grid(alpha=.3)
    plt.suptitle(f'Selección de k — {exp_id}: {cfg["nombre"]}'); plt.tight_layout(); plt.show()

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Entrena el modelo K-Means con la configuración seleccionada y asigna un cluster a cada observación.</div>


In [ ]:
labels_demo,models_demo,assoc_demo,summary_demo={},{},{},[]
for cfg in CONFIGS_DEMO:
    X=matrices_demo[cfg['id']]['X']; km=KMeans(cfg['k'],random_state=RANDOM_STATE_DEMO,n_init=20,max_iter=500)
    labels=km.fit_predict(X); le=labels[idx_demo]
    labels_demo[cfg['id']]=labels; models_demo[cfg['id']]=km
    assoc={v:eta2_demo(df_demo[v],labels) for v in cfg['nums']}
    assoc.update({v:cramer_demo(df_demo[v],labels) for v in cfg['cats']}); assoc_demo[cfg['id']]=assoc
    econ=max([assoc.get(v,0) for v in ['ingreso_numerico','precio_promedio','m2_minimo']])
    sizes=np.bincount(labels)/len(labels)
    summary_demo.append({'experimento':f"{cfg['id']} — {cfg['nombre']}",
      'variables':', '.join(cfg['nums']+cfg['cats']),'preprocesamiento':f"{cfg['trans']}; OHE {cfg['enc']}"+(f"; PCA {cfg['pca']:.0%}" if cfg.get('pca') else ''),
      'k':cfg['k'],'Silhouette':silhouette_score(X[idx_demo],le,sample_size=N_SIL_DEMO,random_state=RANDOM_STATE_DEMO),
      'Davies-Bouldin':davies_bouldin_score(X[idx_demo],le),'Calinski-Harabasz':calinski_harabasz_score(X[idx_demo],le),
      'cluster_mínimo_%':sizes.min()*100,'dominancia_económica_máx':econ,'interpretación / observación':cfg['obs']})
resultados_demo=pd.DataFrame(summary_demo)
display(
    resultados_demo.style
    .format({
        'Silhouette':'{:.3f}',
        'Davies-Bouldin':'{:.3f}',
        'Calinski-Harabasz':'{:,.0f}',
        'cluster_mínimo_%':'{:.1f}',
        'dominancia_económica_máx':'{:.3f}'
    })
    .background_gradient(subset=['Silhouette'], cmap='YlGn')
)

### 5.4 Tabla comparativa de experimentos demográficos

Se evalúa la estabilidad de la segmentación cuando se incorporan restricciones demográficas en la definición del universo analítico. Este análisis permite verificar si la estructura de cluster se conserva bajo condiciones más específicas de segmentación.


### 5.5 Diagnósticos de configuraciones relevantes

Se generan PCA 2D, tamaños, perfiles originales, asociaciones variable–cluster y conversión posterior para D2, D3, D4, D7 y D10. Los perfiles se interpretan siempre en unidades originales.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica las categorías más frecuentes para conocer la composición de las variables cualitativas.</div>


In [ ]:
profiles_num_demo,profiles_cat_demo,conversion_demo={},{},{}
for exp_id in RELEVANTES_DEMO:
    cfg=next(c for c in CONFIGS_DEMO if c['id']==exp_id); X=matrices_demo[exp_id]['X']; labels=labels_demo[exp_id]; k=cfg['k']
    colors=sns.color_palette('Set2',k); prof=df_demo.copy(); prof['cluster']=labels
    p=PCA(2,random_state=RANDOM_STATE_DEMO); xy=p.fit_transform(X); ip=rng_demo.choice(len(X),min(3000,len(X)),replace=False)
    fig,ax=plt.subplots(figsize=(10,6))
    for g in range(k):
        ix=ip[labels[ip]==g]; ax.scatter(xy[ix,0],xy[ix,1],s=14,alpha=.45,color=colors[g],label=f'Cluster {g}')
    ax.set(title=f'PCA diagnóstico — {exp_id}',xlabel=f'PC1 ({p.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({p.explained_variance_ratio_[1]:.1%})'); ax.legend(ncol=min(k,3)); plt.tight_layout(); plt.show()
    dist=pd.Series(labels).value_counts().sort_index(); fig,ax=plt.subplots(figsize=(8,4)); bars=ax.bar(dist.index,dist.values,color=colors)
    for b,n in zip(bars,dist): ax.text(b.get_x()+b.get_width()/2,n,f'{n:,}\n({n/len(labels):.1%})',ha='center',va='bottom',fontsize=9)
    ax.set(title=f'Tamaño de clusters — {exp_id}',xlabel='Cluster',ylabel='Leads'); plt.tight_layout(); plt.show()
    numvars=list(dict.fromkeys(cfg['nums']+['edad','participantes_credito_num','nucleo_familiar_num']))
    pn=prof.groupby('cluster')[numvars].mean(); profiles_num_demo[exp_id]=pn; z=(pn-pn.mean())/pn.std().replace(0,1)
    fig,ax=plt.subplots(figsize=(max(8,k*1.3),2+len(numvars)*.6)); sns.heatmap(z.T,annot=pn.T.round(1),fmt='g',cmap='RdBu_r',center=0,ax=ax); ax.set_title(f'Perfil numérico original — {exp_id}'); plt.tight_layout(); plt.show()
    profiles_cat_demo[exp_id]={}
    for col in CAT_REQ_DEMO+(['tiene_hijos'] if 'tiene_hijos' in cfg['cats'] else []):
        ct=prof.groupby('cluster')[col].value_counts(normalize=True).unstack(fill_value=0)*100; profiles_cat_demo[exp_id][col]=ct
        fig,ax=plt.subplots(figsize=(9,3.5)); ct.T.plot(kind='bar',ax=ax,color=colors); ax.set(title=f'{col} por cluster — {exp_id}',ylabel='%'); ax.tick_params(axis='x',rotation=25); ax.legend(bbox_to_anchor=(1.01,1)); plt.tight_layout(); plt.show()
    assoc=pd.Series(assoc_demo[exp_id]).sort_values(); fig,ax=plt.subplots(figsize=(8,4)); assoc.plot.barh(ax=ax,color='#5B8DB8'); ax.set(title=f'Asociación variable–cluster — {exp_id}',xlabel='η² (numéricas) / V de Cramér (categóricas)'); plt.tight_layout(); plt.show()
    conv=prof.groupby('cluster')['target_won'].agg(['mean','sum','count']); conv.columns=['tasa_conversion','convertidos','total']; conv['tasa_%']=conv.tasa_conversion*100; conversion_demo[exp_id]=conv
    fig,ax=plt.subplots(figsize=(8,4)); bars=ax.bar(conv.index,conv['tasa_%'],color=colors)
    for b,v in zip(bars,conv['tasa_%']): ax.text(b.get_x()+b.get_width()/2,v,f'{v:.2f}%',ha='center',va='bottom')
    ax.set(title=f'Conversión posterior — {exp_id}',xlabel='Cluster',ylabel='%'); plt.tight_layout(); plt.show()

### 5.6 Interpretación, hallazgos y recomendación

**Hallazgo general.** Ninguna configuración demográfica inicial alcanza Silhouette ≥ 0.50. D2 obtiene el máximo (≈0.359) con dos perfiles; D7 (≈0.277, k=3) se recomienda cuando se prioriza riqueza de perfiles y baja dominancia económica.

**Auditoría de dominancia.** En D7, la asociación máxima de ingreso, precio o m² es ≈0.029, muy inferior a edad (≈0.581) y participantes en crédito (≈0.768). En D10, el PCA previo conserva aproximadamente 91 % de la varianza.

#### D7 — tres perfiles demográfico-familiares

1. **Adultos jóvenes principalmente solteros y con un participante en crédito:** edad media ≈33.8, núcleo ≈1.75; conversión ≈3.18%.
2. **Adultos mayores predominantemente casados:** edad media ≈53.8, núcleo ≈2.13; conversión ≈4.46%.
3. **Adultos jóvenes/intermedios con estructura de crédito compartida:** edad media ≈35.2, alrededor de 2.09 participantes y núcleo ≈2.34; conversión ≈2.86%.

Los precios medios de los tres clusters son prácticamente iguales (≈Q138.4–138.7 mil) y m² cambia mínimamente (≈34.2–34.9). Esto verifica que D7 **no** es una segmentación económica disfrazada. Edad, participantes, estado civil y razón de compra aportan mucha más estructura.

#### Por qué el Silhouette no llega a 0.50

- `sexo` y `razon_de_compra` tienen solo dos niveles y muestran fuerte solapamiento con edad y hogar.
- `tipo_unidad` está dominada por 2 y 3 habitaciones; su categoría modal cambia poco entre perfiles.
- `nucleo_familiar` tiene rango estrecho y asociación baja con los clusters.
- `tiene_hijos` conserva 27.5% de ausencia aun en el universo completo obligatorio.
- Las variables demográficas describen gradientes y combinaciones superpuestas, no escalones naturalmente separados como precio y m².

Por tanto, **D7 es el modelo principal para perfiles ricos**, mientras D2 puede reportarse como alternativa parsimoniosa con la mejor calidad geométrica. El límite inferior a 0.50 es un hallazgo sustantivo: maximizar más la métrica exigiría volver a estratos económicos o a combinaciones categóricas casi tautológicas, contradiciendo el objetivo.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Crea la carpeta de salida y exporta etiquetas, perfiles y métricas para reutilizarlos fuera del notebook.</div>


In [ ]:
OUT_DEMO=os.path.join(OUT_DIR,'experimentos_perfiles_demograficos'); os.makedirs(OUT_DEMO,exist_ok=True)
resultados_demo.to_csv(os.path.join(OUT_DEMO,'resumen_experimentos_demograficos.csv'),index=False,encoding='utf-8-sig')
clusters_demo=pd.DataFrame(index=df_demo.index)
for exp_id,labels in labels_demo.items(): clusters_demo[f'cluster_{exp_id}']=labels
clusters_demo.to_csv(os.path.join(OUT_DEMO,'clusters_experimentos_demograficos.csv'),index_label='indice_original',encoding='utf-8-sig')
for exp_id in RELEVANTES_DEMO:
    profiles_num_demo[exp_id].to_csv(os.path.join(OUT_DEMO,f'perfil_numerico_{exp_id}.csv'),encoding='utf-8-sig')
    conversion_demo[exp_id].to_csv(os.path.join(OUT_DEMO,f'conversion_{exp_id}.csv'),encoding='utf-8-sig')
print(f'Exportaciones demográficas: {OUT_DEMO}')

---

## 6. Ronda adicional de optimización demográfica con k > 2

**Objetivo**

Esta ronda se añade sin modificar ni sustituir ningún experimento previo. Se buscan configuraciones con **Silhouette ≥ 0.50 y más de dos clusters**, manteniendo siempre `edad`, `sexo`, `estado_civil`, `razon_de_compra` y `tipo_unidad`.

Se prueban ponderaciones demográficas, edad continua con peso reducido, rangos de edad e interacciones categóricas controladas. No se incorporan ingreso, precio ni m²: cualquier mejora proviene exclusivamente de la representación demográfica, civil, conductual y de preferencia de unidad.



<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Declara las configuraciones que se compararán, especificando variables, transformaciones, ponderaciones y valor final de k.</div>


In [ ]:
df_ronda2=df_demo.copy()
df_ronda2['edad_5']=pd.cut(df_ronda2.edad,[0,25,35,45,55,200],labels=['≤25','26–35','36–45','46–55','>55']).astype(str)
df_ronda2['edad_3']=pd.cut(df_ronda2.edad,[0,35,50,200],labels=['≤35','36–50','>50']).astype(str)
df_ronda2['edad_4']=pd.cut(df_ronda2.edad,[0,30,40,50,200],labels=['≤30','31–40','41–50','>50']).astype(str)
df_ronda2['civil_razon']=df_ronda2.estado_civil+' | '+df_ronda2.razon_de_compra
df_ronda2['sexo_civil']=df_ronda2.sexo+' | '+df_ronda2.estado_civil
df_ronda2['razon_tipo']=df_ronda2.razon_de_compra+' | '+df_ronda2.tipo_unidad

CAT_OBL_R2=['sexo','estado_civil','razon_de_compra','tipo_unidad']
CONFIGS_R2=[
 dict(id='R1',nombre='Demografía ponderada con edad continua',edad='continua',peso_edad=.25,cats=CAT_OBL_R2,pesos={'sexo':1.2,'estado_civil':1.5,'razon_de_compra':1.4,'tipo_unidad':1.0},k=6,obs='Recomendado: seis perfiles, todos >12%.'),
 dict(id='R2',nombre='Edad en cinco etapas',edad='edad_5',peso_edad=1,cats=CAT_OBL_R2,pesos={},k=6,obs='Rangos de edad reducen separación.'),
 dict(id='R3',nombre='Edad en tres etapas',edad='edad_3',peso_edad=1.1,cats=CAT_OBL_R2,pesos={'estado_civil':1.2,'razon_de_compra':1.2},k=8,obs='Ocho perfiles, Silhouette moderado.'),
 dict(id='R4',nombre='Edad en cuatro etapas ponderada',edad='edad_4',peso_edad=.8,cats=CAT_OBL_R2,pesos={'sexo':1.1,'estado_civil':1.4,'razon_de_compra':1.4,'tipo_unidad':.9},k=6,obs='Cerca de 0.50, pero inferior a R1.'),
 dict(id='R5',nombre='Edad mínima, civil y razón dominantes',edad='continua',peso_edad=.1,cats=CAT_OBL_R2,pesos={'sexo':.8,'estado_civil':1.5,'razon_de_compra':1.6,'tipo_unidad':1.1},k=6,obs='Queda cerca de 0.50.'),
 dict(id='R6',nombre='Razón y tipo dominantes',edad='continua',peso_edad=.1,cats=CAT_OBL_R2,pesos={'sexo':.5,'estado_civil':.7,'razon_de_compra':1.8,'tipo_unidad':1.5},k=4,obs='Supera 0.50, pero es casi una tabla razón × tipo.'),
 dict(id='R7',nombre='Interacción estado civil–razón',edad='edad_3',peso_edad=.5,cats=CAT_OBL_R2+['civil_razon'],pesos={'civil_razon':1.5,'estado_civil':.5,'razon_de_compra':.5},k=9,obs='Mayor score, pero fragmentación y circularidad.'),
 dict(id='R8',nombre='Interacción razón–tipo',edad='edad_3',peso_edad=.5,cats=CAT_OBL_R2+['razon_tipo'],pesos={'razon_tipo':1.5,'razon_de_compra':.5,'tipo_unidad':.5},k=8,obs='Interacción de producto no mejora suficiente.'),
 dict(id='R9',nombre='Interacción sexo–estado civil',edad='edad_3',peso_edad=.5,cats=CAT_OBL_R2+['sexo_civil'],pesos={'sexo_civil':1.5,'sexo':.5,'estado_civil':.5},k=7,obs='Cercano a 0.50; perfiles más rígidos.'),
 dict(id='R10',nombre='Dos interacciones categóricas',edad='edad_3',peso_edad=.4,cats=CAT_OBL_R2+['civil_razon','razon_tipo'],pesos={'civil_razon':1.2,'razon_tipo':1.2,'estado_civil':.4,'razon_de_compra':.4,'tipo_unidad':.5},k=10,obs='No conserva ≥0.50 en ajuste completo.'),
]
RELEVANTES_R2=['R1','R6','R7']

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Codifica las variables categóricas y construye la matriz numérica que puede utilizar K-Means.</div>


In [ ]:
def matriz_r2(cfg):
    blocks=[]
    if cfg['edad']=='continua':
        blocks.append(StandardScaler().fit_transform(df_ronda2[['edad']])*cfg['peso_edad'])
    else:
        enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
        one=enc.fit_transform(df_ronda2[[cfg['edad']]])/np.sqrt(df_ronda2[cfg['edad']].nunique())
        blocks.append(one*cfg['peso_edad'])
    for col in cfg['cats']:
        enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
        one=enc.fit_transform(df_ronda2[[col]])/np.sqrt(df_ronda2[col].nunique())
        blocks.append(one*cfg['pesos'].get(col,1))
    return np.hstack(blocks)

rng_r2=np.random.default_rng(42); idx_r2=rng_r2.choice(len(df_ronda2),size=6_000,replace=False)
matrices_r2,curvas_r2={},{ }
for cfg in CONFIGS_R2:
    X=matriz_r2(cfg); matrices_r2[cfg['id']]=X; rows=[]
    for k in range(3,11):
        km=KMeans(k,random_state=42,n_init=10,max_iter=500); lab=km.fit_predict(X[idx_r2])
        sizes=np.bincount(lab)/len(lab)
        rows.append({'k':k,'inercia':km.inertia_,
          'silhouette':silhouette_score(X[idx_r2],lab,sample_size=3_000,random_state=42),
          'davies_bouldin':davies_bouldin_score(X[idx_r2],lab),
          'calinski_harabasz':calinski_harabasz_score(X[idx_r2],lab),
          'cluster_min':sizes.min()})
    curvas_r2[cfg['id']]=pd.DataFrame(rows)

for exp_id in RELEVANTES_R2:
    cfg=next(c for c in CONFIGS_R2 if c['id']==exp_id); curve=curvas_r2[exp_id]
    fig,axes=plt.subplots(2,2,figsize=(13,8))
    specs=[('inercia','#5B8DB8','Inercia'),('silhouette','#E07B54','Silhouette (↑)'),('davies_bouldin','#6BAF72','Davies–Bouldin (↓)'),('calinski_harabasz','#9B59B6','Calinski–Harabasz (↑)')]
    for ax,(col,color,tit) in zip(axes.flat,specs):
        ax.plot(curve.k,curve[col],'o-',color=color,lw=2); ax.axvline(cfg['k'],color='black',ls='--'); ax.set(title=tit,xlabel='k',ylabel=col); ax.grid(alpha=.3)
    plt.suptitle(f'Selección de k — {exp_id}: {cfg["nombre"]}'); plt.tight_layout(); plt.show()

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Entrena el modelo K-Means con la configuración seleccionada y asigna un cluster a cada observación.</div>


In [ ]:
labels_r2,models_r2,summary_r2={},{},[]
for cfg in CONFIGS_R2:
    X=matrices_r2[cfg['id']]; km=KMeans(cfg['k'],random_state=42,n_init=20,max_iter=500); labels=km.fit_predict(X); le=labels[idx_r2]
    labels_r2[cfg['id']]=labels; models_r2[cfg['id']]=km; sizes=np.bincount(labels)/len(labels)
    summary_r2.append({'experimento':f"{cfg['id']} — {cfg['nombre']}",'variables':'edad, sexo, estado_civil, razon_de_compra, tipo_unidad','estrategia':f"edad={cfg['edad']}, peso={cfg['peso_edad']}; OHE balanceado",'k':cfg['k'],
      'Silhouette':silhouette_score(X[idx_r2],le,sample_size=5_000,random_state=42),'Davies-Bouldin':davies_bouldin_score(X[idx_r2],le),'Calinski-Harabasz':calinski_harabasz_score(X[idx_r2],le),'cluster_mínimo_%':sizes.min()*100,'observación':cfg['obs']})
resultados_r2=pd.DataFrame(summary_r2)
display(resultados_r2.style.format({'Silhouette':'{:.3f}','Davies-Bouldin':'{:.3f}','Calinski-Harabasz':'{:,.0f}','cluster_mínimo_%':'{:.1f}'}).background_gradient(subset=['Silhouette'],cmap='YlGn'))

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica las categorías más frecuentes para conocer la composición de las variables cualitativas.</div>


In [ ]:
profiles_r2,conversion_r2={},{ }
for exp_id in RELEVANTES_R2:
    cfg=next(c for c in CONFIGS_R2 if c['id']==exp_id); X=matrices_r2[exp_id]; labels=labels_r2[exp_id]; k=cfg['k']; colors=sns.color_palette('Set2',k); prof=df_ronda2.copy(); prof['cluster']=labels
    p=PCA(2,random_state=42); xy=p.fit_transform(X); ip=rng_r2.choice(len(X),min(3000,len(X)),False); fig,ax=plt.subplots(figsize=(10,6))
    for g in range(k):
        ix=ip[labels[ip]==g]; ax.scatter(xy[ix,0],xy[ix,1],s=14,alpha=.45,color=colors[g],label=f'Cluster {g}')
    ax.set(title=f'PCA diagnóstico — {exp_id}',xlabel=f'PC1 ({p.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({p.explained_variance_ratio_[1]:.1%})'); ax.legend(ncol=min(k,3)); plt.tight_layout(); plt.show()
    dist=pd.Series(labels).value_counts().sort_index(); fig,ax=plt.subplots(figsize=(8,4)); bars=ax.bar(dist.index,dist,color=colors)
    for b,n in zip(bars,dist): ax.text(b.get_x()+b.get_width()/2,n,f'{n:,}\n({n/len(labels):.1%})',ha='center',va='bottom',fontsize=9)
    ax.set(title=f'Tamaños — {exp_id}',xlabel='Cluster',ylabel='Leads'); plt.tight_layout(); plt.show()
    profiles_r2[exp_id]={'edad':prof.groupby('cluster').edad.mean()}
    for col in CAT_OBL_R2:
        ct=prof.groupby('cluster')[col].value_counts(normalize=True).unstack(fill_value=0)*100; profiles_r2[exp_id][col]=ct
        fig,ax=plt.subplots(figsize=(9,3.5)); ct.T.plot(kind='bar',ax=ax,color=colors); ax.set(title=f'{col} por cluster — {exp_id}',ylabel='%'); ax.tick_params(axis='x',rotation=25); ax.legend(bbox_to_anchor=(1.01,1)); plt.tight_layout(); plt.show()
    conv=prof.groupby('cluster').target_won.agg(['mean','sum','count']); conv.columns=['tasa_conversion','convertidos','total']; conv['tasa_%']=conv.tasa_conversion*100; conversion_r2[exp_id]=conv
    fig,ax=plt.subplots(figsize=(8,4)); bars=ax.bar(conv.index,conv['tasa_%'],color=colors)
    for b,v in zip(bars,conv['tasa_%']): ax.text(b.get_x()+b.get_width()/2,v,f'{v:.2f}%',ha='center',va='bottom')
    ax.set(title=f'Conversión posterior — {exp_id}',xlabel='Cluster',ylabel='%'); plt.tight_layout(); plt.show()

## 6. Ronda adicional de experimentos demográficos

Se amplían los experimentos demográficos de la sección 5 con una ronda adicional de configuraciones, con el fin de intentar mejorar la separación de los clusters sin perder su interpretabilidad.

### 6.1 Resultados comparativos


| Exp. | Estrategia principal | k | Silhouette | DB | CH | Cluster mínimo | Evaluación |
|---|---|---:|---:|---:|---:|---:|---|
| **R1** | **Edad continua 0.25; sexo 1.2; civil 1.5; razón 1.4** | **6** | **0.518** | **0.904** | **3 815** | **12.2%** | **Recomendado: supera meta con tamaños sanos** |
| R2 | Edad en cinco etapas | 6 | 0.346 | 1.302 | 2 067 | 9.9% | La discretización simple empeora |
| R3 | Edad en tres etapas ponderada | 8 | 0.386 | 1.212 | 1 834 | 7.7% | Más perfiles, separación moderada |
| R4 | Edad en cuatro etapas; civil/razón altos | 6 | 0.465 | 1.020 | 3 346 | 12.2% | Cercano, no alcanza 0.50 |
| R5 | Edad 0.10; civil/razón dominantes | 6 | 0.490 | 1.005 | 4 782 | 12.2% | Queda justo debajo de la meta |
| R6 | Razón y tipo dominantes | 4 | **0.538** | **0.766** | **7 301** | 7.1% | Alto score, casi tabla razón × tipo |
| R7 | Interacción civil–razón | 9 | **0.541** | 0.884 | 2 638 | 4.2% | Máximo, pero fragmentado y circular |
| R8 | Interacción razón–tipo | 8 | 0.396 | 1.165 | 1 982 | 7.7% | No mejora suficiente |
| R9 | Interacción sexo–civil | 7 | 0.488 | 1.026 | 2 489 | 7.8% | Cercano; perfiles rígidos |
| R10 | Dos interacciones | 10 | 0.492 | 0.975 | 2 624 | 4.4% | No conserva ≥0.50 en ajuste completo |


### 6.2 Hallazgos y selección del modelo

**Hallazgo general.** R1, R6 y R7 superan 0.50. R1 (k=6; Silhouette ≈0.518) es el compromiso recomendado; R6 queda dominado por razón/tipo y R7 genera nueve combinaciones casi deterministas, incluida una menor al 5 %.

R1 genera seis perfiles de tamaño razonable (12.2%–20.4%) y tasas de conversión aproximadas entre 2.15% y 5.87%. Distingue compradores de vivienda e inversión, personas solteras/casadas, sexo y diferencias moderadas de edad. No utiliza variables económicas.

Sin embargo, `razon_de_compra` queda prácticamente determinada por cluster y sexo/estado civil presentan asociaciones fuertes. El aumento de Silhouette proviene de dar prioridad a fronteras categóricas claras. Por ello, R1 cumple la meta solicitada y es preferible a R6/R7, pero debe presentarse como **segmentación demográfico-conductual**, no como evidencia de grupos demográficos naturales perfectamente separados.

R6 tiene mejor geometría, pero sus cuatro grupos se aproximan a vivienda de 1, 2 o 3 habitaciones e inversión. R7 logra el mayor score, aunque reproduce interacciones civil–razón, usa nueve clusters y uno cae por debajo de 5%; no se recomienda como modelo principal.


### 6.3 Exportación de la ronda adicional


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Crea la carpeta de salida y exporta etiquetas, perfiles y métricas para reutilizarlos fuera del notebook.</div>


In [ ]:
OUT_R2=os.path.join(OUT_DIR,'ronda_adicional_demografica'); os.makedirs(OUT_R2,exist_ok=True)
resultados_r2.to_csv(os.path.join(OUT_R2,'resumen_ronda_adicional.csv'),index=False,encoding='utf-8-sig')
clusters_r2=pd.DataFrame(index=df_ronda2.index)
for exp_id,labels in labels_r2.items(): clusters_r2[f'cluster_{exp_id}']=labels
clusters_r2.to_csv(os.path.join(OUT_R2,'clusters_ronda_adicional.csv'),index_label='indice_original',encoding='utf-8-sig')
for exp_id in RELEVANTES_R2: conversion_r2[exp_id].to_csv(os.path.join(OUT_R2,f'conversion_{exp_id}.csv'),encoding='utf-8-sig')
print(f'Exportaciones: {OUT_R2}')


## 7. Modelo final R1 sobre el universo completo

Esta sección reproduce el flujo del Experimento 1 para el modelo seleccionado R1: configuración, preparación del universo, entrenamiento final, evaluación, visualización, perfilamiento, interpretación y exportación. La muestra de 6 000 observaciones se utilizó únicamente para comparar configuraciones; el ajuste presentado aquí utiliza las **18 315 observaciones** disponibles.


### 7.1 Configuración y justificación del modelo seleccionado

La sección final consolida la elección del modelo más estable y presenta validaciones estadísticas y de negocio que sustentan la recomendación metodológica. El objetivo es cerrar el análisis con una interpretación rigurosa y accionable de los resultados.


### 7.2 Preparación del universo completo

Se utiliza `df_ronda2`, compuesto por 18 315 leads con las cinco variables obligatorias completas. La matriz R1 contiene edad estandarizada con peso 0.25 y bloques OHE balanceados y ponderados para sexo, estado civil, razón de compra y tipo de unidad.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Recupera la configuración R1 y verifica que su matriz transformada incluya todas las observaciones del universo completo.</div>


In [ ]:
cfg_R1_final = next(cfg for cfg in CONFIGS_R2 if cfg['id'] == 'R1')

# matrices_r2['R1'] fue construida con todas las filas de df_ronda2.
X_R1_full = matrices_r2['R1']
assert X_R1_full.shape[0] == len(df_ronda2), 'La matriz R1 no cubre todo el universo.'



### 7.3 Entrenamiento del modelo K-Means final


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ajusta un nuevo K-Means con k=6 y 50 inicializaciones sobre las 18,315 observaciones completas, y genera sus etiquetas definitivas.</div>


In [ ]:
modelo_R1_full = KMeans(
    n_clusters=cfg_R1_final['k'],
    random_state=42,
    n_init=50,
    max_iter=500,
    algorithm='lloyd'
)
labels_R1_full = modelo_R1_full.fit_predict(X_R1_full)

print('Entrenamiento final R1 completado')
print(f'  Observaciones utilizadas: {len(labels_R1_full):,} de {len(df_ronda2):,}')
print(f'  Variables transformadas:  {X_R1_full.shape[1]:,}')
print(f'  Clusters:                 {modelo_R1_full.n_clusters}')
print(f'  Iteraciones:              {modelo_R1_full.n_iter_}')


### 7.4 Evaluación del modelo final


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula Silhouette, inercia, Davies–Bouldin y Calinski–Harabasz sobre las 18,315 observaciones del universo completo.</div>


In [ ]:
from sklearn.metrics import adjusted_rand_score

metricas_R1_full = pd.DataFrame([{
    'observaciones_entrenamiento': len(X_R1_full),
    'k': modelo_R1_full.n_clusters,
    'inercia_completa': modelo_R1_full.inertia_,
    'silhouette_completo': silhouette_score(X_R1_full, labels_R1_full),
    'davies_bouldin_completo': davies_bouldin_score(X_R1_full, labels_R1_full),
    'calinski_harabasz_completo': calinski_harabasz_score(X_R1_full, labels_R1_full),
    'ARI_vs_ajuste_ronda': adjusted_rand_score(labels_r2['R1'], labels_R1_full)
}])

display(metricas_R1_full.style.format({
    'inercia_completa':'{:,.2f}',
    'silhouette_completo':'{:.3f}',
    'davies_bouldin_completo':'{:.3f}',
    'calinski_harabasz_completo':'{:,.0f}',
    'ARI_vs_ajuste_ronda':'{:.3f}'
}))
print(f'Todas las métricas fueron calculadas sobre las {len(X_R1_full):,} observaciones.')


### 7.5 Visualización — proyección PCA

PCA se utiliza únicamente como diagnóstico visual; K-Means continúa operando en el espacio completo de 13 variables transformadas.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Proyecta y representa las 18,315 observaciones del universo completo junto con los centroides finales.</div>


In [ ]:
pca_R1_full = PCA(n_components=2, random_state=42)
X_R1_pca = pca_R1_full.fit_transform(X_R1_full)
centroides_R1_pca = pca_R1_full.transform(modelo_R1_full.cluster_centers_)
fig, ax = plt.subplots(figsize=(10, 6))
colors_R1 = sns.color_palette('Set2', modelo_R1_full.n_clusters)
for g in range(modelo_R1_full.n_clusters):
    ix = np.flatnonzero(labels_R1_full == g)
    ax.scatter(X_R1_pca[ix,0], X_R1_pca[ix,1], s=10, alpha=.25,
               color=colors_R1[g], label=f'Cluster {g}')
ax.scatter(centroides_R1_pca[:,0], centroides_R1_pca[:,1], marker='X', s=180,
           color='black', edgecolor='white', linewidth=1, label='Centroides')
ax.set(title='PCA del modelo final R1 — universo completo',
       xlabel=f'PC1 ({pca_R1_full.explained_variance_ratio_[0]:.1%})',
       ylabel=f'PC2 ({pca_R1_full.explained_variance_ratio_[1]:.1%})')
ax.legend(ncol=2); ax.grid(alpha=.2); plt.tight_layout(); plt.show()


### 7.6 Distribución de tamaños por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica la asignación final de las 18,315 observaciones para comprobar cobertura total, balance y ausencia de clusters vacíos.</div>


In [ ]:
distribucion_R1_full = (
    pd.Series(labels_R1_full, name='cluster')
      .value_counts().sort_index().rename('observaciones').to_frame()
)
distribucion_R1_full['porcentaje'] = (
    distribucion_R1_full['observaciones'] / len(labels_R1_full) * 100
)
display(distribucion_R1_full.style.format({'porcentaje':'{:.2f}%'}))

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = sns.color_palette('Set2', modelo_R1_full.n_clusters)
bars = ax.bar(distribucion_R1_full.index, distribucion_R1_full['observaciones'], color=colors)
for bar, row in zip(bars, distribucion_R1_full.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{row.observaciones:,}\n({row.porcentaje:.1f}%)',
            ha='center', va='bottom', fontsize=9)
ax.set(title='Distribución final de R1 — universo completo',
       xlabel='Cluster', ylabel='Leads')
plt.tight_layout()
plt.show()

assert distribucion_R1_full['observaciones'].sum() == len(df_ronda2)
assert len(distribucion_R1_full) == cfg_R1_final['k']


### 7.7 Perfilamiento general de los clusters

Los promedios de ingreso utilizan el punto medio del rango declarado; el 1.9% faltante dentro del universo final fue imputado por mediana en la preparación previa. Precio y m² se emplean únicamente como descriptores posteriores: no participaron en R1.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Vincula las etiquetas del ajuste completo con el universo original y prepara las librerías y rutas necesarias para el perfilamiento final.</div>


In [ ]:
from scipy.stats import f_oneway, chi2_contingency, tukey_hsd, norm
from sklearn.metrics import pairwise_distances

OUT_FINAL_R1=os.path.join(OUT_DIR,'modelo_final_R1')
os.makedirs(OUT_FINAL_R1,exist_ok=True)

df_final_r1=df_ronda2.copy()
df_final_r1['cluster']=labels_R1_full
X_final_r1=X_R1_full
modelo_final_r1=modelo_R1_full

NOMBRES_R1={
 0:'Inversionistas casados de edad intermedia',
 1:'Hombres jóvenes solteros orientados a vivienda',
 2:'Hombres casados orientados a vivienda de 3 habitaciones',
 3:'Mujeres jóvenes solteras orientadas a vivienda',
 4:'Mujeres casadas orientadas a vivienda de 3 habitaciones',
 5:'Inversionistas solteros adultos jóvenes'
}

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Mide la conversión por cluster como validación externa; esta variable no interviene en la formación de los segmentos.</div>


In [ ]:
def moda_r1(s): return s.mode().iloc[0]
resumen_clusters_r1=df_final_r1.groupby('cluster').agg(
 observaciones=('cluster','size'), tasa_conversion=('target_won','mean'),
 edad_promedio=('edad','mean'), ingreso_promedio=('ingreso_numerico','mean'),
 precio_promedio=('precio_promedio','mean'), m2_minimo_promedio=('m2_minimo','mean'),
 razon_dominante=('razon_de_compra',moda_r1), tipo_dominante=('tipo_unidad',moda_r1),
 estado_civil_dominante=('estado_civil',moda_r1), sexo_dominante=('sexo',moda_r1)
).reset_index()
resumen_clusters_r1['porcentaje_dataset']=resumen_clusters_r1.observaciones/len(df_final_r1)*100
resumen_clusters_r1['nombre']=resumen_clusters_r1.cluster.map(NOMBRES_R1)
cols=['cluster','nombre','observaciones','porcentaje_dataset','tasa_conversion','edad_promedio',
      'ingreso_promedio','precio_promedio','m2_minimo_promedio','razon_dominante',
      'tipo_dominante','estado_civil_dominante','sexo_dominante']
display(resumen_clusters_r1[cols].style.format({
 'porcentaje_dataset':'{:.1f}%','tasa_conversion':'{:.2%}','edad_promedio':'{:.1f}',
 'ingreso_promedio':'Q{:,.0f}','precio_promedio':'Q{:,.0f}','m2_minimo_promedio':'{:.1f}'
}).background_gradient(subset=['tasa_conversion'],cmap='YlGn'))

**Interpretación de la tabla.** Los clusters están balanceados. Los grupos 0 y 5 reúnen inversionistas y presentan las conversiones más altas; los clusters 2 y 4 agrupan compradores casados orientados a vivienda de tres habitaciones; los clusters 1 y 3 representan compradores jóvenes y solteros orientados a vivienda, diferenciados principalmente por sexo. Ingreso, precio y m² varían entre grupos, pero no determinaron su formación.


### 7.8 Validación estadística de las diferencias entre clusters

#### 7.8.1 Variables numéricas — ANOVA

Se aplica ANOVA de una vía. Dado el tamaño muestral, la significancia se interpreta junto con η²: <0.01 despreciable, 0.01–0.06 pequeño, 0.06–0.14 mediano y >0.14 grande.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define la función auxiliar `eta2_r1`, reutilizada para ejecutar esta etapa de forma consistente y reproducible.</div>


In [ ]:
def eta2_r1(values,labels):
    x=np.asarray(values,float); total=((x-x.mean())**2).sum()
    return 0 if total==0 else sum(len(x[labels==g])*(x[labels==g].mean()-x.mean())**2 for g in np.unique(labels))/total
def interpretar_eta2(x):
    return 'despreciable' if x<.01 else 'pequeño' if x<.06 else 'mediano' if x<=.14 else 'grande'

NUM_TEST_R1=['edad','ingreso_numerico','precio_promedio','m2_minimo']
anova_rows_r1=[]; tukey_rows_r1=[]
for var in NUM_TEST_R1:
    groups=[df_final_r1.loc[df_final_r1.cluster==g,var].to_numpy() for g in range(6)]
    F,p=f_oneway(*groups); eta=eta2_r1(df_final_r1[var],df_final_r1.cluster)
    anova_rows_r1.append({'variable':var,'F':F,'p_value':p,'eta_cuadrado':eta,
                          'magnitud_practica':interpretar_eta2(eta)})
    test=tukey_hsd(*groups); sig=[]; no_sig=[]; contrasts=[]
    for i in range(6):
        for j in range(i+1,6):
            item=f'{i}–{j}'
            (sig if test.pvalue[i,j]<.05 else no_sig).append(item)
            contrasts.append((abs(groups[i].mean()-groups[j].mean()),i,j,groups[i].mean()-groups[j].mean()))
    top=max(contrasts)
    tukey_rows_r1.append({'variable':var,'comparaciones_significativas':len(sig),
      'total_comparaciones':15,'pares_no_significativos':', '.join(no_sig) if no_sig else 'Ninguno',
      'mayor_contraste':f'C{top[1]} vs C{top[2]} ({top[3]:,.2f})'})
anova_r1=pd.DataFrame(anova_rows_r1); resumen_tukey_r1=pd.DataFrame(tukey_rows_r1)
display(anova_r1.style.format({'F':'{:,.2f}','p_value':'{:.2e}','eta_cuadrado':'{:.3f}'}).background_gradient(subset=['eta_cuadrado'],cmap='YlOrRd'))

**Interpretación del ANOVA.** Todas las variables son estadísticamente significativas (p<0.001), pero su relevancia práctica difiere. Edad presenta efecto grande (η²=0.158) y constituye la única variable numérica con diferenciación sustantiva alta. Ingreso se sitúa en el límite superior de efecto pequeño (η²=0.059); m² (0.031) y precio (0.023) tienen efectos pequeños. En consecuencia, la significancia de estas tres variables no implica que R1 sea una estratificación económica.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Realiza comparaciones post hoc entre pares de clusters para localizar diferencias numéricas específicas.</div>


In [ ]:
display(resumen_tukey_r1)

#### 7.8.2 Variables numéricas — comparaciones post hoc

**Interpretación de Tukey HSD.** Edad difiere en las 15 parejas: el orden es C0 (45.0 años), C2 (43.1), C4 (42.1), C5 (36.5), C3 (35.2) y C1 (33.9). Para ingreso, C2, C4 y C5 no difieren entre sí; C0 presenta el promedio mayor y C3 el menor. En precio, C1, C2 y C3 son estadísticamente equivalentes, mientras C5 registra el promedio mayor. Para m², las diferencias relevantes se concentran en C5 y C0 frente a los demás; varias diferencias inferiores a medio metro cuadrado son significativas por el gran n, pero de baja importancia práctica.


#### 7.8.3 Variables categóricas

Se aplica χ² de independencia y V de Cramér: <0.10 despreciable, 0.10–0.20 débil, 0.20–0.40 moderado y >0.40 fuerte.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula la distribución porcentual de las variables categóricas dentro de cada cluster y genera sus gráficos comparativos.</div>


In [ ]:
def interpretar_cramer(x):
    return 'despreciable' if x<.10 else 'débil' if x<.20 else 'moderado' if x<=.40 else 'fuerte'
cat_rows_r1=[]
for var in ['sexo','estado_civil','razon_de_compra','tipo_unidad']:
    ct=pd.crosstab(df_final_r1.cluster,df_final_r1[var])
    chi,p,_,_=chi2_contingency(ct); q=min(ct.shape)-1
    v=np.sqrt((chi/ct.to_numpy().sum())/q)
    cat_rows_r1.append({'variable':var,'chi_cuadrado':chi,'p_value':p,
                        'cramer_v':v,'magnitud_practica':interpretar_cramer(v)})
categoricas_r1=pd.DataFrame(cat_rows_r1)
display(categoricas_r1.style.format({'chi_cuadrado':'{:,.1f}','p_value':'{:.2e}','cramer_v':'{:.3f}'}).background_gradient(subset=['cramer_v'],cmap='YlOrRd'))

**Interpretación categórica.** Todas las asociaciones son significativas (p<0.001). `razon_de_compra` tiene asociación perfecta (V=1.000), porque cada cluster queda compuesto exclusivamente por vivienda o inversión. Sexo (V=0.858) y estado civil (V=0.713) muestran asociaciones fuertes y dan contenido demográfico a los perfiles. `tipo_unidad` solo alcanza asociación débil (V=0.151): aporta matices de producto, pero no domina la segmentación. Comercialmente, R1 distingue primero motivación, sexo y situación civil; la unidad preferida refina el perfil.


### 7.9 Variables que más diferencian los clusters


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Cuantifica la asociación entre las variables originales y los clusters mediante tamaños de efecto y pruebas estadísticas.</div>


In [ ]:
ranking_r1=pd.concat([
 anova_r1[['variable','eta_cuadrado']].rename(columns={'eta_cuadrado':'tamano_efecto'}).assign(metrica='η²'),
 categoricas_r1[['variable','cramer_v']].rename(columns={'cramer_v':'tamano_efecto'}).assign(metrica='V de Cramér')
]).sort_values('tamano_efecto',ascending=False).reset_index(drop=True)
ranking_r1.index=ranking_r1.index+1; ranking_r1.index.name='rango'
display(ranking_r1.style.format({'tamano_efecto':'{:.3f}'}).background_gradient(subset=['tamano_efecto'],cmap='YlOrRd'))
fig,ax=plt.subplots(figsize=(9,5)); plot_rank=ranking_r1.sort_values('tamano_efecto')
ax.barh(plot_rank.variable,plot_rank.tamano_efecto,color='#5B8DB8'); ax.set(title='Poder discriminante de las variables en el modelo final R1',xlabel='Tamaño de efecto',ylabel='Variable')
plt.tight_layout(); plt.savefig(os.path.join(OUT_FINAL_R1,'poder_discriminante_R1.png'),dpi=300,bbox_inches='tight'); plt.show()

**Interpretación del ranking.** La segmentación está impulsada por razón de compra, sexo y estado civil. Edad ocupa el primer lugar entre las numéricas y exhibe efecto grande. Tipo de unidad es débil; ingreso, m² y precio presentan efectos pequeños y, además, no participaron en el ajuste. Este orden confirma que R1 representa perfiles demográfico-conductuales, aunque la asociación perfecta de razón de compra exige reconocer un componente de segmentación casi determinista.


### 7.10 Perfil detallado de cada cluster

#### Cluster 0 — Inversionistas casados de edad intermedia

Incluye 2 907 leads (15.9%). Su edad media es 45.0 años, ingreso medio Q30 369, precio de interés Q144 296 y m² mínimo 35.6. El 100% declara inversión; predomina 2 habitaciones (59.1%), estado casado (99.9%) y sexo masculino (65.6%). Su conversión es 5.26%. El grupo se distingue por combinar motivación de inversión, matrimonio, mayor edad e ingreso más alto que el resto.

#### Cluster 1 — Hombres jóvenes solteros orientados a vivienda

Reúne 3 656 leads (20.0%). Edad media 33.9 años, ingreso Q24 893, precio Q134 606 y m² 34.0. Está compuesto por vivienda (100%), hombres (100%) y principalmente solteros (88.6%); predomina 2 habitaciones (54.5%). Convierte 3.09%. Se diferencia de C3 por sexo y de C2 por menor edad y soltería.

#### Cluster 2 — Hombres casados orientados a vivienda de 3 habitaciones

Es el grupo mayor: 3 729 leads (20.4%). Edad media 43.1 años, ingreso Q28 004, precio Q135 030 y m² 33.8. Está formado por vivienda (100%), hombres casados (100%) y muestra preferencia modal por 3 habitaciones (53.2%). Conversión 2.49%. Frente a C1 representa un perfil masculino de mayor edad y casado; frente a C4 comparte estado civil, motivación y tipo, pero difiere por sexo.

#### Cluster 3 — Mujeres jóvenes solteras orientadas a vivienda

Contiene 3 304 leads (18.0%). Edad media 35.2 años, ingreso Q23 710, precio Q134 778 y m² 33.9. Está compuesto por vivienda (100%), mujeres (100%) y principalmente solteras (89.0%); predomina 2 habitaciones (53.1%). Registra la menor conversión, 2.15%. Es el análogo femenino de C1, con ingreso y conversión inferiores en los descriptivos observados.

#### Cluster 4 — Mujeres casadas orientadas a vivienda de 3 habitaciones

Agrupa 2 489 leads (13.6%). Edad media 42.1 años, ingreso Q28 015, precio Q138 294 y m² 34.2. Está compuesto por vivienda (100%), mujeres casadas (100%) y preferencia modal por 3 habitaciones (52.0%). Conversión 2.29%. Comparte con C2 edad, ingreso, estado civil, motivación y producto; su principal diferenciador es sexo.

#### Cluster 5 — Inversionistas solteros adultos jóvenes

Es el menor, aunque sustantivo: 2 230 leads (12.2%). Edad media 36.5 años, ingreso Q28 292, precio Q149 067 y m² 36.7. El 100% declara inversión; predomina soltería (97.6%), sexo masculino (63.9%) y 2 habitaciones (56.6%). Presenta la conversión más alta, 5.87%. Se diferencia de C0 por menor edad y soltería, y de todos los perfiles de vivienda por motivación, precio y conversión.


### 7.11 Conversión por cluster

Se calculan intervalos de Wilson al 95%, lift respecto a la tasa global y una prueba bilateral de dos proporciones entre cada cluster y el resto del universo.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Mide la conversión por cluster como validación externa; esta variable no interviene en la formación de los segmentos.</div>


In [ ]:
global_conversion_r1=df_final_r1.target_won.mean(); conv_rows=[]
for g,x in df_final_r1.groupby('cluster'):
    n=len(x); wins=int(x.target_won.sum()); ph=wins/n; z=1.96
    den=1+z*z/n; center=(ph+z*z/(2*n))/den
    half=z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    rest=df_final_r1.loc[df_final_r1.cluster!=g,'target_won']; pooled=(wins+rest.sum())/(n+len(rest))
    se=np.sqrt(pooled*(1-pooled)*(1/n+1/len(rest))); p_vs=2*norm.sf(abs((ph-rest.mean())/se))
    conv_rows.append({'cluster':g,'observaciones':n,'convertidos':wins,'tasa_conversion':ph,
      'IC95_inferior':center-half,'IC95_superior':center+half,'lift':ph/global_conversion_r1,
      'p_vs_resto':p_vs,'clasificacion':('superior' if ph>rest.mean() and p_vs<.05 else 'inferior' if ph<rest.mean() and p_vs<.05 else 'sin diferencia')})
conversion_final_r1=pd.DataFrame(conv_rows)
display(conversion_final_r1.style.format({'tasa_conversion':'{:.2%}','IC95_inferior':'{:.2%}','IC95_superior':'{:.2%}','lift':'{:.2f}×','p_vs_resto':'{:.2e}'}).background_gradient(subset=['lift'],cmap='YlGn'))

fig,ax=plt.subplots(figsize=(9,5)); rates=conversion_final_r1.tasa_conversion*100
err=np.vstack([(conversion_final_r1.tasa_conversion-conversion_final_r1.IC95_inferior)*100,(conversion_final_r1.IC95_superior-conversion_final_r1.tasa_conversion)*100])
bars=ax.bar(conversion_final_r1.cluster,rates,yerr=err,capsize=5,color=sns.color_palette('Set2',6),edgecolor='white')
ax.axhline(global_conversion_r1*100,color='red',ls='--',label=f'Conversión global: {global_conversion_r1:.2%}')
for b,v,lift in zip(bars,rates,conversion_final_r1.lift): ax.text(b.get_x()+b.get_width()/2,v+.25,f'{v:.2f}%\n{lift:.2f}×',ha='center',fontsize=9)
ax.set(title='Conversión e intervalos de confianza del 95% por cluster — R1',xlabel='Cluster',ylabel='Conversión (%)'); ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_FINAL_R1,'conversion_clusters_R1.png'),dpi=300,bbox_inches='tight'); plt.show()

**Interpretación de conversión.** La tasa global es 3.37%. C5 alcanza 5.87% (lift 1.74; IC95% 4.97%–6.93%) y C0 5.26% (lift 1.56; IC95% 4.51%–6.14%); ambos superan significativamente al resto (p<0.001). C1 no difiere significativamente (3.09%; p=0.289). C2, C3 y C4 convierten significativamente menos, con lifts de 0.74, 0.64 y 0.68. La prioridad comercial debe concentrar respuesta rápida y seguimiento intensivo en C5 y C0. C1 amerita atención estándar; C2–C4 requieren nutrición y revisión de barreras antes de aumentar inversión comercial.


### 7.12 Separación entre clusters

Las distancias se calculan en el espacio exacto de R1: edad estandarizada y ponderada, más bloques OHE balanceados y ponderados.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula las distancias entre centroides para identificar qué clusters son más próximos y cuáles están más separados.</div>


In [ ]:
dist_centroides_r1=pairwise_distances(modelo_final_r1.cluster_centers_)
tabla_dist_r1=pd.DataFrame(dist_centroides_r1,index=[f'C{i}' for i in range(6)],columns=[f'C{i}' for i in range(6)])
tri=np.triu_indices(6,1); values=dist_centroides_r1[tri]; imin=values.argmin(); imax=values.argmax()
resumen_dist_r1=pd.DataFrame([{'distancia_promedio':values.mean(),'distancia_minima':values.min(),
 'clusters_mas_similares':f'C{tri[0][imin]}–C{tri[1][imin]}','distancia_maxima':values.max(),
 'clusters_mas_diferentes':f'C{tri[0][imax]}–C{tri[1][imax]}'}])
display(tabla_dist_r1.style.format('{:.3f}').background_gradient(cmap='YlOrRd',axis=None))
display(resumen_dist_r1.style.format({'distancia_promedio':'{:.3f}','distancia_minima':'{:.3f}','distancia_maxima':'{:.3f}'}))
fig,ax=plt.subplots(figsize=(7,6)); sns.heatmap(tabla_dist_r1,annot=True,fmt='.3f',cmap='YlOrRd',ax=ax,square=True)
ax.set_title('Distancia euclidiana entre centroides del modelo final R1'); plt.tight_layout()
plt.savefig(os.path.join(OUT_FINAL_R1,'distancias_centroides_R1.png'),dpi=300,bbox_inches='tight'); plt.show()

**Interpretación de distancias.** La distancia promedio es 1.555. C3–C4 son los más próximos (1.179): ambos son perfiles femeninos orientados a vivienda, aunque difieren en edad, estado civil y tipo modal. C4–C5 son los más separados (2.016): contraponen mujeres casadas orientadas a vivienda de tres habitaciones con inversionistas solteros más jóvenes y mayoritariamente masculinos. La geometría coincide con la lectura empresarial de los perfiles.


### 7.13 Visualizaciones adicionales

Se selecciona un mapa de calor estandarizado porque resume simultáneamente variables numéricas y proporciones categóricas relevantes sin repetir las gráficas anteriores.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume las variables originales por cluster para identificar las características que distinguen cada perfil.</div>


In [ ]:
profile_visual_r1=df_final_r1.groupby('cluster').agg(
 edad=('edad','mean'),ingreso=('ingreso_numerico','mean'),precio=('precio_promedio','mean'),m2=('m2_minimo','mean'),
 inversion=('razon_de_compra',lambda s:(s=='Inversión').mean()*100),
 casados=('estado_civil',lambda s:(s=='Casad@').mean()*100),
 femenino=('sexo',lambda s:(s=='Femenino').mean()*100),
 tres_habitaciones=('tipo_unidad',lambda s:(s=='3 Habs').mean()*100))
z_profile=(profile_visual_r1-profile_visual_r1.mean())/profile_visual_r1.std().replace(0,1)
fig,ax=plt.subplots(figsize=(10,6)); sns.heatmap(z_profile.T,annot=True,fmt='.2f',cmap='RdBu_r',center=0,linewidths=.5,ax=ax,cbar_kws={'label':'Z-score entre clusters'})
ax.set(title='Comparación estandarizada de los perfiles del modelo final R1',xlabel='Cluster',ylabel='Indicador'); plt.tight_layout()
plt.savefig(os.path.join(OUT_FINAL_R1,'perfil_estandarizado_R1.png'),dpi=300,bbox_inches='tight'); plt.show()

**Interpretación de la visualización.** C0 sobresale por edad, ingreso e inversión; C5 por inversión, precio y m²; C2 y C4 por matrimonio y tres habitaciones; C1 y C3 por menor edad y soltería. El patrón confirma que los clusters no se ordenan simplemente de menor a mayor poder adquisitivo, sino por combinaciones demográficas, motivacionales y de producto.


### 7.14 Interpretación de negocio e implicaciones para la empresa

- **Prioridad comercial alta — C5 y C0.** Ambos segmentos de inversión convierten significativamente por encima del promedio. C5 requiere respuesta rápida para leads solteros de menor edad interesados mayoritariamente en dos habitaciones y en el rango de precio más alto observado. C0 justifica seguimiento especializado para inversionistas casados de mayor edad e ingreso. Los argumentos deben centrarse en la motivación de inversión declarada; cualquier promesa de rentabilidad debe sustentarse con información financiera adicional no disponible en esta base.
- **Seguimiento estándar — C1.** Su conversión no difiere de la global. El equipo puede priorizar alternativas de dos habitaciones cercanas al precio promedio observado y un proceso de acompañamiento acorde con su menor ingreso relativo.
- **Nutrición diferenciada — C2 y C4.** Son compradores casados de vivienda, con preferencia modal por tres habitaciones, pero conversión inferior. Conviene revisar disponibilidad, adecuación y objeciones asociadas a esa tipología antes de intensificar contacto. C2 y C4 no deben recibir estrategias distintas únicamente por sexo; este atributo se utiliza para describir, no para discriminar.
- **Revisión de barreras — C3.** Presenta la menor conversión y el menor ingreso promedio. El seguimiento debe validar tempranamente presupuesto y adecuación de unidades de dos habitaciones, evitando dedicar recursos intensivos antes de confirmar viabilidad.
- **Marketing.** La principal bifurcación accionable es inversión frente a vivienda. Las campañas y mensajes deben separarse por motivación y luego adaptarse por estado civil, edad y tipo de unidad. Sexo no debe emplearse como criterio excluyente; su valor es analítico y descriptivo.
- **Priorización.** El lift de conversión permite ordenar la atención C5 → C0 → C1 → C2/C4 → C3, sujeto a capacidad operativa y sin reemplazar la evaluación individual del lead.


### 7.15 Limitaciones

- El universo final incluye 18 315 de 42 464 leads calificados porque exige completitud en las cinco variables obligatorias. Esta selección puede introducir sesgo frente a leads con información demográfica incompleta.
- Ingreso se deriva del punto medio de rangos declarados y una fracción residual se imputa por mediana; no equivale a ingreso verificado. Precio y m² describen el proyecto de interés y no necesariamente la compra final.
- El CRM no dispone con cobertura suficiente de hijos, dependientes, ocupación, estabilidad laboral, capacidad de cuota, patrimonio, localización residencial ni comportamiento detallado de seguimiento. Estas ausencias limitan la riqueza del perfil.
- K-Means presupone distancias euclidianas, centroides y grupos aproximadamente compactos. R1 aplica el algoritmo a edad estandarizada y categorías OHE ponderadas; las ponderaciones reflejan una decisión metodológica explícita.
- `razon_de_compra` presenta V de Cramér=1.000 y queda determinada por cluster. Esto mejora separación, pero introduce un componente casi circular que debe declararse al interpretar el modelo.
- La conversión se usa después del clustering y no demuestra causalidad. Las diferencias pueden reflejar variables omitidas o condiciones comerciales no registradas.
- Futuros enriquecimientos del CRM deberían mejorar cobertura y estandarización de hogar, dependientes, empleo, presupuesto, financiamiento e interacciones comerciales. Con esos datos conviene evaluar estabilidad temporal, validación externa y algoritmos para datos mixtos, manteniendo R1 como referencia congelada.


### 7.16 Exportación del modelo final y resultados


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Crea la carpeta de salida y exporta etiquetas, perfiles y métricas para reutilizarlos fuera del notebook.</div>


In [ ]:
resumen_clusters_r1.to_csv(os.path.join(OUT_FINAL_R1,'resumen_clusters_R1.csv'),index=False,encoding='utf-8-sig')
anova_r1.to_csv(os.path.join(OUT_FINAL_R1,'anova_eta2_R1.csv'),index=False,encoding='utf-8-sig')
resumen_tukey_r1.to_csv(os.path.join(OUT_FINAL_R1,'tukey_resumen_R1.csv'),index=False,encoding='utf-8-sig')
categoricas_r1.to_csv(os.path.join(OUT_FINAL_R1,'chi2_cramer_R1.csv'),index=False,encoding='utf-8-sig')
ranking_r1.to_csv(os.path.join(OUT_FINAL_R1,'ranking_variables_R1.csv'),encoding='utf-8-sig')
conversion_final_r1.to_csv(os.path.join(OUT_FINAL_R1,'conversion_intervalos_lift_R1.csv'),index=False,encoding='utf-8-sig')
tabla_dist_r1.to_csv(os.path.join(OUT_FINAL_R1,'distancias_centroides_R1.csv'),encoding='utf-8-sig')
df_final_r1[['cluster']].to_csv(os.path.join(OUT_FINAL_R1,'asignacion_clusters_R1.csv'),index_label='indice_original',encoding='utf-8-sig')
print(f'Resultados finales R1 exportados en: {OUT_FINAL_R1}')

import joblib
pd.DataFrame({'cluster_R1_final': labels_R1_full}, index=df_ronda2.index).to_csv(
    os.path.join(OUT_FINAL_R1, 'asignacion_R1_universo_completo.csv'),
    index_label='indice_original', encoding='utf-8-sig')
metricas_R1_full.to_csv(os.path.join(OUT_FINAL_R1, 'metricas_R1_universo_completo.csv'), index=False, encoding='utf-8-sig')
distribucion_R1_full.to_csv(os.path.join(OUT_FINAL_R1, 'distribucion_R1_universo_completo.csv'), encoding='utf-8-sig')
joblib.dump(modelo_R1_full, os.path.join(OUT_FINAL_R1, 'modelo_KMeans_R1_universo_completo.joblib'))
print('Modelo R1 completo, métricas y asignaciones exportados.')


## 9. Modelo final R1 sin variables económicas descriptivas

Esta sección replica el flujo de la sección 7 para el modelo seleccionado R1, pero elimina `precio_promedio`, `m2_minimo` e `ingreso_numerico` del perfilamiento, las validaciones estadísticas, las visualizaciones y las exportaciones. Estas variables tampoco participaron en la formación original de los clusters. El ajuste utiliza las **18 315 observaciones** disponibles.

> **Nota de corrección (2026-08-12).** Las variables, archivos y gráficos de esta sección usaban internamente el prefijo/sufijo `R9`, que coincidía por error con el identificador de una configuración distinta ("R9 — Interacción sexo–civil") de la tabla comparativa de la Sección 8. El modelo entrenado aquí siempre fue **R1 sin variables económicas** — el cálculo nunca fue incorrecto — pero el nombre generaba confusión con la configuración R9 real. Se corrigió toda la nomenclatura interna (variables, nombres de archivo, carpeta de salida, títulos de gráficos) a `R1_sin_economicas`. Ejecute **Kernel → Restart Kernel and Run All Cells** para regenerar las salidas de esta sección con los nombres corregidos.


### 9.1 Configuración y justificación del modelo seleccionado

Se conserva la configuración R1 seleccionada previamente. La réplica permite presentar una lectura estrictamente demográfico-conductual y de producto, sin incorporar ingreso, precio ni superficie mínima como descriptores posteriores.

### 9.2 Preparación del universo completo

Se utiliza `df_ronda2`, compuesto por 18 315 leads con las cinco variables obligatorias completas. La matriz R1 contiene edad estandarizada con peso 0.25 y bloques OHE balanceados y ponderados para sexo, estado civil, razón de compra y tipo de unidad. `ingreso_numerico`, `precio_promedio` y `m2_minimo` quedan excluidas de toda la sección.

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Recupera la configuración R1 (sin variables económicas) y verifica que su matriz transformada incluya todas las observaciones del universo completo.</div>


In [ ]:
cfg_r1_sin_econ = next(cfg for cfg in CONFIGS_R2 if cfg['id'] == 'R1')

# matrices_r2['R1'] fue construida con todas las filas de df_ronda2.
X_r1_sin_econ_full = matrices_r2['R1']
assert X_r1_sin_econ_full.shape[0] == len(df_ronda2), 'La matriz R1 (sin variables económicas) no cubre todo el universo.'



### 9.3 Entrenamiento del modelo K-Means final


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ajusta un nuevo K-Means con k=6 y 50 inicializaciones sobre las 18,315 observaciones completas, y genera sus etiquetas definitivas.</div>


In [ ]:
modelo_r1_sin_econ_full = KMeans(
    n_clusters=cfg_r1_sin_econ['k'],
    random_state=42,
    n_init=50,
    max_iter=500,
    algorithm='lloyd'
)
labels_r1_sin_econ_full = modelo_r1_sin_econ_full.fit_predict(X_r1_sin_econ_full)

print('Entrenamiento final R1 (sin variables económicas) completado')
print(f'  Observaciones utilizadas: {len(labels_r1_sin_econ_full):,} de {len(df_ronda2):,}')
print(f'  Variables transformadas:  {X_r1_sin_econ_full.shape[1]:,}')
print(f'  Clusters:                 {modelo_r1_sin_econ_full.n_clusters}')
print(f'  Iteraciones:              {modelo_r1_sin_econ_full.n_iter_}')


### 9.4 Evaluación del modelo final


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula Silhouette, inercia, Davies–Bouldin y Calinski–Harabasz sobre las 18,315 observaciones del universo completo.</div>


In [ ]:
from sklearn.metrics import adjusted_rand_score

metricas_r1_sin_econ_full = pd.DataFrame([{
    'observaciones_entrenamiento': len(X_r1_sin_econ_full),
    'k': modelo_r1_sin_econ_full.n_clusters,
    'inercia_completa': modelo_r1_sin_econ_full.inertia_,
    'silhouette_completo': silhouette_score(X_r1_sin_econ_full, labels_r1_sin_econ_full),
    'davies_bouldin_completo': davies_bouldin_score(X_r1_sin_econ_full, labels_r1_sin_econ_full),
    'calinski_harabasz_completo': calinski_harabasz_score(X_r1_sin_econ_full, labels_r1_sin_econ_full),
    'ARI_vs_ajuste_ronda': adjusted_rand_score(labels_r2['R1'], labels_r1_sin_econ_full)
}])

display(metricas_r1_sin_econ_full.style.format({
    'inercia_completa':'{:,.2f}',
    'silhouette_completo':'{:.3f}',
    'davies_bouldin_completo':'{:.3f}',
    'calinski_harabasz_completo':'{:,.0f}',
    'ARI_vs_ajuste_ronda':'{:.3f}'
}))
print(f'Todas las métricas fueron calculadas sobre las {len(X_r1_sin_econ_full):,} observaciones.')


### 9.5 Visualización — proyección PCA

PCA se utiliza únicamente como diagnóstico visual; K-Means continúa operando en el espacio completo de 13 variables transformadas.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Proyecta y representa las 18,315 observaciones del universo completo junto con los centroides finales.</div>


In [ ]:
pca_r1_sin_econ_full = PCA(n_components=2, random_state=42)
X_r1_sin_econ_pca = pca_r1_sin_econ_full.fit_transform(X_r1_sin_econ_full)
centroides_r1_sin_econ_pca = pca_r1_sin_econ_full.transform(modelo_r1_sin_econ_full.cluster_centers_)
fig, ax = plt.subplots(figsize=(10, 6))
colors_r1_sin_econ = sns.color_palette('Set2', modelo_r1_sin_econ_full.n_clusters)
for g in range(modelo_r1_sin_econ_full.n_clusters):
    ix = np.flatnonzero(labels_r1_sin_econ_full == g)
    ax.scatter(X_r1_sin_econ_pca[ix,0], X_r1_sin_econ_pca[ix,1], s=10, alpha=.25,
               color=colors_r1_sin_econ[g], label=f'Cluster {g}')
ax.scatter(centroides_r1_sin_econ_pca[:,0], centroides_r1_sin_econ_pca[:,1], marker='X', s=180,
           color='black', edgecolor='white', linewidth=1, label='Centroides')
ax.set(title='PCA del modelo final R1 (sin variables económicas) — universo completo',
       xlabel=f'PC1 ({pca_r1_sin_econ_full.explained_variance_ratio_[0]:.1%})',
       ylabel=f'PC2 ({pca_r1_sin_econ_full.explained_variance_ratio_[1]:.1%})')
ax.legend(ncol=2); ax.grid(alpha=.2); plt.tight_layout(); plt.show()


### 9.6 Distribución de tamaños por cluster


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume y grafica la asignación final de las 18,315 observaciones para comprobar cobertura total, balance y ausencia de clusters vacíos.</div>


In [ ]:
distribucion_r1_sin_econ_full = (
    pd.Series(labels_r1_sin_econ_full, name='cluster')
      .value_counts().sort_index().rename('observaciones').to_frame()
)
distribucion_r1_sin_econ_full['porcentaje'] = (
    distribucion_r1_sin_econ_full['observaciones'] / len(labels_r1_sin_econ_full) * 100
)
display(distribucion_r1_sin_econ_full.style.format({'porcentaje':'{:.2f}%'}))

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = sns.color_palette('Set2', modelo_r1_sin_econ_full.n_clusters)
bars = ax.bar(distribucion_r1_sin_econ_full.index, distribucion_r1_sin_econ_full['observaciones'], color=colors)
for bar, row in zip(bars, distribucion_r1_sin_econ_full.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{row.observaciones:,}\n({row.porcentaje:.1f}%)',
            ha='center', va='bottom', fontsize=9)
ax.set(title='Distribución final de R1 (sin variables económicas) — universo completo',
       xlabel='Cluster', ylabel='Leads')
plt.tight_layout()
plt.show()

assert distribucion_r1_sin_econ_full['observaciones'].sum() == len(df_ronda2)
assert len(distribucion_r1_sin_econ_full) == cfg_r1_sin_econ['k']


### 9.7 Perfilamiento general de los clusters

El perfilamiento utiliza únicamente edad, razón de compra, tipo de unidad, estado civil, sexo y conversión. No se calculan ni muestran indicadores de ingreso, precio o m² mínimo.

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Vincula las etiquetas del ajuste completo con el universo original y prepara las librerías y rutas necesarias para el perfilamiento final.</div>


In [ ]:
from scipy.stats import f_oneway, chi2_contingency, tukey_hsd, norm
from sklearn.metrics import pairwise_distances

OUT_FINAL_R1_SIN_ECON=os.path.join(OUT_DIR,'modelo_final_R1_sin_economicas')
os.makedirs(OUT_FINAL_R1_SIN_ECON,exist_ok=True)

df_final_r1_sin_econ=df_ronda2.copy()
df_final_r1_sin_econ['cluster']=labels_r1_sin_econ_full
X_final_r1_sin_econ=X_r1_sin_econ_full
modelo_final_r1_sin_econ=modelo_r1_sin_econ_full

NOMBRES_R1_SIN_ECON={
 0:'Inversionistas casados de edad intermedia',
 1:'Hombres jóvenes solteros orientados a vivienda',
 2:'Hombres casados orientados a vivienda de 3 habitaciones',
 3:'Mujeres jóvenes solteras orientadas a vivienda',
 4:'Mujeres casadas orientadas a vivienda de 3 habitaciones',
 5:'Inversionistas solteros adultos jóvenes'
}

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Mide la conversión por cluster como validación externa; esta variable no interviene en la formación de los segmentos.</div>


In [ ]:
def moda_r1_sin_econ(s): return s.mode().iloc[0]
resumen_clusters_r1_sin_econ=df_final_r1_sin_econ.groupby('cluster').agg(
 observaciones=('cluster','size'), tasa_conversion=('target_won','mean'),
 edad_promedio=('edad','mean'), razon_dominante=('razon_de_compra',moda_r1_sin_econ),
 tipo_dominante=('tipo_unidad',moda_r1_sin_econ), estado_civil_dominante=('estado_civil',moda_r1_sin_econ),
 sexo_dominante=('sexo',moda_r1_sin_econ)
).reset_index()
resumen_clusters_r1_sin_econ['porcentaje_dataset']=resumen_clusters_r1_sin_econ.observaciones/len(df_final_r1_sin_econ)*100
resumen_clusters_r1_sin_econ['nombre']=resumen_clusters_r1_sin_econ.cluster.map(NOMBRES_R1_SIN_ECON)
cols=['cluster','nombre','observaciones','porcentaje_dataset','tasa_conversion','edad_promedio',
      'razon_dominante','tipo_dominante','estado_civil_dominante','sexo_dominante']
display(resumen_clusters_r1_sin_econ[cols].style.format({
 'porcentaje_dataset':'{:.1f}%','tasa_conversion':'{:.2%}','edad_promedio':'{:.1f}'
}).background_gradient(subset=['tasa_conversion'],cmap='YlGn'))

**Interpretación de la tabla.** Los clusters están balanceados. Los grupos 0 y 5 reúnen inversionistas y presentan las conversiones más altas; los clusters 2 y 4 agrupan compradores casados orientados a vivienda de tres habitaciones; los clusters 1 y 3 representan compradores jóvenes y solteros orientados a vivienda, diferenciados principalmente por sexo.

### 9.8 Validación estadística de las diferencias entre clusters

#### 9.8.1 Variable numérica — ANOVA

Se aplica ANOVA de una vía únicamente a edad. Dado el tamaño muestral, la significancia se interpreta junto con η²: <0.01 despreciable, 0.01–0.06 pequeño, 0.06–0.14 mediano y >0.14 grande.

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define la función auxiliar `eta2_r1_sin_econ`, reutilizada para ejecutar esta etapa de forma consistente y reproducible.</div>


In [ ]:
def eta2_r1_sin_econ(values,labels):
    x=np.asarray(values,float); total=((x-x.mean())**2).sum()
    return 0 if total==0 else sum(len(x[labels==g])*(x[labels==g].mean()-x.mean())**2 for g in np.unique(labels))/total
def interpretar_eta2(x):
    return 'despreciable' if x<.01 else 'pequeño' if x<.06 else 'mediano' if x<=.14 else 'grande'

NUM_TEST_R1_SIN_ECON=['edad']
anova_rows_r1_sin_econ=[]; tukey_rows_r1_sin_econ=[]
for var in NUM_TEST_R1_SIN_ECON:
    groups=[df_final_r1_sin_econ.loc[df_final_r1_sin_econ.cluster==g,var].to_numpy() for g in range(6)]
    F,p=f_oneway(*groups); eta=eta2_r1_sin_econ(df_final_r1_sin_econ[var],df_final_r1_sin_econ.cluster)
    anova_rows_r1_sin_econ.append({'variable':var,'F':F,'p_value':p,'eta_cuadrado':eta,
                          'magnitud_practica':interpretar_eta2(eta)})
    test=tukey_hsd(*groups); sig=[]; no_sig=[]; contrasts=[]
    for i in range(6):
        for j in range(i+1,6):
            item=f'{i}–{j}'
            (sig if test.pvalue[i,j]<.05 else no_sig).append(item)
            contrasts.append((abs(groups[i].mean()-groups[j].mean()),i,j,groups[i].mean()-groups[j].mean()))
    top=max(contrasts)
    tukey_rows_r1_sin_econ.append({'variable':var,'comparaciones_significativas':len(sig),
      'total_comparaciones':15,'pares_no_significativos':', '.join(no_sig) if no_sig else 'Ninguno',
      'mayor_contraste':f'C{top[1]} vs C{top[2]} ({top[3]:,.2f})'})
anova_r1_sin_econ=pd.DataFrame(anova_rows_r1_sin_econ); resumen_tukey_r1_sin_econ=pd.DataFrame(tukey_rows_r1_sin_econ)
display(anova_r1_sin_econ.style.format({'F':'{:,.2f}','p_value':'{:.2e}','eta_cuadrado':'{:.3f}'}).background_gradient(subset=['eta_cuadrado'],cmap='YlOrRd'))

**Interpretación del ANOVA.** Edad presenta diferencias estadísticamente significativas entre clusters (p<0.001) y un efecto grande (η²=0.158), por lo que constituye el diferenciador numérico sustantivo del modelo.

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Realiza comparaciones post hoc entre pares de clusters para localizar diferencias numéricas específicas.</div>


In [ ]:
display(resumen_tukey_r1_sin_econ)

#### 9.8.2 Variable numérica — comparaciones post hoc

**Interpretación de Tukey HSD.** Edad difiere en las 15 parejas: el orden es C0 (45.0 años), C2 (43.1), C4 (42.1), C5 (36.5), C3 (35.2) y C1 (33.9).

#### 9.8.3 Variables categóricas

Se aplica χ² de independencia y V de Cramér: <0.10 despreciable, 0.10–0.20 débil, 0.20–0.40 moderado y >0.40 fuerte.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula la distribución porcentual de las variables categóricas dentro de cada cluster y genera sus gráficos comparativos.</div>


In [ ]:
def interpretar_cramer(x):
    return 'despreciable' if x<.10 else 'débil' if x<.20 else 'moderado' if x<=.40 else 'fuerte'
cat_rows_r9=[]
for var in ['sexo','estado_civil','razon_de_compra','tipo_unidad']:
    ct=pd.crosstab(df_final_r1_sin_econ.cluster,df_final_r1_sin_econ[var])
    chi,p,_,_=chi2_contingency(ct); q=min(ct.shape)-1
    v=np.sqrt((chi/ct.to_numpy().sum())/q)
    cat_rows_r9.append({'variable':var,'chi_cuadrado':chi,'p_value':p,
                        'cramer_v':v,'magnitud_practica':interpretar_cramer(v)})
categoricas_r1_sin_econ=pd.DataFrame(cat_rows_r9)
display(categoricas_r1_sin_econ.style.format({'chi_cuadrado':'{:,.1f}','p_value':'{:.2e}','cramer_v':'{:.3f}'}).background_gradient(subset=['cramer_v'],cmap='YlOrRd'))

**Interpretación categórica.** Todas las asociaciones son significativas (p<0.001). `razon_de_compra` tiene asociación perfecta (V=1.000), porque cada cluster queda compuesto exclusivamente por vivienda o inversión. Sexo (V=0.858) y estado civil (V=0.713) muestran asociaciones fuertes y dan contenido demográfico a los perfiles. `tipo_unidad` solo alcanza asociación débil (V=0.151): aporta matices de producto, pero no domina la segmentación. Comercialmente, el modelo R1 sin variables económicas distingue primero motivación, sexo y situación civil; la unidad preferida refina el perfil.


### 9.9 Variables que más diferencian los clusters


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Cuantifica la asociación entre las variables originales y los clusters mediante tamaños de efecto y pruebas estadísticas.</div>


In [ ]:
ranking_r1_sin_econ=pd.concat([
 anova_r1_sin_econ[['variable','eta_cuadrado']].rename(columns={'eta_cuadrado':'tamano_efecto'}).assign(metrica='η²'),
 categoricas_r1_sin_econ[['variable','cramer_v']].rename(columns={'cramer_v':'tamano_efecto'}).assign(metrica='V de Cramér')
]).sort_values('tamano_efecto',ascending=False).reset_index(drop=True)
ranking_r1_sin_econ.index=ranking_r1_sin_econ.index+1; ranking_r1_sin_econ.index.name='rango'
display(ranking_r1_sin_econ.style.format({'tamano_efecto':'{:.3f}'}).background_gradient(subset=['tamano_efecto'],cmap='YlOrRd'))
fig,ax=plt.subplots(figsize=(9,5)); plot_rank=ranking_r1_sin_econ.sort_values('tamano_efecto')
ax.barh(plot_rank.variable,plot_rank.tamano_efecto,color='#5B8DB8'); ax.set(title='Poder discriminante de las variables en el modelo final R1 (sin variables económicas)',xlabel='Tamaño de efecto',ylabel='Variable')
plt.tight_layout(); plt.savefig(os.path.join(OUT_FINAL_R1_SIN_ECON,'poder_discriminante_R1_sin_economicas.png'),dpi=300,bbox_inches='tight'); plt.show()

**Interpretación del ranking.** La segmentación está impulsada por razón de compra, sexo y estado civil. Edad ocupa el primer lugar entre las variables numéricas y exhibe efecto grande; tipo de unidad aporta una diferenciación débil. El orden confirma que R1 representa perfiles demográfico-conductuales y de producto, aunque la asociación perfecta de razón de compra exige reconocer un componente de segmentación casi determinista.

### 9.10 Perfil detallado de cada cluster

#### Cluster 0 — Inversionistas casados de edad intermedia

Incluye 2 907 leads (15.9%). Su edad media es 45.0 años. El 100% declara inversión; predomina 2 habitaciones (59.1%), estado casado (99.9%) y sexo masculino (65.6%). Su conversión es 5.26%. El grupo se distingue por combinar motivación de inversión, matrimonio y mayor edad.

#### Cluster 1 — Hombres jóvenes solteros orientados a vivienda

Reúne 3 656 leads (20.0%). Su edad media es 33.9 años. Está compuesto por vivienda (100%), hombres (100%) y principalmente solteros (88.6%); predomina 2 habitaciones (54.5%). Convierte 3.09%. Se diferencia de C3 por sexo y de C2 por menor edad y soltería.

#### Cluster 2 — Hombres casados orientados a vivienda de 3 habitaciones

Es el grupo mayor: 3 729 leads (20.4%). Su edad media es 43.1 años. Está formado por vivienda (100%), hombres casados (100%) y muestra preferencia modal por 3 habitaciones (53.2%). Conversión 2.49%. Frente a C1 representa un perfil masculino de mayor edad y casado; frente a C4 comparte estado civil, motivación y tipo, pero difiere por sexo.

#### Cluster 3 — Mujeres jóvenes solteras orientadas a vivienda

Contiene 3 304 leads (18.0%). Su edad media es 35.2 años. Está compuesto por vivienda (100%), mujeres (100%) y principalmente solteras (89.0%); predomina 2 habitaciones (53.1%). Registra la menor conversión, 2.15%. Es el análogo femenino de C1.

#### Cluster 4 — Mujeres casadas orientadas a vivienda de 3 habitaciones

Agrupa 2 489 leads (13.6%). Su edad media es 42.1 años. Está compuesto por vivienda (100%), mujeres casadas (100%) y preferencia modal por 3 habitaciones (52.0%). Conversión 2.29%. Comparte con C2 edad, estado civil, motivación y producto; su principal diferenciador es sexo.

#### Cluster 5 — Inversionistas solteros adultos jóvenes

Es el menor, aunque sustantivo: 2 230 leads (12.2%). Su edad media es 36.5 años. El 100% declara inversión; predomina soltería (97.6%), sexo masculino (63.9%) y 2 habitaciones (56.6%). Presenta la conversión más alta, 5.87%. Se diferencia de C0 por menor edad y soltería, y de los perfiles de vivienda por motivación y conversión.

### 9.11 Conversión por cluster

Se calculan intervalos de Wilson al 95%, lift respecto a la tasa global y una prueba bilateral de dos proporciones entre cada cluster y el resto del universo.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Mide la conversión por cluster como validación externa; esta variable no interviene en la formación de los segmentos.</div>


In [ ]:
global_conversion_r1_sin_econ=df_final_r1_sin_econ.target_won.mean(); conv_rows=[]
for g,x in df_final_r1_sin_econ.groupby('cluster'):
    n=len(x); wins=int(x.target_won.sum()); ph=wins/n; z=1.96
    den=1+z*z/n; center=(ph+z*z/(2*n))/den
    half=z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    rest=df_final_r1_sin_econ.loc[df_final_r1_sin_econ.cluster!=g,'target_won']; pooled=(wins+rest.sum())/(n+len(rest))
    se=np.sqrt(pooled*(1-pooled)*(1/n+1/len(rest))); p_vs=2*norm.sf(abs((ph-rest.mean())/se))
    conv_rows.append({'cluster':g,'observaciones':n,'convertidos':wins,'tasa_conversion':ph,
      'IC95_inferior':center-half,'IC95_superior':center+half,'lift':ph/global_conversion_r1_sin_econ,
      'p_vs_resto':p_vs,'clasificacion':('superior' if ph>rest.mean() and p_vs<.05 else 'inferior' if ph<rest.mean() and p_vs<.05 else 'sin diferencia')})
conversion_final_r1_sin_econ=pd.DataFrame(conv_rows)
display(conversion_final_r1_sin_econ.style.format({'tasa_conversion':'{:.2%}','IC95_inferior':'{:.2%}','IC95_superior':'{:.2%}','lift':'{:.2f}×','p_vs_resto':'{:.2e}'}).background_gradient(subset=['lift'],cmap='YlGn'))

fig,ax=plt.subplots(figsize=(9,5)); rates=conversion_final_r1_sin_econ.tasa_conversion*100
err=np.vstack([(conversion_final_r1_sin_econ.tasa_conversion-conversion_final_r1_sin_econ.IC95_inferior)*100,(conversion_final_r1_sin_econ.IC95_superior-conversion_final_r1_sin_econ.tasa_conversion)*100])
bars=ax.bar(conversion_final_r1_sin_econ.cluster,rates,yerr=err,capsize=5,color=sns.color_palette('Set2',6),edgecolor='white')
ax.axhline(global_conversion_r1_sin_econ*100,color='red',ls='--',label=f'Conversión global: {global_conversion_r1_sin_econ:.2%}')
for b,v,lift in zip(bars,rates,conversion_final_r1_sin_econ.lift): ax.text(b.get_x()+b.get_width()/2,v+.25,f'{v:.2f}%\n{lift:.2f}×',ha='center',fontsize=9)
ax.set(title='Conversión e intervalos de confianza del 95% por cluster — R1 (sin variables económicas)',xlabel='Cluster',ylabel='Conversión (%)'); ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_FINAL_R1_SIN_ECON,'conversion_clusters_R1_sin_economicas.png'),dpi=300,bbox_inches='tight'); plt.show()

**Interpretación de conversión.** La tasa global es 3.37%. C5 alcanza 5.87% (lift 1.74; IC95% 4.97%–6.93%) y C0 5.26% (lift 1.56; IC95% 4.51%–6.14%); ambos superan significativamente al resto (p<0.001). C1 no difiere significativamente (3.09%; p=0.289). C2, C3 y C4 convierten significativamente menos, con lifts de 0.74, 0.64 y 0.68. La prioridad comercial debe concentrar respuesta rápida y seguimiento intensivo en C5 y C0. C1 amerita atención estándar; C2–C4 requieren nutrición y revisión de barreras antes de aumentar inversión comercial.


### 9.12 Separación entre clusters

Las distancias se calculan en el espacio exacto de R1 (sin variables económicas): edad estandarizada y ponderada, más bloques OHE balanceados y ponderados.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula las distancias entre centroides para identificar qué clusters son más próximos y cuáles están más separados.</div>


In [ ]:
dist_centroides_r1_sin_econ=pairwise_distances(modelo_final_r1_sin_econ.cluster_centers_)
tabla_dist_r1_sin_econ=pd.DataFrame(dist_centroides_r1_sin_econ,index=[f'C{i}' for i in range(6)],columns=[f'C{i}' for i in range(6)])
tri=np.triu_indices(6,1); values=dist_centroides_r1_sin_econ[tri]; imin=values.argmin(); imax=values.argmax()
resumen_dist_r1_sin_econ=pd.DataFrame([{'distancia_promedio':values.mean(),'distancia_minima':values.min(),
 'clusters_mas_similares':f'C{tri[0][imin]}–C{tri[1][imin]}','distancia_maxima':values.max(),
 'clusters_mas_diferentes':f'C{tri[0][imax]}–C{tri[1][imax]}'}])
display(tabla_dist_r1_sin_econ.style.format('{:.3f}').background_gradient(cmap='YlOrRd',axis=None))
display(resumen_dist_r1_sin_econ.style.format({'distancia_promedio':'{:.3f}','distancia_minima':'{:.3f}','distancia_maxima':'{:.3f}'}))
fig,ax=plt.subplots(figsize=(7,6)); sns.heatmap(tabla_dist_r1_sin_econ,annot=True,fmt='.3f',cmap='YlOrRd',ax=ax,square=True)
ax.set_title('Distancia euclidiana entre centroides del modelo final R1 (sin variables económicas)'); plt.tight_layout()
plt.savefig(os.path.join(OUT_FINAL_R1_SIN_ECON,'distancias_centroides_R1_sin_economicas.png'),dpi=300,bbox_inches='tight'); plt.show()

**Interpretación de distancias.** La distancia promedio es 1.555. C3–C4 son los más próximos (1.179): ambos son perfiles femeninos orientados a vivienda, aunque difieren en edad, estado civil y tipo modal. C4–C5 son los más separados (2.016): contraponen mujeres casadas orientadas a vivienda de tres habitaciones con inversionistas solteros más jóvenes y mayoritariamente masculinos. La geometría coincide con la lectura empresarial de los perfiles.


### 9.13 Visualizaciones adicionales

Se selecciona un mapa de calor estandarizado porque resume simultáneamente variables numéricas y proporciones categóricas relevantes sin repetir las gráficas anteriores.


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume las variables originales por cluster para identificar las características que distinguen cada perfil.</div>


In [ ]:
profile_visual_r1_sin_econ=df_final_r1_sin_econ.groupby('cluster').agg(
 edad=('edad','mean'),
 inversion=('razon_de_compra',lambda s:(s=='Inversión').mean()*100),
 casados=('estado_civil',lambda s:(s=='Casad@').mean()*100),
 femenino=('sexo',lambda s:(s=='Femenino').mean()*100),
 tres_habitaciones=('tipo_unidad',lambda s:(s=='3 Habs').mean()*100))
z_profile=(profile_visual_r1_sin_econ-profile_visual_r1_sin_econ.mean())/profile_visual_r1_sin_econ.std().replace(0,1)
fig,ax=plt.subplots(figsize=(10,6)); sns.heatmap(z_profile.T,annot=True,fmt='.2f',cmap='RdBu_r',center=0,linewidths=.5,ax=ax,cbar_kws={'label':'Z-score entre clusters'})
ax.set(title='Comparación estandarizada de los perfiles del modelo final R1 (sin variables económicas)',xlabel='Cluster',ylabel='Indicador'); plt.tight_layout()
plt.savefig(os.path.join(OUT_FINAL_R1_SIN_ECON,'perfil_estandarizado_R1_sin_economicas.png'),dpi=300,bbox_inches='tight'); plt.show()

**Interpretación de la visualización.** C0 sobresale por edad e inversión; C5 por inversión y soltería; C2 y C4 por matrimonio y tres habitaciones; C1 y C3 por menor edad y soltería. El patrón confirma que los clusters se organizan por combinaciones demográficas, motivacionales y de producto.

### 9.14 Interpretación de negocio e implicaciones para la empresa

- **Prioridad comercial alta — C5 y C0.** Ambos segmentos de inversión convierten significativamente por encima del promedio. C5 reúne leads solteros adultos jóvenes, mientras C0 concentra inversionistas casados de mayor edad. Los argumentos deben centrarse en la motivación de inversión declarada; cualquier promesa de rentabilidad requiere información financiera adicional no disponible en esta base.
- **Seguimiento estándar — C1.** Su conversión no difiere de la global. El equipo puede ofrecer acompañamiento acorde con un perfil joven, soltero y orientado a vivienda de dos habitaciones.
- **Nutrición diferenciada — C2 y C4.** Son compradores casados de vivienda, con preferencia modal por tres habitaciones, pero conversión inferior. Conviene revisar disponibilidad, adecuación y objeciones asociadas a esa tipología antes de intensificar contacto. No deben recibir estrategias distintas únicamente por sexo; este atributo se utiliza para describir, no para discriminar.
- **Revisión de barreras — C3.** Presenta la menor conversión. El seguimiento debe validar tempranamente la adecuación de unidades de dos habitaciones y las principales objeciones antes de dedicar recursos intensivos.
- **Marketing.** La principal bifurcación accionable es inversión frente a vivienda. Las campañas y mensajes deben separarse por motivación y luego adaptarse por estado civil, edad y tipo de unidad. Sexo no debe emplearse como criterio excluyente; su valor es analítico y descriptivo.
- **Priorización.** El lift de conversión permite ordenar la atención C5 → C0 → C1 → C2/C4 → C3, sujeto a capacidad operativa y sin reemplazar la evaluación individual del lead.

### 9.15 Limitaciones

- El universo final incluye 18 315 de 42 464 leads calificados porque exige completitud en las cinco variables obligatorias. Esta selección puede introducir sesgo frente a leads con información demográfica incompleta.
- Esta réplica excluye deliberadamente ingreso, precio y m² mínimo, por lo que no permite caracterizar capacidad económica, presupuesto ni superficie de interés.
- El CRM no dispone con cobertura suficiente de hijos, dependientes, ocupación, estabilidad laboral, capacidad de cuota, patrimonio, localización residencial ni comportamiento detallado de seguimiento. Estas ausencias limitan la riqueza del perfil.
- K-Means presupone distancias euclidianas, centroides y grupos aproximadamente compactos. R1 aplica el algoritmo a edad estandarizada y categorías OHE ponderadas; las ponderaciones reflejan una decisión metodológica explícita.
- `razon_de_compra` presenta V de Cramér=1.000 y queda determinada por cluster. Esto mejora separación, pero introduce un componente casi circular que debe declararse al interpretar el modelo.
- La conversión se usa después del clustering y no demuestra causalidad. Las diferencias pueden reflejar variables omitidas o condiciones comerciales no registradas.
- Futuros enriquecimientos del CRM deberían mejorar cobertura y estandarización de hogar, dependientes, empleo, presupuesto, financiamiento e interacciones comerciales. Con esos datos conviene evaluar estabilidad temporal, validación externa y algoritmos para datos mixtos, manteniendo R1 como referencia congelada.

### 9.16 Exportación del modelo final y resultados


<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Crea la carpeta de salida y exporta etiquetas, perfiles y métricas para reutilizarlos fuera del notebook.</div>


In [ ]:
resumen_clusters_r1_sin_econ.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON,'resumen_clusters_R1_sin_economicas.csv'),index=False,encoding='utf-8-sig')
anova_r1_sin_econ.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON,'anova_eta2_R1_sin_economicas.csv'),index=False,encoding='utf-8-sig')
resumen_tukey_r1_sin_econ.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON,'tukey_resumen_R1_sin_economicas.csv'),index=False,encoding='utf-8-sig')
categoricas_r1_sin_econ.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON,'chi2_cramer_R1_sin_economicas.csv'),index=False,encoding='utf-8-sig')
ranking_r1_sin_econ.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON,'ranking_variables_R1_sin_economicas.csv'),encoding='utf-8-sig')
conversion_final_r1_sin_econ.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON,'conversion_intervalos_lift_R1_sin_economicas.csv'),index=False,encoding='utf-8-sig')
tabla_dist_r1_sin_econ.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON,'distancias_centroides_R1_sin_economicas.csv'),encoding='utf-8-sig')
df_final_r1_sin_econ[['cluster']].to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON,'asignacion_clusters_R1_sin_economicas.csv'),index_label='indice_original',encoding='utf-8-sig')
print(f'Resultados finales R1 (sin variables económicas) exportados en: {OUT_FINAL_R1_SIN_ECON}')

import joblib
pd.DataFrame({'cluster': labels_r1_sin_econ_full}, index=df_ronda2.index).to_csv(
    os.path.join(OUT_FINAL_R1_SIN_ECON, 'asignacion_R1_sin_economicas_universo_completo.csv'),
    index_label='indice_original', encoding='utf-8-sig')
metricas_r1_sin_econ_full.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON, 'metricas_R1_sin_economicas_universo_completo.csv'), index=False, encoding='utf-8-sig')
distribucion_r1_sin_econ_full.to_csv(os.path.join(OUT_FINAL_R1_SIN_ECON, 'distribucion_R1_sin_economicas_universo_completo.csv'), encoding='utf-8-sig')
joblib.dump(modelo_r1_sin_econ_full, os.path.join(OUT_FINAL_R1_SIN_ECON, 'modelo_KMeans_R1_sin_economicas_universo_completo.joblib'))
print('Modelo R1 (sin variables económicas) completo, métricas y asignaciones exportados.')